# Knowledge Base and Dense Retrieval

## 1. Цель этапа

На этом этапе клинические рекомендации преобразуются в структурированную базу знаний, по которой затем можно искать релевантные фрагменты для медицинского вопроса.

Основные задачи:

- загрузить и проверить исходные PDF;
- извлечь текст и структуру документов;
- сохранить информацию об источнике, разделе и странице;
- разбить документы на смысловые фрагменты;
- построить baseline dense retrieval и проверить корректность его работы на вопросах из `debug`.

На этом этапе генерация ответов с помощью Qwen не используется. Сначала отдельно проверяется качество retrieval.

## 2. Подготовка документов

PDF хранит в первую очередь расположение текста на странице, а не готовую структуру вида «заголовок - раздел - абзац».

Поэтому простого извлечения всего текста недостаточно. Для RAG важно по возможности сохранить структуру документа: название рекомендации, раздел, страницу и сам текст.

Это позволит не только находить релевантный фрагмент, но и понимать, из какого источника он был получен.

In [1]:
import pymupdf
import pandas as pd

import re

from collections import Counter

In [2]:
from pathlib import Path

GUIDELINES_DIR = Path("../data/source_documents/guidelines")

pdf_paths = sorted(GUIDELINES_DIR.rglob("*.pdf"))

print("Количество PDF:", len(pdf_paths))

for path in pdf_paths:
    print(path)

Количество PDF: 8
..\data\source_documents\guidelines\cdc\cdc_sti_2021.pdf
..\data\source_documents\guidelines\va_dod\va_dod_asthma_2025.pdf
..\data\source_documents\guidelines\va_dod\va_dod_ckd_2025.pdf
..\data\source_documents\guidelines\va_dod\va_dod_low_back_pain_2022.pdf
..\data\source_documents\guidelines\va_dod\va_dod_major_depression_2022.pdf
..\data\source_documents\guidelines\va_dod\va_dod_pregnancy_2023.pdf
..\data\source_documents\guidelines\va_dod\va_dod_type2_diabetes_2023.pdf
..\data\source_documents\guidelines\who\who_hypertension_2021.pdf


## 3. Проверка извлечения текста

Перед построением parser проверим, насколько хорошо текст извлекается из исходных PDF.

PDF описывает расположение элементов на странице, поэтому после извлечения могут появляться:

- повторяющиеся headers и footers;
- неправильный порядок текста;
- разрывы слов и строк;
- смешивание текста из таблиц;
- потеря структуры заголовков.

Сначала проверим наличие текстового слоя во всех документах, затем посмотрим несколько страниц подробнее.

In [3]:
pdf_stats = []

for path in pdf_paths:
    doc = pymupdf.open(path)

    empty_pages = 0
    total_chars = 0

    for page in doc:
        text = page.get_text("text").strip()

        total_chars += len(text)

        if not text:
            empty_pages += 1

    pdf_stats.append(
        {
            "file": path.name,
            "pages": len(doc),
            "empty_pages": empty_pages,
            "characters": total_chars,
        }
    )

    doc.close()

pdf_stats_df = pd.DataFrame(pdf_stats)

pdf_stats_df

,file,pages,empty_pages,characters
0,cdc_sti_2021.pdf,192,2,1122703
1,va_dod_asthma_2025.pdf,149,0,322893
2,va_dod_ckd_2025.pdf,198,0,554221
3,va_dod_low_back_pain_2022.pdf,141,0,435526
4,va_dod_major_depression_2022.pdf,159,0,456683
5,va_dod_pregnancy_2023.pdf,193,0,539638
6,va_dod_type2_diabetes_2023.pdf,165,0,421845
7,who_hypertension_2021.pdf,61,3,159357


In [4]:
sample_paths = [
    GUIDELINES_DIR / "who" / "who_hypertension_2021.pdf",
    GUIDELINES_DIR / "cdc" / "cdc_sti_2021.pdf",
    GUIDELINES_DIR / "va_dod" / "va_dod_asthma_2025.pdf",
]

for path in sample_paths:
    doc = pymupdf.open(path)

    page_number = 10
    text = doc[page_number].get_text("text")

    print("\n" + "=" * 100)
    print(path.name)
    print("=" * 100)
    print(text[:3000])

    doc.close()


who_hypertension_2021.pdf
WHO recommends a target systolic blood pressure treatment goal of <130 mmHg in patients 
with hypertension and known cardiovascular disease (CVD).
Strong recommendation, moderate-certainty evidence
WHO suggests a target systolic blood pressure treatment goal of <130 mmHg in high-risk 
patients with hypertension (those with high CVD risk, diabetes mellitus, chronic kidney 
disease).
Conditional recommendation, moderate-certainty evidence
7.	 RECOMMENDATIONS ON FREQUENCY OF ASSESSMENT
WHO suggests a monthly follow up after initiation or a change in antihypertensive 
medications until patients reach target.
Conditional recommendation, low-certainty evidence
WHO suggests a follow up every 3–6 months for patients whose blood pressure is under 
control.
Conditional recommendation, low-certainty evidence
8.	 RECOMMENDATION ON TREATMENT BY NONPHYSICIAN PROFESSIONALS
WHO suggests that pharmacological treatment of hypertension can be provided by 
nonphysician professio

### Результат проверки

Все документы содержат доступный текстовый слой, поэтому OCR для корпуса не требуется.

Основной клинический текст извлекается корректно, однако присутствуют элементы PDF-разметки, которые не должны оставаться в текстовом содержимом chunks: повторяющиеся headers и footers, служебные номера страниц и переносы слов между строками.

При этом информация о происхождении фрагмента должна быть сохранена отдельно в metadata: источник, название документа, год, раздел и страница.

Обычное извлечение текста не сохраняет явно структуру разделов. Поэтому перед chunking необходимо определить, можно ли восстановить заголовки по форматированию PDF.

## 4. Структура текста внутри PDF

Для retrieval полезно знать не только страницу, но и раздел, к которому относится найденный фрагмент.

PDF не хранит структуру как HTML, однако `PyMuPDF` позволяет получить отдельные text spans вместе с размером и названием шрифта. Проверим, отличаются ли таким образом заголовки от обычного текста.

In [5]:
def inspect_page_spans(path, page_number, limit=40):
    doc = pymupdf.open(path)
    page = doc[page_number]

    page_dict = page.get_text("dict")

    rows = []

    for block in page_dict["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()

                if text:
                    rows.append(
                        {
                            "text": text,
                            "font_size": round(span["size"], 1),
                            "font": span["font"],
                        }
                    )

    doc.close()

    return pd.DataFrame(rows).head(limit)

In [6]:
asthma_path = (
    GUIDELINES_DIR
    / "va_dod"
    / "va_dod_asthma_2025.pdf"
)

inspect_page_spans(
    asthma_path,
    page_number=29,
    limit=50,
)

,text,font_size,font
0,VA/DOD Clinical Practice Guideline for the Pri...,10.0,Arial-ItalicMT
1,March 2025,10.0,ArialMT
2,Page 30 of 149,10.0,ArialMT
3,IX. Recommendations,14.0,Arial-BoldMT
4,The evidence-based clinical practice recommend...,11.0,ArialMT
5,using a systematic approach considering four d...,11.0,ArialMT
6,Summary of Guideline Development Methodology).,11.0,ArialMT
7,These domains include confidence in the,11.0,ArialMT
8,"quality of the evidence, balance of desirable ...",11.0,ArialMT
9,"patient values and preferences, and other impl...",11.0,ArialMT


По форматированию VA/DoD видно, что заголовки разделов отличаются от основного текста размером и начертанием шрифта. Это позволяет использовать информацию о форматировании для восстановления структуры документа.

При этом жирный шрифт используется не только для заголовков, но и внутри таблиц, поэтому определять раздел только по признаку `bold` нельзя.

Перед написанием parser необходимо проверить, как оформлены заголовки в документах WHO и CDC.

In [7]:
def find_pages_with_text(path, query):
    doc = pymupdf.open(path)

    matches = []

    for page_number, page in enumerate(doc):
        text = page.get_text("text")

        if query.lower() in text.lower():
            matches.append(page_number)

    doc.close()

    return matches

In [8]:
who_path = (
    GUIDELINES_DIR
    / "who"
    / "who_hypertension_2021.pdf"
)

find_pages_with_text(
    who_path,
    "Blood pressure threshold",
)

[4, 8, 13, 18, 55]

In [9]:
cdc_path = (
    GUIDELINES_DIR
    / "cdc"
    / "cdc_sti_2021.pdf"
)

find_pages_with_text(
    cdc_path,
    "Chlamydial Infections",
)

[1, 11, 12, 15, 17, 23, 28, 63, 64, 65, 66, 70, 73, 76, 98, 143, 148, 163, 164]

In [10]:
inspect_page_spans(
    who_path,
    page_number=18,
    limit=50,
)

,text,font_size,font
0,3\t Recommendations,25.0,Frutiger-Bold
1,3.1,12.0,Frutiger-Bold
2,Blood pressure threshold for initiation of pha...,12.0,Frutiger-Bold
3,1.,10.0,Frutiger-Bold
4,RECOMMENDATION ON BLOOD PRESSURE THRESHOLD FOR...,10.0,Frutiger-Bold
5,PHARMACOLOGICAL TREATMENT,10.0,Frutiger-Bold
6,WHO recommends initiation of pharmacological a...,10.0,Frutiger-Roman
7,with a confirmed diagnosis of hypertension and...,10.0,Frutiger-Roman
8,≥,10.0,Symbol
9,140 mmHg or,10.0,Frutiger-Roman


In [11]:
inspect_page_spans(
    cdc_path,
    page_number=64,
    limit=50,
)

,text,font_size,font
0,Recommendations and Reports,9.0,MyriadPro-Regular
1,"MMWR / July 23, 2021 / Vol. 70 / No. 4",8.0,MyriadPro-Regular
2,63,8.0,MyriadPro-Regular
3,US Department of Health and Human Services/Cen...,8.0,MyriadPro-Regular
4,completion of a 7-day regimen and symptoms hav...,11.0,AGaramondPro-Regular
5,for 7 days after single-dose therapy). Men wit...,11.0,AGaramondPro-Regular
6,be tested for HIV and syphilis.,11.0,AGaramondPro-Regular
7,Follow-Up,12.0,MyriadPro-Semibold
8,Men should be provided their testing results o...,11.0,AGaramondPro-Regular
9,part of the NGU evaluation. Those with a speci...,11.0,AGaramondPro-Regular


In [12]:
def find_matching_spans(path, query):
    doc = pymupdf.open(path)

    matches = []

    for page_number, page in enumerate(doc):
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:
            if "lines" not in block:
                continue

            for line in block["lines"]:
                line_text = "".join(
                    span["text"] for span in line["spans"]
                ).strip()

                if query.lower() in line_text.lower():
                    matches.append(
                        {
                            "pdf_page": page_number + 1,
                            "text": line_text,
                            "spans": [
                                {
                                    "text": span["text"],
                                    "size": round(span["size"], 1),
                                    "font": span["font"],
                                }
                                for span in line["spans"]
                            ],
                        }
                    )

    doc.close()

    return matches

In [13]:
find_matching_spans(
    cdc_path,
    "Chlamydial Infections",
)

[{'pdf_page': 2,
  'text': 'Chlamydial Infections........................................................................................ 65',
  'spans': [{'text': 'Chlamydial Infections........................................................................................ 65',
    'size': 9.0,
    'font': 'MyriadPro-Regular'}]},
 {'pdf_page': 12,
  'text': 'or chlamydial infections among MSM, compared with',
  'spans': [{'text': 'or chlamydial infections among MSM, compared with ',
    'size': 11.0,
    'font': 'AGaramondPro-Regular'}]},
 {'pdf_page': 12,
  'text': 'or chlamydial infections (133,134); however, more recent data',
  'spans': [{'text': 'or chlamydial infections (',
    'size': 11.0,
    'font': 'AGaramondPro-Regular'},
   {'text': '133', 'size': 11.0, 'font': 'AGaramondPro-Italic'},
   {'text': ',', 'size': 11.0, 'font': 'AGaramondPro-Regular'},
   {'text': '134', 'size': 11.0, 'font': 'AGaramondPro-Italic'},
   {'text': '); however, more recent data ',
    'size': 11.0

In [14]:
find_matching_spans(
    who_path,
    "Blood pressure threshold for initiation",
)

[{'pdf_page': 5,
  'text': 'Blood pressure threshold for initiation of pharmacological treatment',
  'spans': [{'text': 'Blood pressure threshold for initiation of pharmacological treatment\t',
    'size': 10.0,
    'font': 'Frutiger-Light'}]},
 {'pdf_page': 9,
  'text': '1.\t RECOMMENDATION ON BLOOD PRESSURE THRESHOLD FOR INITIATION OF',
  'spans': [{'text': '1.\t', 'size': 10.0, 'font': 'Frutiger-Bold'},
   {'text': ' ', 'size': 10.0, 'font': 'Frutiger-Bold'},
   {'text': 'RECOMMENDATION ON BLOOD PRESSURE THRESHOLD FOR INITIATION OF ',
    'size': 10.0,
    'font': 'Frutiger-Bold'}]},
 {'pdf_page': 19,
  'text': '3.1\t Blood pressure threshold for initiation of pharmacological treatment',
  'spans': [{'text': '3.1\t', 'size': 12.0, 'font': 'Frutiger-Bold'},
   {'text': ' ', 'size': 12.0, 'font': 'Frutiger-Bold'},
   {'text': 'Blood pressure threshold for initiation of pharmacological treatment',
    'size': 12.0,
    'font': 'Frutiger-Bold'}]},
 {'pdf_page': 19,
  'text': '1.\t RECOM

### Результат анализа структуры

Структура документов может быть частично восстановлена по размеру и начертанию шрифта, однако правила различаются между издателями.

Для VA/DoD крупные разделы и подразделы преимущественно выделяются Arial Bold разного размера.

В WHO наблюдается несколько уровней заголовков: крупные заголовки глав, нумерованные подразделы и внутренние блоки рекомендаций.

В CDC основной текст и заголовки используют разные семейства шрифтов: основной текст преимущественно набран AGaramondPro, тогда как заголовки разделов используют MyriadPro-Semibold.

Поэтому один универсальный критерий вида `bold = heading` ненадёжен. Для трёх семейств документов будут использоваться отдельные правила определения структуры, но результат parser будет приведён к единой схеме metadata.

Для каждого текстового фрагмента будут сохраняться:

- источник;
- название документа;
- год;
- иерархический путь раздела (`section_path`);
- номер страницы документа;
- физическая страница PDF;
- текст.

In [15]:
doc = pymupdf.open(asthma_path)

for pdf_index in [0, 1, 10, 29, 30]:
    page = doc[pdf_index]

    print(
        "pdf_index =", pdf_index,
        "| pdf_page =", pdf_index + 1,
        "| page_label =", page.get_label(),
    )

doc.close()

pdf_index = 0 | pdf_page = 1 | page_label = 
pdf_index = 1 | pdf_page = 2 | page_label = 
pdf_index = 10 | pdf_page = 11 | page_label = 
pdf_index = 29 | pdf_page = 30 | page_label = 
pdf_index = 30 | pdf_page = 31 | page_label = 


In [16]:
doc = pymupdf.open(who_path)

for pdf_index in [0, 4, 8, 18]:
    page = doc[pdf_index]

    print(
        "pdf_index =", pdf_index,
        "| pdf_page =", pdf_index + 1,
        "| page_label =", page.get_label(),
    )

doc.close()

pdf_index = 0 | pdf_page = 1 | page_label = a
pdf_index = 4 | pdf_page = 5 | page_label = iii
pdf_index = 8 | pdf_page = 9 | page_label = vii
pdf_index = 18 | pdf_page = 19 | page_label = 7


In [17]:
doc = pymupdf.open(cdc_path)

for pdf_index in [0, 1, 66, 67]:
    page = doc[pdf_index]

    print(
        "pdf_index =", pdf_index,
        "| pdf_page =", pdf_index + 1,
        "| page_label =", page.get_label(),
    )

doc.close()

pdf_index = 0 | pdf_page = 1 | page_label = i
pdf_index = 1 | pdf_page = 2 | page_label = ii
pdf_index = 66 | pdf_page = 67 | page_label = 65
pdf_index = 67 | pdf_page = 68 | page_label = 66


### Нумерация страниц

PDF metadata различается между источниками.

В документах WHO логические номера страниц доступны через `page.get_label()`. Они учитывают внутреннюю нумерацию документа, включая римские номера во вводных разделах и обычные номера в основном тексте.

В документах CDC логические page labels также доступны через `page.get_label()` и корректно отражают номер страницы, напечатанный в документе, который может отличаться от физической позиции страницы в PDF.

В документах VA/DoD page labels отсутствуют. В проверенных страницах физическая нумерация PDF совпадает с номером страницы, указанным в самом документе, поэтому для этих документов можно использовать физический номер страницы как номер документа.

Таким образом:

- `pdf_page` — физическая позиция страницы в PDF, начиная с 1;
- `page` — логический номер страницы документа, используемый для цитирования.

`pdf_page` будет сохраняться всегда. Для WHO и CDC значение `page` будет получаться через `page.get_label()`, а для VA/DoD — из физического номера страницы.

Для VA/DoD отдельно необходимо учитывать таблицы: жирный текст внутри таблиц может визуально совпадать с форматированием заголовков разделов. Поэтому признаки шрифта будут использоваться вместе с положением текста и структурой страницы, а не как единственный критерий.

## 5. Дизайн parser

Документы WHO, CDC и VA/DoD имеют разные правила форматирования, поэтому определение заголовков будет выполняться отдельно для каждого семейства документов.

При этом результат parsing должен иметь единую структуру, чтобы последующие этапы chunking и retrieval не зависели от формата исходного PDF.

Для каждого текстового фрагмента сохраняются:

- `document_id` — уникальный идентификатор документа;
- `source` — организация-источник;
- `document_title` — название исходного документа;
- `year` — год публикации;
- `section_path` — иерархический путь раздела;
- `page` — логический номер страницы документа;
- `pdf_page` — физическая страница PDF;
- `text` — очищенный текст.

Сначала документы будут преобразованы в структурированные текстовые фрагменты. Chunking выполняется отдельным этапом после проверки качества parsing.

In [18]:
DOCUMENT_METADATA = {
    "cdc_sti_2021.pdf": {
    "source": "CDC",
    "year": 2021,
    "family": "cdc",
    "document_title": (
        "Sexually Transmitted Infections Treatment Guidelines, 2021"
    ),
    },
    "who_hypertension_2021.pdf": {
    "source": "WHO",
    "year": 2021,
    "family": "who",
    "document_title": (
        "Guideline for the pharmacological treatment "
        "of hypertension in adults"
    ),
    },
    "va_dod_asthma_2025.pdf": {
        "source": "VA/DoD",
        "year": 2025,
        "family": "va_dod",
        "document_title": (
            "VA/DOD Clinical Practice Guideline for the "
            "Primary Care Management of Asthma"
        ),
    },
    "va_dod_ckd_2025.pdf": {
        "source": "VA/DoD",
        "year": 2025,
        "family": "va_dod",
        "document_title": (
            "VA/DOD Clinical Practice Guideline for the "
            "Primary Care Management of Chronic Kidney Disease"
        ),
    },
    "va_dod_low_back_pain_2022.pdf": {
        "source": "VA/DoD",
        "year": 2022,
        "family": "va_dod",
        "document_title": (
            "VA/DoD Clinical Practice Guideline for the "
            "Diagnosis and Treatment of Low Back Pain"
        ),
    },
    "va_dod_major_depression_2022.pdf": {
        "source": "VA/DoD",
        "year": 2022,
        "family": "va_dod",
        "document_title": (
            "VA/DoD Clinical Practice Guideline for the "
            "Management of Major Depressive Disorder"
        ),
    },
    "va_dod_pregnancy_2023.pdf": {
        "source": "VA/DoD",
        "year": 2023,
        "family": "va_dod",
        "document_title": (
            "VA/DoD Clinical Practice Guideline for the "
            "Management of Pregnancy"
        ),
    },
    "va_dod_type2_diabetes_2023.pdf": {
        "source": "VA/DoD",
        "year": 2023,
        "family": "va_dod",
        "document_title": (
            "VA/DoD Clinical Practice Guideline for the "
            "Management of Type 2 Diabetes Mellitus"
        ),
    },

}

In [19]:
def get_page_number(page):
    label = page.get_label().strip()

    if label:
        return label

    return str(page.number + 1)

In [20]:
for path in pdf_paths:
    metadata = DOCUMENT_METADATA[path.name]

    with pymupdf.open(path) as doc:
        sample_page = doc[min(10, len(doc) - 1)]

        print(
            path.name,
            "| source:", metadata["source"],
            "| family:", metadata["family"],
            "| page:", get_page_number(sample_page),
            "| pdf_page:", sample_page.number + 1,
        )

cdc_sti_2021.pdf | source: CDC | family: cdc | page: 9 | pdf_page: 11
va_dod_asthma_2025.pdf | source: VA/DoD | family: va_dod | page: 11 | pdf_page: 11
va_dod_ckd_2025.pdf | source: VA/DoD | family: va_dod | page: 11 | pdf_page: 11
va_dod_low_back_pain_2022.pdf | source: VA/DoD | family: va_dod | page: 11 | pdf_page: 11
va_dod_major_depression_2022.pdf | source: VA/DoD | family: va_dod | page: 11 | pdf_page: 11
va_dod_pregnancy_2023.pdf | source: VA/DoD | family: va_dod | page: 11 | pdf_page: 11
va_dod_type2_diabetes_2023.pdf | source: VA/DoD | family: va_dod | page: 11 | pdf_page: 11
who_hypertension_2021.pdf | source: WHO | family: who | page: ix | pdf_page: 11


### Проверка встроенной структуры PDF

Перед восстановлением заголовков по форматированию проверим, содержат ли PDF встроенный outline (bookmarks).

Если он доступен, его предпочтительнее использовать для определения `section_path`, поскольку он уже содержит иерархию разделов и номера страниц и не требует эвристик по размеру или начертанию шрифта.

In [21]:
def inspect_toc(path, limit=30):
    with pymupdf.open(path) as doc:
        toc = doc.get_toc()

    return pd.DataFrame(
        toc[:limit],
        columns=["level", "title", "pdf_page"],
    )

In [22]:
inspect_toc(asthma_path, limit=40)

,level,title,pdf_page
0,1,Structure Bookmarks,-1
1,2,,1
2,3,,1
3,3,VA/DOD CLINICAL PRACTICE GUIDELINE FOR THE PRI...,1
4,3,,1
5,3,Department of Veterans Affairs,1
6,3,Department of Defense,1
7,3,QUALIFYING STATEMENTS,1
8,3,The Department of Veterans Affairs (VA) and th...,1
9,3,This clinical practice guideline (CPG) is base...,1


In [23]:
inspect_toc(who_path, limit=40)

,level,title,pdf_page
0,1,Acknowledgements,7
1,1,Acronyms and abbreviations,8
2,1,Executive summary,9
3,2,1\tIntroduction,13
4,2,2\tMethod for developing the guideline,15
5,3,2.1\tGuideline contributors,15
6,4,2.2\tAnalytical framework and PICOs,15
7,5,2.3\tOutcome importance rating,16
8,5,2.4\tReviews of evidence,16
9,5,2.5\tCertainty of evidence and strength of rec...,17


In [24]:
inspect_toc(cdc_path, limit=40)

,level,title,pdf_page
0,1,Sexually Transmitted Infections Treatment Guid...,1
1,1,Introduction,3
2,1,Methods,3
3,1,Clinical Prevention Guidance,4
4,1,STI Detection Among Special Populations,13
5,1,HIV Infection,26
6,1,"Diseases Characterized by Genital, Anal, or Pe...",29
7,1,Syphilis,41
8,1,Management of Persons Who Have a History of Pe...,58
9,1,Diseases Characterized by Urethritis and Cervi...,62


### Результат проверки PDF outline

Встроенная структура PDF неодинакова по качеству и не может использоваться одинаково для всех источников.

**WHO.** Bookmarks содержат полезные названия разделов и страницы, однако значения `level` не всегда корректно отражают реальную иерархию. Поэтому названия и границы разделов можно использовать, а иерархию восстанавливать по нумерации заголовков.

**CDC.** Bookmarks корректно определяют крупные тематические разделы, но не содержат полной иерархии внутренних подразделов. Более мелкие headings необходимо определять по форматированию текста.

**VA/DoD.** Встроенный outline содержит большое количество технических и текстовых элементов и не представляет собой надёжное оглавление. Для этих документов структура будет восстанавливаться непосредственно из текста и форматирования.

Таким образом, parser будет использовать общую выходную схему, но разные стратегии определения структуры для WHO, CDC и VA/DoD.

## 6. Извлечение страниц и базовая очистка

Перед определением структуры разделов создадим общий слой извлечения страниц.

Для каждой страницы сохраняются исходный текст и metadata документа. На этом уровне выполняется только очистка PDF-артефактов, которая не зависит от конкретной клинической тематики.

Определение `section_path` будет выполняться после этого отдельной стратегией для каждого семейства документов.

In [25]:
def clean_page_text(text):
    text = re.sub(r"-\n(?=\w)", "-", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [26]:
def extract_page_lines(page):
    page_dict = page.get_text("dict", sort=True)

    lines = []

    for block in page_dict["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            spans = [
                span
                for span in line["spans"]
                if span["text"].strip()
            ]

            if not spans:
                continue

            text = "".join(span["text"] for span in line["spans"]).strip()

            if not text:
                continue

            lines.append(
                {
                    "text": text,
                    "bbox": tuple(round(x, 1) for x in line["bbox"]),
                    "font_size": round(max(span["size"] for span in spans), 1),
                    "fonts": sorted({span["font"] for span in spans}),
                }
            )

    return lines

In [27]:
def extract_document_pages(path):
    metadata = DOCUMENT_METADATA[path.name]
    document_id = path.stem

    pages = []

    with pymupdf.open(path) as doc:
        for page in doc:
            text = page.get_text("text", sort=True)

            pages.append(
                {
                    "document_id": document_id,
                    "source": metadata["source"],
                    "year": metadata["year"],
                    "family": metadata["family"],
                    "page": get_page_number(page),
                    "pdf_page": page.number + 1,
                    "width": round(page.rect.width, 1),
                    "height": round(page.rect.height, 1),
                    "text": clean_page_text(text),
                    "lines": extract_page_lines(page),
                }
            )

    return pages

In [28]:
who_pages = extract_document_pages(who_path)

pd.DataFrame(who_pages[18]["lines"]).tail(15)

,text,bbox,font_size,fonts
25,summaries are presented for the general popula...,"(79.4, 547.9, 526.9, 559.2)",10.0,[Frutiger-Light]
26,"coronary artery disease (CAD), prior stroke) a...","(79.4, 561.9, 522.3, 573.2)",10.0,[Frutiger-Light]
27,thresholds (Web Annex A).,"(79.4, 575.9, 193.5, 587.2)",10.0,[Frutiger-Light]
28,The anticipated benefits of a lower blood pres...,"(79.4, 595.6, 494.1, 606.9)",10.0,[Frutiger-Light]
29,and 130 SBP in a high-risk population) were re...,"(79.4, 609.6, 501.9, 620.9)",10.0,[Frutiger-Light]
30,myocardial infarction (MI) and heart failure e...,"(79.4, 623.6, 497.6, 634.9)",10.0,[Frutiger-Light]
31,"side-effects, and some were a surrogate outcom...","(79.4, 637.6, 510.3, 648.9)",10.0,[Frutiger-Light]
32,"relevant. On average, treatment was associated...","(79.4, 651.6, 509.8, 662.9)",10.0,[Frutiger-Light]
33,that ranged from 5 to 10/1000 and harms that r...,"(79.4, 665.6, 491.7, 676.9)",10.0,[Frutiger-Light]
34,reduction in severe events with significant mo...,"(79.4, 679.6, 518.1, 690.9)",10.0,[Frutiger-Light]


## 7. Parser для WHO

Для WHO структура документа достаточно хорошо представлена встроенными bookmarks, а служебные элементы страницы можно отделить от основного текста по их положению.

Сначала удалим элементы из боковых и нижних полей страницы, сохранив при этом номер страницы отдельно в metadata.

In [29]:
def clean_who_lines(page_data):
    clean_lines = []

    width = page_data["width"]
    height = page_data["height"]

    for line in page_data["lines"]:
        x0, y0, x1, y1 = line["bbox"]
        text = line["text"].strip()

        # Вертикальный footer в правом поле
        if x0 > 0.9 * width:
            continue

        # Номер страницы внизу
        if y0 > 0.95 * height:
            continue

        # PDF-артефакт маркировки
        if text == "[":
            continue

        if text:
            clean_lines.append(
                {
                    **line,
                    "text": text,
                }
            )

    return clean_lines

In [30]:
who_clean_lines = clean_who_lines(who_pages[18])

pd.DataFrame(who_clean_lines)[
    ["text", "bbox", "font_size", "fonts"]
]

,text,bbox,font_size,fonts
0,3\t Recommendations,"(79.4, 52.8, 324.4, 81.7)",25.0,[Frutiger-Bold]
1,3.1\t Blood pressure threshold for initiation ...,"(79.4, 118.6, 496.6, 132.5)",12.0,[Frutiger-Bold]
2,1.\t RECOMMENDATION ON BLOOD PRESSURE THRESHOL...,"(88.1, 152.4, 454.6, 163.9)",10.0,[Frutiger-Bold]
3,PHARMACOLOGICAL TREATMENT,"(102.3, 165.4, 259.8, 176.9)",10.0,[Frutiger-Bold]
4,WHO recommends initiation of pharmacological a...,"(88.1, 190.1, 512.7, 201.7)",10.0,[Frutiger-Roman]
5,with a confirmed diagnosis of hypertension and...,"(88.1, 203.1, 505.9, 215.5)",10.0,"[Frutiger-Roman, Symbol]"
6,diastolic blood pressure of ≥90 mmHg.,"(88.1, 216.1, 267.0, 228.5)",10.0,"[Frutiger-Roman, Symbol]"
7,"Strong recommendation, moderate- to high-certa...","(88.1, 241.0, 381.5, 252.5)",10.0,[Frutiger-Italic]
8,WHO recommends pharmacological antihypertensiv...,"(88.1, 265.7, 517.1, 277.3)",10.0,[Frutiger-Roman]
9,cardiovascular disease and systolic blood pres...,"(88.1, 278.7, 408.8, 290.3)",10.0,[Frutiger-Roman]


In [31]:
who_clean_text = "\n".join(
    line["text"]
    for line in who_clean_lines
)

print(who_clean_text)

3	 Recommendations
3.1	 Blood pressure threshold for initiation of pharmacological treatment
1.	 RECOMMENDATION ON BLOOD PRESSURE THRESHOLD FOR INITIATION OF
PHARMACOLOGICAL TREATMENT
WHO recommends initiation of pharmacological antihypertensive treatment of individuals
with a confirmed diagnosis of hypertension and systolic blood pressure of ≥140 mmHg or
diastolic blood pressure of ≥90 mmHg.
Strong recommendation, moderate- to high-certainty evidence
WHO recommends pharmacological antihypertensive treatment of individuals with existing
cardiovascular disease and systolic blood pressure of 130–139 mmHg.
Strong recommendation, moderate- to high-certainty evidence
WHO suggests pharmacological antihypertensive treatment of individuals without
cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney
disease, and systolic blood pressure of 130–139 mmHg.
Conditional recommendation, moderate- to high-certainty evidence
Implementation remarks:
Initiation o

## 8. Восстановление структуры WHO

Для WHO основные разделы имеют явную числовую иерархию (`3`, `3.1`, `3.2`, ...).

Будем использовать такие нумерованные заголовки для формирования `section_path`. Внутренние элементы вроде `Evidence and rationale` и `Implementation remarks` сохраняются в тексте, но не создают отдельный уровень metadata.

In [32]:
WHO_SECTION_PATTERN = re.compile(
    r"^(\d+(?:\.\d+)*)\s+(.+)$"
)


def get_who_section_heading(line):
    text = re.sub(r"\s+", " ", line["text"]).strip()

    if "Frutiger-Bold" not in line["fonts"]:
        return None

    match = WHO_SECTION_PATTERN.match(text)

    if not match:
        return None

    section_number = match.group(1)
    section_title = match.group(2)

    return {
        "number": section_number,
        "title": section_title,
    }

In [33]:
def merge_who_heading_lines(lines):
    merged = []
    i = 0

    while i < len(lines):
        line = lines[i].copy()

        heading = get_who_section_heading(line)

        # Объединяем только крупные top-level headings:
        # 2 Method for developing the
        # guideline
        if (
            heading is not None
            and "." not in heading["number"]
        ):
            while i + 1 < len(lines):
                next_line = lines[i + 1]

                next_heading = get_who_section_heading(
                    next_line
                )

                same_size = (
                    abs(
                        next_line["font_size"]
                        - line["font_size"]
                    )
                    <= 0.3
                )

                current_bold = any(
                    "bold" in font.lower()
                    for font in line["fonts"]
                )

                next_bold = any(
                    "bold" in font.lower()
                    for font in next_line["fonts"]
                )

                vertical_gap = (
                    next_line["bbox"][1]
                    - line["bbox"][3]
                )

                close_alignment = (
                    next_line["bbox"][0]
                    <= line["bbox"][0] + 60
                )

                if not (
                    next_heading is None
                    and same_size
                    and current_bold
                    and next_bold
                    and vertical_gap <= 5
                    and close_alignment
                ):
                    break

                line["text"] = re.sub(
                    r"\s+",
                    " ",
                    (
                        f'{line["text"]} '
                        f'{next_line["text"]}'
                    ),
                ).strip()

                line["bbox"] = (
                    min(
                        line["bbox"][0],
                        next_line["bbox"][0],
                    ),
                    min(
                        line["bbox"][1],
                        next_line["bbox"][1],
                    ),
                    max(
                        line["bbox"][2],
                        next_line["bbox"][2],
                    ),
                    max(
                        line["bbox"][3],
                        next_line["bbox"][3],
                    ),
                )

                line["fonts"] = sorted(
                    set(line["fonts"])
                    | set(next_line["fonts"])
                )

                i += 1

        merged.append(line)
        i += 1

    return merged

In [34]:
who_clean_lines = clean_who_lines(who_pages[18])

for line in who_clean_lines:
    heading = get_who_section_heading(line)

    if heading:
        print(heading)

{'number': '3', 'title': 'Recommendations'}
{'number': '3.1', 'title': 'Blood pressure threshold for initiation of pharmacological treatment'}


## 9. Структурированный parser WHO

Parser проходит документ последовательно по страницам и поддерживает текущую иерархию разделов.

При появлении нового нумерованного заголовка обновляется `section_path`. Текст между заголовками сохраняется как отдельный структурированный фрагмент вместе с информацией об источнике и странице.

Это ещё не финальные chunks для retrieval: сначала проверяется корректность структуры документа.

In [35]:
def build_section_path(section_stack):
    return " > ".join(
        section_stack[level]
        for level in sorted(section_stack)
    )

In [36]:
def get_references_start_page(path):
    with pymupdf.open(path) as doc:
        toc = doc.get_toc()

    for level, title, pdf_page in toc:
        if title.strip().lower() == "references":
            return pdf_page

    return None

In [37]:
def get_who_body_start_page(path):
    with pymupdf.open(path) as doc:
        toc = doc.get_toc()

    section_pages = []

    for level, title, pdf_page in toc:
        title = re.sub(
            r"\s+",
            " ",
            title,
        ).strip()

        match = WHO_SECTION_PATTERN.match(title)

        if not match:
            continue

        section_number = match.group(1)

        # Нас интересуют только разделы верхнего уровня:
        # 1 Introduction, 2 Method..., 3 Recommendations...
        if "." not in section_number:
            section_pages.append(pdf_page)

    if not section_pages:
        return None

    return min(section_pages)

In [38]:
get_who_body_start_page(who_path)

13

In [39]:
def parse_who(path):
    pages = extract_document_pages(path)
    metadata = DOCUMENT_METADATA[path.name]

    body_start = get_who_body_start_page(path)
    references_start = get_references_start_page(path)

    section_stack = {}
    records = []
    started = False

    for page_data in pages:
        # Не читаем title pages / licence / contents.
        if (
            body_start is not None
            and page_data["pdf_page"] < body_start
        ):
            continue

        if (
            references_start is not None
            and page_data["pdf_page"] >= references_start
        ):
            break

        lines = clean_who_lines(page_data)

        lines = merge_who_heading_lines(
            lines
        )

        current_text = []

        def flush_text():
            if not current_text:
                return

            if not started:
                current_text.clear()
                return

            text = clean_page_text(
                "\n".join(current_text)
            )

            if text:
                records.append(
                    {
                        "document_id": page_data["document_id"],
                        "source": metadata["source"],
                        "document_title": metadata["document_title"],
                        "year": metadata["year"],
                        "section_path": build_section_path(
                            section_stack
                        ),
                        "page": page_data["page"],
                        "pdf_page": page_data["pdf_page"],
                        "text": text,
                    }
                )

            current_text.clear()

        for line in lines:
            heading = get_who_section_heading(line)

            if heading:
                flush_text()

                started = True

                depth = len(
                    heading["number"].split(".")
                )

                section_stack[depth] = (
                    f'{heading["number"]} '
                    f'{heading["title"]}'
                )

                for level in list(section_stack):
                    if level > depth:
                        del section_stack[level]

                continue

            current_text.append(
                line["text"]
            )

        flush_text()

    return records

In [40]:
who_records = parse_who(who_path)

len(who_records)

42

In [41]:
who_records_df = pd.DataFrame(who_records)

who_records_df[
    who_records_df["section_path"].str.contains(
        "Blood pressure threshold",
        case=False,
        na=False,
    )
][
    ["page", "pdf_page", "section_path", "text"]
]

,page,pdf_page,section_path,text
9,7,19,3 Recommendations > 3.1 Blood pressure thresho...,1.\t RECOMMENDATION ON BLOOD PRESSURE THRESHOL...
10,8,20,3 Recommendations > 3.1 Blood pressure thresho...,Evidence-to-decision considerations\nThe value...


In [42]:
record = who_records_df[
    who_records_df["section_path"].str.contains(
        "Blood pressure threshold",
        case=False,
        na=False,
    )
].iloc[0]

print("page:", record["page"])
print("pdf_page:", record["pdf_page"])
print("section:", record["section_path"])
print()
print(record["text"][:3000])

page: 7
pdf_page: 19
section: 3 Recommendations > 3.1 Blood pressure threshold for initiation of pharmacological treatment

1.	 RECOMMENDATION ON BLOOD PRESSURE THRESHOLD FOR INITIATION OF
PHARMACOLOGICAL TREATMENT
WHO recommends initiation of pharmacological antihypertensive treatment of individuals
with a confirmed diagnosis of hypertension and systolic blood pressure of ≥140 mmHg or
diastolic blood pressure of ≥90 mmHg.
Strong recommendation, moderate- to high-certainty evidence
WHO recommends pharmacological antihypertensive treatment of individuals with existing
cardiovascular disease and systolic blood pressure of 130–139 mmHg.
Strong recommendation, moderate- to high-certainty evidence
WHO suggests pharmacological antihypertensive treatment of individuals without
cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney
disease, and systolic blood pressure of 130–139 mmHg.
Conditional recommendation, moderate- to high-certainty evidence
Imple

## 10. Parser для CDC

CDC содержит полезные bookmarks для крупных тематических разделов. Внутренние подразделы, например `Follow-Up` и `Management of Sex Partners`, выделяются шрифтом `MyriadPro-Semibold`.

Перед построением parser необходимо проверить порядок извлечения текста на странице, поскольку журнальная верстка может содержать несколько колонок.

In [43]:
cdc_pages = extract_document_pages(cdc_path)

cdc_page = cdc_pages[66]  # pdf_page = 67

pd.DataFrame(cdc_page["lines"])[
    ["text", "bbox", "font_size", "fonts"]
].head(40)

,text,bbox,font_size,fonts
0,Recommendations and Reports,"(246.9, 34.2, 365.1, 45.0)",9.0,[MyriadPro-Regular]
1,reduces HIV shedding from the cervix and there...,"(318.6, 69.9, 578.9, 83.9)",11.0,[AGaramondPro-Regular]
2,reduce HIV transmission to susceptible sex par...,"(318.6, 82.9, 576.0, 96.9)",11.0,"[AGaramondPro-Italic, AGaramondPro-Regular]"
3,Pregnancy,"(318.6, 102.5, 369.2, 115.9)",11.0,[MyriadPro-Semibold]
4,Diagnosis and treatment of cervicitis for preg...,"(327.6, 119.7, 578.8, 133.7)",11.0,[AGaramondPro-Regular]
5,does not differ from that for women who are no...,"(318.6, 132.7, 578.8, 146.7)",11.0,[AGaramondPro-Regular]
6,Diagnostic Considerations; Treatment).,"(318.6, 145.7, 486.3, 159.7)",11.0,[AGaramondPro-Regular]
7,cervicitis has resolved. Women with a specific...,"(36.0, 69.9, 296.1, 83.9)",11.0,[AGaramondPro-Regular]
8,"chlamydia, gonorrhea, or trichomoniasis should...","(36.0, 82.9, 296.1, 96.9)",11.0,[AGaramondPro-Regular]
9,partner services and instructed to return in 3...,"(36.0, 95.9, 296.2, 109.9)",11.0,[AGaramondPro-Regular]


### Порядок чтения CDC

Документ CDC использует двухколоночную верстку. Стандартное извлечение с `sort=True` не всегда восстанавливает правильный порядок чтения и может чередовать строки из левой и правой колонок.

Поэтому для CDC порядок текста восстанавливается явно: сначала строки левой колонки сверху вниз, затем строки правой колонки сверху вниз. Служебные элементы header/footer удаляются отдельно.

In [44]:
def clean_and_order_cdc_lines(page_data):
    clean_lines = []

    width = page_data["width"]
    height = page_data["height"]
    middle = width / 2

    for line in page_data["lines"]:
        x0, y0, x1, y1 = line["bbox"]

        text = re.sub(r"\s+", " ", line["text"]).strip()

        if not text:
            continue

        # Повторяющиеся элементы MMWR
        if text == "Recommendations and Reports":
            continue

        if text.startswith("MMWR"):
            continue

        if text.startswith(
            "US Department of Health and Human Services"
        ):
            continue

        # Номер страницы в header/footer
        if (
            re.fullmatch(r"\d+", text)
            and (y0 < 60 or y0 > 0.9 * height)
        ):
            continue

        column = "left" if x0 < middle else "right"

        clean_lines.append(
            {
                **line,
                "text": text,
                "column": column,
            }
        )

    clean_lines.sort(
        key=lambda line: (
            0 if line["column"] == "left" else 1,
            line["bbox"][1],
            line["bbox"][0],
        )
    )

    return clean_lines

In [45]:
cdc_clean_lines = clean_and_order_cdc_lines(
    cdc_pages[66]
)

pd.DataFrame(cdc_clean_lines)[
    ["text", "column", "bbox", "font_size", "fonts"]
].head(50)

,text,column,bbox,font_size,fonts
0,cervicitis has resolved. Women with a specific...,left,"(36.0, 69.9, 296.1, 83.9)",11.0,[AGaramondPro-Regular]
1,"chlamydia, gonorrhea, or trichomoniasis should...",left,"(36.0, 82.9, 296.1, 96.9)",11.0,[AGaramondPro-Regular]
2,partner services and instructed to return in 3...,left,"(36.0, 95.9, 296.2, 109.9)",11.0,[AGaramondPro-Regular]
3,treatment for repeat testing because of high r...,left,"(36.0, 108.9, 295.9, 122.9)",11.0,[AGaramondPro-Regular]
4,regardless of whether their sex partners were ...,left,"(36.0, 121.9, 296.1, 135.9)",11.0,"[AGaramondPro-Italic, AGaramondPro-Regular]"
5,"symptoms persist or recur, women should be ins...",left,"(36.0, 134.9, 296.0, 148.9)",11.0,[AGaramondPro-Regular]
6,return for reevaluation.,left,"(36.0, 147.9, 134.3, 161.9)",11.0,[AGaramondPro-Regular]
7,Management of Sex Partners,left,"(36.0, 167.5, 185.5, 182.2)",12.0,[MyriadPro-Semibold]
8,Management of sex partners of women treated fo...,left,"(45.0, 185.7, 296.1, 199.7)",11.0,[AGaramondPro-Regular]
9,should be tailored for the specific infection ...,left,"(36.0, 198.7, 296.2, 212.7)",11.0,[AGaramondPro-Regular]


Порядок чтения после явного разделения на колонки восстановлен корректно.

На одной физической странице могут одновременно находиться конец предыдущего раздела и начало нового. Поэтому `section_path` должен обновляться в момент появления заголовка внутри последовательности строк, а не назначаться целиком по номеру страницы.

In [46]:
def get_cdc_major_headings(path):
    with pymupdf.open(path) as doc:
        toc = doc.get_toc()

    return {
        (
            pdf_page,
            re.sub(r"\s+", " ", title).strip(),
        )
        for level, title, pdf_page in toc
        if level == 1
        and title.strip()
        and title != "References"
        and not title.startswith(
            "Sexually Transmitted Infections Treatment Guidelines"
        )
    }


CDC_MAJOR_HEADINGS = get_cdc_major_headings(cdc_path)

In [47]:
def normalize_cdc_text(text):
    return re.sub(r"\s+", " ", text).strip()


CDC_MAJOR_BY_PAGE = {}

for pdf_page, title in CDC_MAJOR_HEADINGS:
    CDC_MAJOR_BY_PAGE.setdefault(
        pdf_page,
        set(),
    ).add(title)

In [48]:
page_82_lines = clean_and_order_cdc_lines(
    cdc_pages[81]
)

[
    line
    for line in page_82_lines
    if "mycoplasma" in line["text"].lower()
    or "genitalium" in line["text"].lower()
]

[{'text': 'Mycoplasma genitalium',
  'bbox': (366.8, 360.0, 527.8, 379.5),
  'font_size': 16.0,
  'fonts': ['MyriadPro-SemiboldIt'],
  'column': 'right'},
 {'text': 'M. genitalium causes symptomatic and asymptomatic',
  'bbox': (327.6, 382.0, 578.7, 396.0),
  'font_size': 11.0,
  'fonts': ['AGaramondPro-Italic', 'AGaramondPro-Regular'],
  'column': 'right'},
 {'text': 'areas (911–913), although M. genitalium is often the sole',
  'bbox': (318.6, 447.0, 578.8, 461.0),
  'font_size': 11.0,
  'fonts': ['AGaramondPro-Italic', 'AGaramondPro-Regular'],
  'column': 'right'},
 {'text': 'pathogen. Data are insufficient to implicate M. genitalium',
  'bbox': (318.6, 460.0, 578.8, 474.0),
  'font_size': 11.0,
  'fonts': ['AGaramondPro-Italic', 'AGaramondPro-Regular'],
  'column': 'right'},
 {'text': 'of asymptomatic infection with M. genitalium among men',
  'bbox': (318.6, 499.0, 578.7, 513.0),
  'font_size': 11.0,
  'fonts': ['AGaramondPro-Italic', 'AGaramondPro-Regular'],
  'column': 'right'},

In [49]:
def is_cdc_semibold(line):
    return any(
        font.startswith("MyriadPro-Semibold")
        for font in line["fonts"]
    )

In [50]:
def match_cdc_major_heading(
    lines,
    start,
    pdf_page,
    max_lines=6,
):
    targets = CDC_MAJOR_BY_PAGE.get(
        pdf_page,
        set(),
    )

    if not targets:
        return None

    first_column = lines[start]["column"]
    parts = []

    for j in range(
        start,
        min(start + max_lines, len(lines)),
    ):
        line = lines[j]

        if line["column"] != first_column:
            break

        if not is_cdc_semibold(line):
            break

        parts.append(line["text"])

        candidate = normalize_cdc_text(
            " ".join(parts)
        )

        if candidate in targets:
            return {
                "title": candidate,
                "end": j + 1,
            }

        if not any(
            target.startswith(candidate)
            for target in targets
        ):
            break

    return None

In [51]:
def merge_cdc_heading_lines(lines, pdf_page):
    merged = []
    i = 0

    while i < len(lines):
        major_heading = match_cdc_major_heading(
            lines,
            i,
            pdf_page,
        )

        if major_heading is not None:
            line = lines[i].copy()
            line["text"] = major_heading["title"]
            line["heading_level"] = 1

            merged.append(line)
            i = major_heading["end"]
            continue

        line = lines[i].copy()

        if not is_cdc_semibold(line):
            merged.append(line)
            i += 1
            continue

        size = line["font_size"]

        if size >= 13.5:
            level = 2
        elif size >= 11.5:
            level = 3
        elif size >= 10.5:
            level = 4
        else:
            merged.append(line)
            i += 1
            continue

        parts = [line["text"]]
        current_bottom = line["bbox"][3]
        column = line["column"]

        j = i + 1

        while j < len(lines):
            next_line = lines[j]

            if next_line["column"] != column:
                break

            if not is_cdc_semibold(next_line):
                break

            next_size = next_line["font_size"]

            if level == 2:
                same_level = next_size >= 13.5
            elif level == 3:
                same_level = 11.5 <= next_size < 13.5
            else:
                same_level = 10.5 <= next_size < 11.5

            if not same_level:
                break

            vertical_gap = (
                next_line["bbox"][1]
                - current_bottom
            )

            if vertical_gap > 5:
                break

            parts.append(next_line["text"])
            current_bottom = next_line["bbox"][3]
            j += 1

        line["text"] = normalize_cdc_text(
            " ".join(parts)
        )
        line["heading_level"] = level

        merged.append(line)
        i = j

    return merged

In [52]:
cdc_structured_lines = merge_cdc_heading_lines(
    cdc_clean_lines,
    cdc_pages[66]["pdf_page"],
)

for line in cdc_structured_lines:
    if line.get("heading_level") is not None:
        print(
            line["heading_level"],
            "|",
            line["text"],
        )

3 | Management of Sex Partners
3 | Persistent or Recurrent Cervicitis
3 | Special Considerations
4 | HIV Infection
4 | Pregnancy
4 | Contraceptive Management
1 | Chlamydial Infections
2 | Chlamydial Infection Among Adolescents and Adults


In [53]:
def parse_cdc(path):
    pages = extract_document_pages(path)
    metadata = DOCUMENT_METADATA[path.name]

    references_start = get_references_start_page(path)

    section_stack = {}
    records = []
    started = False

    for page_data in pages:
        if (
            references_start is not None
            and page_data["pdf_page"] >= references_start
        ):
            break

        lines = clean_and_order_cdc_lines(
            page_data
        )

        lines = merge_cdc_heading_lines(
            lines,
            page_data["pdf_page"],
        )

        current_text = []

        def flush_text():
            if not current_text:
                return

            if not started:
                current_text.clear()
                return

            text = " ".join(current_text)

            text = re.sub(
                r"\s+",
                " ",
                text,
            ).strip()

            text = re.sub(
                r"-\s+(?=\w)",
                "-",
                text,
            )

            if text:
                records.append(
                    {
                        "document_id": page_data["document_id"],
                        "source": metadata["source"],
                        "document_title": metadata["document_title"],
                        "year": metadata["year"],
                        "section_path": build_section_path(
                            section_stack
                        ),
                        "page": page_data["page"],
                        "pdf_page": page_data["pdf_page"],
                        "text": text,
                    }
                )

            current_text.clear()

        for line in lines:
            level = line.get("heading_level")

            if level is not None:
                flush_text()

                started = True

                section_stack[level] = line["text"]

                for existing_level in list(
                    section_stack
                ):
                    if existing_level > level:
                        del section_stack[
                            existing_level
                        ]

                continue

            current_text.append(
                line["text"]
            )

        flush_text()

    return records

In [54]:
cdc_records = parse_cdc(cdc_path)

len(cdc_records)

599

In [55]:
cdc_records_df = pd.DataFrame(cdc_records)

cdc_records_df[
    cdc_records_df["pdf_page"].isin([66, 67, 68])
][
    ["page", "pdf_page", "section_path", "text"]
]

,page,pdf_page,section_path,text
292,64,66,Diseases Characterized by Urethritis and Cervi...,2) sustained endocervical bleeding easily indu...
293,64,66,Diseases Characterized by Urethritis and Cervi...,C. trachomatis or N. gonorrhoeae is the most c...
294,64,66,Diseases Characterized by Urethritis and Cervi...,Because cervicitis might be a sign of upper ge...
295,64,66,Diseases Characterized by Urethritis and Cervi...,Multiple factors should affect the decision to...
296,64,66,Diseases Characterized by Urethritis and Cervi...,Doxycycline 100 mg orally 2 times/day for 7 da...
297,64,66,Diseases Characterized by Urethritis and Cervi...,"To minimize transmission and reinfection, wome..."
298,64,66,Diseases Characterized by Urethritis and Cervi...,Women receiving treatment should return to the...
299,65,67,Diseases Characterized by Urethritis and Cervi...,cervicitis has resolved. Women with a specific...
300,65,67,Diseases Characterized by Urethritis and Cervi...,Management of sex partners of women treated fo...
301,65,67,Diseases Characterized by Urethritis and Cervi...,Women with persistent or recurrent cervicitis ...


In [56]:
cdc_records_df[
    cdc_records_df["section_path"].str.contains(
        "Chlamydial Infections",
        case=False,
        na=False,
    )
][
    ["page", "pdf_page", "section_path"]
].drop_duplicates().head(20)

,page,pdf_page,section_path
305,65,67,Chlamydial Infections > Chlamydial Infection A...
306,66,68,Chlamydial Infections > Chlamydial Infection A...
307,66,68,Chlamydial Infections > Chlamydial Infection A...
308,66,68,Chlamydial Infections > Chlamydial Infection A...
309,67,69,Chlamydial Infections > Chlamydial Infection A...
310,67,69,Chlamydial Infections > Chlamydial Infection A...
311,67,69,Chlamydial Infections > Chlamydial Infection A...
312,67,69,Chlamydial Infections > Chlamydial Infection A...
313,68,70,Chlamydial Infections > Chlamydial Infection A...
314,68,70,Chlamydial Infections > Chlamydial Infection A...


### Проверка структуры CDC

Крупные тематические разделы CDC присутствуют во встроенных bookmarks PDF. Используем их как независимую проверку font-based parser: основные разделы, найденные по форматированию текста, должны в целом совпадать с bookmarks документа.

In [57]:
parsed_cdc_level1 = []

for page_data in cdc_pages:
    lines = clean_and_order_cdc_lines(page_data)

    lines = merge_cdc_heading_lines(
        lines,
        page_data["pdf_page"],
    )

    for line in lines:
        if line.get("heading_level") == 1:
            parsed_cdc_level1.append(
                (
                    page_data["pdf_page"],
                    line["text"],
                )
            )

parsed_cdc_level1 = set(parsed_cdc_level1)

print("Missing:")
print(CDC_MAJOR_HEADINGS - parsed_cdc_level1)

print("\nExtra:")
print(parsed_cdc_level1 - CDC_MAJOR_HEADINGS)

Missing:
set()

Extra:
set()


In [58]:
pd.set_option("display.max_colwidth", None)
cdc_records_df[
    cdc_records_df["pdf_page"].isin([66, 67, 68])
][
    ["page", "pdf_page", "section_path"]
].drop_duplicates()

,page,pdf_page,section_path
292,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis
293,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Etiology
294,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Diagnostic Considerations
295,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Treatment
296,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Treatment > Recommended Regimen for Cervicitis*
297,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Other Management Considerations
298,64,66,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Follow-Up
299,65,67,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Follow-Up
300,65,67,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Management of Sex Partners
301,65,67,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Persistent or Recurrent Cervicitis


## 11. Parser для VA/DoD

В отличие от WHO и CDC, встроенные bookmarks документов VA/DoD не дают надёжной иерархии разделов.

Кроме того, корпус содержит несколько рекомендаций разных лет, поэтому нельзя заранее предполагать, что все документы используют абсолютно одинаковое форматирование.

Сначала проверим, как оформлены основные нумерованные разделы во всех шести документах, и только после этого сформулируем правила parser.

In [59]:
va_dod_paths = sorted(
    (GUIDELINES_DIR / "va_dod").glob("*.pdf")
)

for path in va_dod_paths:
    print(path.name)

va_dod_asthma_2025.pdf
va_dod_ckd_2025.pdf
va_dod_low_back_pain_2022.pdf
va_dod_major_depression_2022.pdf
va_dod_pregnancy_2023.pdf
va_dod_type2_diabetes_2023.pdf


In [60]:
VA_DOD_ROMAN_PATTERN = re.compile(
    r"^"
    r"(?=[MDCLXVI]+\.)"
    r"M{0,4}"
    r"(?:CM|CD|D?C{0,3})"
    r"(?:XC|XL|L?X{0,3})"
    r"(?:IX|IV|V?I{0,3})"
    r"\.\s+\S"
)


def collect_va_dod_section_candidates(path):
    rows = []

    with pymupdf.open(path) as doc:
        for page in doc:
            lines = extract_page_lines(page)

            for line in lines:
                text = re.sub(
                    r"\s+",
                    " ",
                    line["text"],
                ).strip()

                if not VA_DOD_ROMAN_PATTERN.match(text):
                    continue

                # Строки Table of Contents с dot leaders
                if re.search(r"\.{4,}", text):
                    continue

                rows.append(
                    {
                        "file": path.name,
                        "pdf_page": page.number + 1,
                        "text": text,
                        "font_size": line["font_size"],
                        "fonts": line["fonts"],
                        "bbox": line["bbox"],
                    }
                )

    return rows

In [61]:
va_dod_section_candidates = []

for path in va_dod_paths:
    va_dod_section_candidates.extend(
        collect_va_dod_section_candidates(path)
    )

va_dod_sections_df = pd.DataFrame(
    va_dod_section_candidates
)

pd.set_option("display.max_colwidth", None)


In [62]:
va_dod_sections_df.groupby(
    ["file", "font_size"]
).size().reset_index(name="count")

,file,font_size,count
0,va_dod_asthma_2025.pdf,11.0,1
1,va_dod_asthma_2025.pdf,13.0,12
2,va_dod_asthma_2025.pdf,14.0,5
3,va_dod_ckd_2025.pdf,11.0,2
4,va_dod_ckd_2025.pdf,13.0,14
5,va_dod_ckd_2025.pdf,14.0,6
6,va_dod_low_back_pain_2022.pdf,10.4,1
7,va_dod_low_back_pain_2022.pdf,12.0,2
8,va_dod_low_back_pain_2022.pdf,14.4,4
9,va_dod_major_depression_2022.pdf,10.0,1


### Результат анализа VA/DoD

Несмотря на различия между версиями документов, основные уровни структуры оформлены достаточно последовательно.

Крупные разделы документа (`III. Scope`, `VIII. Algorithm`, `IX. Recommendations` и т.д.) имеют жирное начертание и размер около 14 pt.

Внутренние буквенные подразделы (`A.`, `B.`, `C.` и т.д.) также выделены жирным шрифтом, но имеют меньший размер — обычно 12–13 pt.

Конкретные семейства шрифтов различаются между версиями документов, поэтому название шрифта не используется как основной признак. Для определения структуры будут использоваться сочетание размера, жирного начертания и формы нумерации.

In [63]:
def is_va_dod_bold(line):
    return any(
        "bold" in font.lower()
        for font in line["fonts"]
    )

In [64]:
VA_DOD_MAJOR_PATTERN = re.compile(
    r"^"
    r"(?=[MDCLXVI]+\.)"
    r"M{0,4}"
    r"(?:CM|CD|D?C{0,3})"
    r"(?:XC|XL|L?X{0,3})"
    r"(?:IX|IV|V?I{0,3})"
    r"\.\s+\S"
)

VA_DOD_APPENDIX_PATTERN = re.compile(
    r"^Appendix\s+[A-Z]\s*[:.]\s*\S",
    flags=re.IGNORECASE,
)

VA_DOD_SUBSECTION_PATTERN = re.compile(
    r"^[A-Z]\.\s+\S"
)

In [65]:
def get_va_dod_heading_level(line):
    text = re.sub(
        r"\s+",
        " ",
        line["text"],
    ).strip()

    if not is_va_dod_bold(line):
        return None

    size = line["font_size"]

    # Основные римские разделы:
    # I. Introduction, IX. Recommendations ...
    if (
        size >= 13.8
        and VA_DOD_MAJOR_PATTERN.match(text)
    ):
        return 1

    # Appendices являются самостоятельными
    # разделами верхнего уровня
    if (
        size >= 13.8
        and VA_DOD_APPENDIX_PATTERN.match(text)
    ):
        return 1

    # Буквенные подразделы:
    # A. Background, B. Treatment ...
    if (
        11.8 <= size < 13.8
        and VA_DOD_SUBSECTION_PATTERN.match(text)
    ):
        return 2

    return None

In [66]:
def normalize_va_dod_text(text):
    return re.sub(r"\s+", " ", text).strip()

In [67]:
def merge_va_dod_inline_labels(lines):
    merged = []
    i = 0

    def is_standalone_subsection_label(text):
        return bool(
            re.fullmatch(
                r"[A-Z]\.",
                text.strip(),
            )
        )

    def is_standalone_roman_label(text):
        text = text.strip()

        if not text.endswith("."):
            return False

        roman = text[:-1]

        return bool(
            re.fullmatch(
                r"(?=[MDCLXVI]+$)"
                r"M{0,4}"
                r"(?:CM|CD|D?C{0,3})"
                r"(?:XC|XL|L?X{0,3})"
                r"(?:IX|IV|V?I{0,3})",
                roman,
            )
        )

    while i < len(lines):
        current = lines[i]

        if i + 1 >= len(lines):
            merged.append(current)
            break

        next_line = lines[i + 1]

        current_text = current["text"].strip()
        next_text = next_line["text"].strip()

        current_bbox = current["bbox"]
        next_bbox = next_line["bbox"]

        current_x0, current_y0, current_x1, current_y1 = (
            current_bbox
        )
        next_x0, next_y0, next_x1, next_y1 = (
            next_bbox
        )

        vertical_overlap = (
            min(current_y1, next_y1)
            - max(current_y0, next_y0)
        )

        horizontal_gap = (
            next_x0 - current_x1
        )

        same_visual_line = (
            vertical_overlap > 0
            and -2 <= horizontal_gap <= 40
        )

        both_bold = (
            is_va_dod_bold(current)
            and is_va_dod_bold(next_line)
        )

        is_subsection_label = (
            is_standalone_subsection_label(
                current_text
            )
        )

        is_roman_label = (
            is_standalone_roman_label(
                current_text
            )
        )

        should_merge = (
            same_visual_line
            and both_bold
            and (
                is_subsection_label
                or is_roman_label
            )
        )

        if should_merge:
            combined = current.copy()

            combined["text"] = (
                f"{current_text} {next_text}"
            )

            combined["bbox"] = (
                min(current_x0, next_x0),
                min(current_y0, next_y0),
                max(current_x1, next_x1),
                max(current_y1, next_y1),
            )

            combined["fonts"] = list(
                dict.fromkeys(
                    current["fonts"]
                    + next_line["fonts"]
                )
            )

            # Для Roman major heading вроде:
            # II. + Background
            # IX. + Recommendations
            #
            # берём размер основного текста heading.
            #
            # Для subsection вроде:
            # C. + Diabetes Mellitus
            # F. + Guideline Development Team
            #
            # сохраняем размер буквенного label,
            # чтобы heading остался level 2.
            if (
                is_roman_label
                and next_line["font_size"] >= 13.8
            ):
                combined["font_size"] = max(
                    current["font_size"],
                    next_line["font_size"],
                )
            else:
                combined["font_size"] = (
                    current["font_size"]
                )

            merged.append(combined)

            i += 2
            continue

        merged.append(current)
        i += 1

    return merged

In [68]:
def merge_va_dod_heading_lines(lines):
    lines = merge_va_dod_inline_labels(lines)

    merged = []
    i = 0

    while i < len(lines):
        line = lines[i].copy()

        level = get_va_dod_heading_level(
            line
        )

        if level is None:
            merged.append(line)
            i += 1
            continue

        heading_x0 = line["bbox"][0]

        is_appendix = bool(
            VA_DOD_APPENDIX_PATTERN.match(
                normalize_va_dod_text(
                    line["text"]
                )
            )
        )

        i += 1

        while i < len(lines):
            next_line = lines[i]

            # Если следующая строка сама является
            # новым heading, текущий heading закончен.
            if (
                get_va_dod_heading_level(
                    next_line
                )
                is not None
            ):
                break

            if not is_va_dod_bold(
                next_line
            ):
                break

            same_size = (
                abs(
                    next_line["font_size"]
                    - line["font_size"]
                )
                <= 0.3
            )

            if not same_size:
                break

            vertical_gap = (
                next_line["bbox"][1]
                - line["bbox"][3]
            )

            if vertical_gap > 6:
                break

            # В VA/DoD продолжения Appendix headings
            # имеют больший отступ вправо.
            max_indent = (
                100
                if is_appendix
                else 60
            )

            next_x0 = next_line["bbox"][0]

            if not (
                heading_x0 - 5
                <= next_x0
                <= heading_x0 + max_indent
            ):
                break

            line["text"] = (
                f'{normalize_va_dod_text(line["text"])} '
                f'{normalize_va_dod_text(next_line["text"])}'
            )

            line["bbox"] = (
                min(
                    line["bbox"][0],
                    next_line["bbox"][0],
                ),
                min(
                    line["bbox"][1],
                    next_line["bbox"][1],
                ),
                max(
                    line["bbox"][2],
                    next_line["bbox"][2],
                ),
                max(
                    line["bbox"][3],
                    next_line["bbox"][3],
                ),
            )

            line["fonts"] = sorted(
                set(line["fonts"])
                | set(next_line["fonts"])
            )

            i += 1

        line["heading_level"] = level
        merged.append(line)

    return merged

In [69]:
lbp_pages = extract_document_pages(
    GUIDELINES_DIR
    / "va_dod"
    / "va_dod_low_back_pain_2022.pdf"
)

lbp_lines = merge_va_dod_heading_lines(
    lbp_pages[14]["lines"]
)

for line in lbp_lines:
    if line.get("heading_level") is not None:
        print(
            line["heading_level"],
            "|",
            line["text"],
        )

2 | D. Patient Perspective
2 | E. External Peer Review
2 | F. Implementation
1 | VII. Approach to Care in Department of Veterans Affairs and Department of Defense
2 | A. Patient-centered Care


In [70]:
asthma_pages = extract_document_pages(
    GUIDELINES_DIR
    / "va_dod"
    / "va_dod_asthma_2025.pdf"
)

asthma_lines = merge_va_dod_heading_lines(
    asthma_pages[7]["lines"]
)

for line in asthma_lines:
    if line.get("heading_level") is not None:
        print(
            line["heading_level"],
            "|",
            line["text"],
        )

2 | C. Epidemiology and Impact in the General Population
2 | D. Asthma in the Department of Defense and the Department of Veterans Affairs Populations


### Служебные элементы VA/DoD

Перед формированием структурированных records необходимо удалить повторяющиеся headers и footers.

Поскольку документы VA/DoD относятся к разным версиям шаблона, не будем заранее задавать конкретные строки вручную. Сначала проверим, какие элементы регулярно встречаются в верхней и нижней частях страниц.

In [71]:
def collect_va_dod_margin_texts(path):
    top_texts = Counter()
    bottom_texts = Counter()

    pages = extract_document_pages(path)

    for page_data in pages:
        height = page_data["height"]

        for line in page_data["lines"]:
            text = normalize_va_dod_text(
                line["text"]
            )

            y0 = line["bbox"][1]

            if y0 < 0.08 * height:
                top_texts[text] += 1

            if y0 > 0.92 * height:
                bottom_texts[text] += 1

    return top_texts, bottom_texts

In [72]:
def get_va_dod_margin_noise(path, min_fraction=0.25):
    top_texts, bottom_texts = collect_va_dod_margin_texts(path)

    with pymupdf.open(path) as doc:
        n_pages = len(doc)

    min_count = max(
        3,
        int(n_pages * min_fraction),
    )

    noise = {
        text
        for text, count in top_texts.items()
        if count >= min_count
    }

    noise.update(
        text
        for text, count in bottom_texts.items()
        if count >= min_count
    )

    return noise

In [73]:
def clean_va_dod_lines(page_data, margin_noise):
    clean_lines = []

    height = page_data["height"]

    for line in page_data["lines"]:
        text = normalize_va_dod_text(
            line["text"]
        )

        if not text:
            continue

        y0 = line["bbox"][1]

        in_margin = (
            y0 < 0.08 * height
            or y0 > 0.92 * height
        )

        # Повторяющийся header/footer
        if in_margin and text in margin_noise:
            continue

        # Уникальный номер каждой страницы:
        # Page 31 of 149 и т.п.
        if (
            y0 > 0.90 * height
            and re.fullmatch(
                r"Page\s+\d+\s+of\s+\d+",
                text,
                flags=re.IGNORECASE,
            )
        ):
            continue

        clean_lines.append(
            {
                **line,
                "text": text,
            }
        )

    return clean_lines

In [74]:
for path in va_dod_paths:
    margin_noise = get_va_dod_margin_noise(path)

    print("\n", path.name)

    for text in sorted(margin_noise):
        print("REMOVE:", text)


 va_dod_asthma_2025.pdf
REMOVE: March 2025
REMOVE: VA/DOD Clinical Practice Guideline for the Primary Care Management of Asthma

 va_dod_ckd_2025.pdf
REMOVE: April 2025
REMOVE: VA/DOD Clinical Practice Guideline for the Primary Care Management of Chronic Kidney Disease

 va_dod_low_back_pain_2022.pdf
REMOVE: February 2022
REMOVE: VA/DoD Clinical Practice Guideline for the Diagnosis and Treatment of Low Back Pain

 va_dod_major_depression_2022.pdf
REMOVE: February 2022
REMOVE: VA/DoD Clinical Practice Guideline for the Management of Major Depressive Disorder

 va_dod_pregnancy_2023.pdf
REMOVE: July 2023
REMOVE: VA/DoD Clinical Practice Guideline for Management of Pregnancy
REMOVE: VA/DoD Clinical Practice Guideline for the Management of Pregnancy

 va_dod_type2_diabetes_2023.pdf
REMOVE: May 2023
REMOVE: VA/DoD Clinical Practice Guideline for Management of Type 2 Diabetes Mellitus
REMOVE: VA/DoD Clinical Practice Guideline for the Management of Type 2 Diabetes Mellitus


In [75]:
remaining_page_markers = []

for path in va_dod_paths:
    margin_noise = get_va_dod_margin_noise(path)
    pages = extract_document_pages(path)

    for page_data in pages:
        lines = clean_va_dod_lines(
            page_data,
            margin_noise,
        )

        for line in lines:
            if re.fullmatch(
                r"Page\s+\d+\s+of\s+\d+",
                line["text"],
                flags=re.IGNORECASE,
            ):
                remaining_page_markers.append(
                    (
                        path.name,
                        page_data["pdf_page"],
                        line["text"],
                    )
                )

remaining_page_markers

[]

In [76]:
def get_va_dod_references_start_page(path):
    with pymupdf.open(path) as doc:
        for page in doc:
            lines = extract_page_lines(page)

            for line in lines:
                text = normalize_va_dod_text(
                    line["text"]
                )

                x0, y0, x1, y1 = line["bbox"]

                if (
                    text.lower() == "references"
                    and is_va_dod_bold(line)
                    and line["font_size"] >= 13.8
                    and x0 < 150
                    and y0 < 120
                ):
                    return page.number + 1

    return None

In [77]:
va_dod_references_pages = []

for path in va_dod_paths:
    va_dod_references_pages.append(
        {
            "file": path.name,
            "references_pdf_page": (
                get_va_dod_references_start_page(path)
            ),
        }
    )

pd.DataFrame(va_dod_references_pages)

,file,references_pdf_page
0,va_dod_asthma_2025.pdf,142
1,va_dod_ckd_2025.pdf,174
2,va_dod_low_back_pain_2022.pdf,125
3,va_dod_major_depression_2022.pdf,144
4,va_dod_pregnancy_2023.pdf,163
5,va_dod_type2_diabetes_2023.pdf,150


### Структурированный parser VA/DoD

Для каждого документа parser:

1. удаляет повторяющиеся headers, footers и номера страниц;
2. объединяет части многострочных заголовков;
3. восстанавливает два уровня структуры: крупный раздел и буквенный подраздел;
4. сохраняет текущий `section_path` при переходе между страницами;
5. прекращает обработку перед началом библиографии `References`.

Результат приводится к той же metadata-схеме, которая используется для WHO и CDC.

In [78]:
def parse_va_dod(path):
    pages = extract_document_pages(path)
    metadata = DOCUMENT_METADATA[path.name]

    margin_noise = get_va_dod_margin_noise(path)
    references_start = get_va_dod_references_start_page(path)

    section_stack = {}
    records = []
    started = False

    for page_data in pages:
        if (
            references_start is not None
            and page_data["pdf_page"] >= references_start
        ):
            break

        lines = clean_va_dod_lines(
            page_data,
            margin_noise,
        )

        lines = merge_va_dod_heading_lines(
            lines
        )

        current_text = []

        def flush_text():
            if not current_text:
                return

            if not started:
                current_text.clear()
                return

            text = " ".join(
                current_text
            )

            text = re.sub(
                r"\s+",
                " ",
                text,
            ).strip()

            text = re.sub(
                r"-\s+(?=\w)",
                "-",
                text,
            )

            if text:
                records.append(
                    {
                        "document_id": page_data["document_id"],
                        "source": metadata["source"],
                        "document_title": metadata["document_title"],
                        "year": metadata["year"],
                        "section_path": build_section_path(
                            section_stack
                        ),
                        "page": page_data["page"],
                        "pdf_page": page_data["pdf_page"],
                        "text": text,
                    }
                )

            current_text.clear()

        for line in lines:
            level = line.get(
                "heading_level"
            )

            if level is not None:
                flush_text()

                started = True

                section_stack[level] = (
                    line["text"]
                )

                for existing_level in list(
                    section_stack
                ):
                    if existing_level > level:
                        del section_stack[
                            existing_level
                        ]

                continue

            current_text.append(
                line["text"]
            )

        flush_text()

    return records

In [79]:
va_dod_records = []

for path in va_dod_paths:
    va_dod_records.extend(
        parse_va_dod(path)
    )

va_dod_records_df = pd.DataFrame(
    va_dod_records
)

len(va_dod_records_df)

1110

In [80]:
va_dod_records_df.groupby(
    [
        "document_id",
        "document_title",
    ]
).size().reset_index(
    name="records"
)

,document_id,document_title,records
0,va_dod_asthma_2025,VA/DOD Clinical Practice Guideline for the Primary Care Management of Asthma,174
1,va_dod_ckd_2025,VA/DOD Clinical Practice Guideline for the Primary Care Management of Chronic Kidney Disease,205
2,va_dod_low_back_pain_2022,VA/DoD Clinical Practice Guideline for the Diagnosis and Treatment of Low Back Pain,160
3,va_dod_major_depression_2022,VA/DoD Clinical Practice Guideline for the Management of Major Depressive Disorder,174
4,va_dod_pregnancy_2023,VA/DoD Clinical Practice Guideline for the Management of Pregnancy,212
5,va_dod_type2_diabetes_2023,VA/DoD Clinical Practice Guideline for the Management of Type 2 Diabetes Mellitus,185


In [81]:
reference_leaks = []

for path in va_dod_paths:
    reference_page = (
        get_va_dod_references_start_page(path)
    )

    document_id = path.stem

    leaked = va_dod_records_df[
        (va_dod_records_df["document_id"] == document_id)
        & (
            va_dod_records_df["pdf_page"]
            >= reference_page
        )
    ]

    if not leaked.empty:
        reference_leaks.append(
            {
                "document_id": document_id,
                "records": len(leaked),
            }
        )

reference_leaks

[]

In [82]:
va_dod_major_headings = []

for path in va_dod_paths:
    margin_noise = get_va_dod_margin_noise(path)
    references_start = (
        get_va_dod_references_start_page(path)
    )

    pages = extract_document_pages(path)

    for page_data in pages:
        if (
            references_start is not None
            and page_data["pdf_page"] >= references_start
        ):
            break

        lines = clean_va_dod_lines(
            page_data,
            margin_noise,
        )

        lines = merge_va_dod_heading_lines(lines)

        for line in lines:
            if line.get("heading_level") == 1:
                va_dod_major_headings.append(
                    {
                        "file": path.name,
                        "pdf_page": page_data["pdf_page"],
                        "title": line["text"],
                    }
                )

va_dod_major_headings_df = pd.DataFrame(
    va_dod_major_headings
)

va_dod_major_headings_df

,file,pdf_page,title
0,va_dod_asthma_2025.pdf,6,I. Introduction
1,va_dod_asthma_2025.pdf,6,II. Background
2,va_dod_asthma_2025.pdf,9,III. Scope of This Guideline
3,va_dod_asthma_2025.pdf,10,IV. Highlighted Features of This Guideline
4,va_dod_asthma_2025.pdf,12,V. Guideline Development Team
...,...,...,...
127,va_dod_type2_diabetes_2023.pdf,114,Appendix F: 2017 CPG Recommendation Categorization Table
128,va_dod_type2_diabetes_2023.pdf,118,Appendix G: Participant List
129,va_dod_type2_diabetes_2023.pdf,120,Appendix H: Literature Review Search Terms and Strategy
130,va_dod_type2_diabetes_2023.pdf,145,Appendix I: Alternative Text Descriptions of Algorithm


## 12. Проверка качества parsing

Перед chunking проверяем, что структурированный корпус удовлетворяет базовым инвариантам:

- каждый документ представлен в корпусе;
- `text` не пустой;
- `section_path` не пустой;
- front matter не попадает в корпус до первого содержательного раздела;
- библиография `References` не индексируется;
- основные разделы документов корректно восстанавливаются.

In [83]:
parser_dfs = {
    "WHO": who_records_df,
    "CDC": cdc_records_df,
    "VA/DoD": va_dod_records_df,
}

parser_summary = []

for source, df in parser_dfs.items():
    parser_summary.append(
        {
            "source": source,
            "records": len(df),
            "documents": df["document_id"].nunique(),
            "empty_section_path": (
                df["section_path"].fillna("").str.strip().eq("").sum()
            ),
            "empty_text": (
                df["text"].fillna("").str.strip().eq("").sum()
            ),
        }
    )

pd.DataFrame(parser_summary)

,source,records,documents,empty_section_path,empty_text
0,WHO,42,1,0,0
1,CDC,599,1,0,0
2,VA/DoD,1110,6,0,0


### Результат parsing

Все 8 документов успешно преобразованы в единую структурированную схему.

После удаления front matter и восстановления структуры разделов:

- WHO: 42 records;
- CDC: 599 records;
- VA/DoD: 1110 records;
- пустых `section_path` нет;
- пустых текстовых records нет;
- библиографические разделы `References` исключены;
- многострочные заголовки корректно объединяются;
- основные разделы и подразделы отдельно проверены для каждого семейства документов;
- VA/DoD appendices распознаются как самостоятельные разделы верхнего уровня.

На этом этапе parsing считается завершённым.

## 13. Формирование retrieval corpus

После parsing в корпусе остаются не только клинические разделы, но и служебные части документов: описание разработки guideline, списки участников, поисковые стратегии, abbreviations и другие материалы.

Для dense retrieval такие разделы могут создавать шум: они формально относятся к медицинскому документу, но редко содержат информацию, полезную для ответа на клинический вопрос.

Поэтому перед chunking сначала анализируем состав структурированного корпуса и явно определяем, какие разделы будут индексироваться.

In [84]:
corpus_records_df = pd.concat(
    [
        who_records_df,
        cdc_records_df,
        va_dod_records_df,
    ],
    ignore_index=True,
)

len(corpus_records_df)

1751

In [85]:
corpus_records_df["top_level_section"] = (
    corpus_records_df["section_path"]
    .str.split(" > ")
    .str[0]
)

corpus_records_df[
    [
        "document_id",
        "top_level_section",
    ]
].drop_duplicates()

,document_id,top_level_section
0,who_hypertension_2021,1 Introduction
2,who_hypertension_2021,2 Method for developing the guideline
9,who_hypertension_2021,3 Recommendations
28,who_hypertension_2021,4 Special settings
32,who_hypertension_2021,"5 Publication, implementation, evaluation and research gaps"
...,...,...
1715,va_dod_type2_diabetes_2023,Appendix F: 2017 CPG Recommendation Categorization Table
1719,va_dod_type2_diabetes_2023,Appendix G: Participant List
1721,va_dod_type2_diabetes_2023,Appendix H: Literature Review Search Terms and Strategy
1746,va_dod_type2_diabetes_2023,Appendix I: Alternative Text Descriptions of Algorithm


In [86]:
section_inventory = (
    corpus_records_df
    .groupby(
        [
            "document_id",
            "top_level_section",
        ]
    )
    .agg(
        records=("text", "size"),
        first_pdf_page=("pdf_page", "min"),
        last_pdf_page=("pdf_page", "max"),
    )
    .reset_index()
)

pd.set_option(
    "display.max_colwidth",
    None,
)

section_inventory

,document_id,top_level_section,records,first_pdf_page,last_pdf_page
0,cdc_sti_2021,Chlamydial Infections,29,67,73
1,cdc_sti_2021,Clinical Prevention Guidance,30,4,13
2,cdc_sti_2021,"Diseases Characterized by Genital, Anal, or Perianal Ulcers",62,29,41
3,cdc_sti_2021,Diseases Characterized by Urethritis and Cervicitis,28,62,67
4,cdc_sti_2021,"Diseases Characterized by Vulvovaginal Itching, Burning, Irritation, Odor, or Discharge",47,84,96
...,...,...,...,...,...
160,who_hypertension_2021,2 Method for developing the guideline,7,15,18
161,who_hypertension_2021,3 Recommendations,19,19,32
162,who_hypertension_2021,4 Special settings,4,33,35
163,who_hypertension_2021,"5 Publication, implementation, evaluation and research gaps",6,36,37


In [87]:
who_records_df["top_level_section"] = (
    who_records_df["section_path"]
    .str.split(" > ")
    .str[0]
)

(
    who_records_df
    .groupby("top_level_section")
    .agg(
        records=("text", "size"),
        first_pdf_page=("pdf_page", "min"),
        last_pdf_page=("pdf_page", "max"),
    )
    .reset_index()
)

,top_level_section,records,first_pdf_page,last_pdf_page
0,1 Introduction,2,13,14
1,2 Method for developing the guideline,7,15,18
2,3 Recommendations,19,19,32
3,4 Special settings,4,33,35
4,"5 Publication, implementation, evaluation and research gaps",6,36,37
5,6 Implementation tools,4,38,41


In [88]:
appendix_inventory = (
    section_inventory[
        section_inventory["top_level_section"]
        .str.startswith(
            "Appendix",
            na=False,
        )
    ][
        [
            "document_id",
            "top_level_section",
            "records",
        ]
    ]
    .sort_values(
        [
            "document_id",
            "top_level_section",
        ]
    )
    .reset_index(drop=True)
)

pd.set_option(
    "display.max_rows",
    None,
)

appendix_inventory

,document_id,top_level_section,records
0,va_dod_asthma_2025,Appendix A: Guideline Development Methodology,23
1,va_dod_asthma_2025,Appendix B: Patient Focus Group Methods and Findings,3
2,va_dod_asthma_2025,Appendix C: Assessments of Asthma Severity and Control,10
3,va_dod_asthma_2025,Appendix D: Details of a Comprehensive History and Physical Exam,5
4,va_dod_asthma_2025,Appendix E: DOD Service-Specific Regulation Concerning Asthma,2
5,va_dod_asthma_2025,Appendix F: Example Asthma Action Plan Templates,8
6,va_dod_asthma_2025,Appendix G: Additional Information on Pharmacotherapy,10
7,va_dod_asthma_2025,Appendix H: Evidence Table,4
8,va_dod_asthma_2025,Appendix I: 2019 Recommendation Categorization,3
9,va_dod_asthma_2025,Appendix J: Participant List,2


### Отбор VA/DoD appendices для retrieval

Не все приложения клинических рекомендаций одинаково полезны для поиска.

Для baseline retrieval исключаем приложения, которые описывают создание документа, участников, поисковую стратегию, служебные сокращения и историческую категоризацию рекомендаций. Такие разделы увеличивают вероятность нерелевантных совпадений, но редко помогают ответить на клинический вопрос.

Клинические приложения — схемы ведения, pharmacotherapy, dosing, monitoring, diagnostic tools, definitions и текстовые описания алгоритмов — сохраняем.

Фильтрация выполняется только для retrieval corpus. Исходные parsed records остаются неизменными.

In [89]:
APPENDIX_DROP_RULES = [
    (
        "Guideline Development Methodology",
        "guideline development methodology",
    ),
    (
        "Patient Focus Group Methods and Findings",
        "patient focus group",
    ),
    (
        "Evidence Table",
        "evidence table duplicated by clinical recommendations",
    ),
    (
        "Recommendation Categorization",
        "historical recommendation categorization",
    ),
    (
        "Participant List",
        "participant list",
    ),
    (
        "Literature Review Search Terms and Strategy",
        "literature search methodology",
    ),
    (
        "Abbreviation",
        "abbreviation list",
    ),
]


def classify_va_dod_appendix(title):
    title_lower = title.lower()

    for pattern, reason in APPENDIX_DROP_RULES:
        if pattern.lower() in title_lower:
            return "DROP", reason


    if (
        "service-specific regulation"
        in title_lower
    ):
        return (
            "DROP",
            "military-specific regulatory content",
        )

    return (
        "KEEP",
        "clinically useful appendix",
    )

In [90]:
appendix_classification = (
    appendix_inventory.copy()
)

appendix_classification[
    ["retrieval_status", "reason"]
] = (
    appendix_classification[
        "top_level_section"
    ]
    .apply(
        lambda title: pd.Series(
            classify_va_dod_appendix(
                title
            )
        )
    )
)

appendix_classification[
    [
        "document_id",
        "top_level_section",
        "records",
        "retrieval_status",
        "reason",
    ]
]

,document_id,top_level_section,records,retrieval_status,reason
0,va_dod_asthma_2025,Appendix A: Guideline Development Methodology,23,DROP,guideline development methodology
1,va_dod_asthma_2025,Appendix B: Patient Focus Group Methods and Findings,3,DROP,patient focus group
2,va_dod_asthma_2025,Appendix C: Assessments of Asthma Severity and Control,10,KEEP,clinically useful appendix
3,va_dod_asthma_2025,Appendix D: Details of a Comprehensive History and Physical Exam,5,KEEP,clinically useful appendix
4,va_dod_asthma_2025,Appendix E: DOD Service-Specific Regulation Concerning Asthma,2,DROP,military-specific regulatory content
5,va_dod_asthma_2025,Appendix F: Example Asthma Action Plan Templates,8,KEEP,clinically useful appendix
6,va_dod_asthma_2025,Appendix G: Additional Information on Pharmacotherapy,10,KEEP,clinically useful appendix
7,va_dod_asthma_2025,Appendix H: Evidence Table,4,DROP,evidence table duplicated by clinical recommendations
8,va_dod_asthma_2025,Appendix I: 2019 Recommendation Categorization,3,DROP,historical recommendation categorization
9,va_dod_asthma_2025,Appendix J: Participant List,2,DROP,participant list


In [91]:
appendix_classification.groupby(
    [
        "retrieval_status",
        "reason",
    ]
).agg(
    sections=(
        "top_level_section",
        "size",
    ),
    records=(
        "records",
        "sum",
    ),
).reset_index()

,retrieval_status,reason,sections,records
0,DROP,abbreviation list,6,18
1,DROP,evidence table duplicated by clinical recommendations,6,30
2,DROP,guideline development methodology,6,122
3,DROP,historical recommendation categorization,6,27
4,DROP,literature search methodology,6,105
5,DROP,military-specific regulatory content,1,2
6,DROP,participant list,6,12
7,DROP,patient focus group,6,18
8,KEEP,clinically useful appendix,27,138


In [92]:
appendix_classification.loc[
    appendix_classification[
        "retrieval_status"
    ].eq("KEEP"),
    [
        "document_id",
        "top_level_section",
        "records",
    ],
].reset_index(drop=True)

,document_id,top_level_section,records
0,va_dod_asthma_2025,Appendix C: Assessments of Asthma Severity and Control,10
1,va_dod_asthma_2025,Appendix D: Details of a Comprehensive History and Physical Exam,5
2,va_dod_asthma_2025,Appendix F: Example Asthma Action Plan Templates,8
3,va_dod_asthma_2025,Appendix G: Additional Information on Pharmacotherapy,10
4,va_dod_asthma_2025,Appendix K: Alternative Text Descriptions of Algorithms,5
5,va_dod_ckd_2025,Appendix G. Alternative Text Descriptions of Algorithms,5
6,va_dod_ckd_2025,Appendix H. Management of CKD Table,3
7,va_dod_ckd_2025,Appendix I. Monitoring of CKD Table,4
8,va_dod_ckd_2025,Appendix J. Approaches for eGFR Calculation,4
9,va_dod_ckd_2025,Appendix K. Nephrotoxic Agents and Medication Dose Adjustments in CKD,6


### Фильтрация служебных VA/DoD appendices

Исходные parsed records сохраняются без изменений.

Для retrieval создаётся отдельная версия корпуса, из которой исключаются только заранее классифицированные служебные VA/DoD appendices. Клинические приложения сохраняются.

In [93]:
va_dod_appendices_to_drop = set(
    appendix_classification.loc[
        appendix_classification[
            "retrieval_status"
        ].eq("DROP"),
        "top_level_section",
    ]
)

len(va_dod_appendices_to_drop)

28

In [94]:
retrieval_records_df = (
    corpus_records_df.copy()
)

drop_appendix_mask = (
    retrieval_records_df["source"].eq("VA/DoD")
    & retrieval_records_df[
        "top_level_section"
    ].isin(
        va_dod_appendices_to_drop
    )
)

retrieval_records_df = (
    retrieval_records_df.loc[
        ~drop_appendix_mask
    ]
    .reset_index(drop=True)
)

In [ ]:
records_after_appendix_filtering = len(
    retrieval_records_df
)

print(
    "Records after appendix filtering:",
    records_after_appendix_filtering,
)

In [95]:
pd.DataFrame(
    {
        "stage": [
            "parsed corpus",
            "after appendix filtering",
        ],
        "records": [
            len(corpus_records_df),
            len(retrieval_records_df),
        ],
    }
)

,stage,records
0,parsed corpus,1751
1,after appendix filtering,1417


In [96]:
drop_appendix_mask.sum()

np.int64(334)

In [97]:
remaining_dropped_appendices = (
    retrieval_records_df[
        retrieval_records_df[
            "top_level_section"
        ].isin(
            va_dod_appendices_to_drop
        )
    ][
        [
            "document_id",
            "top_level_section",
        ]
    ]
    .drop_duplicates()
)

remaining_dropped_appendices

,document_id,top_level_section


In [98]:
non_appendix_inventory = (
    retrieval_records_df[
        ~retrieval_records_df[
            "top_level_section"
        ].str.startswith(
            "Appendix",
            na=False,
        )
    ]
    .groupby(
        [
            "document_id",
            "top_level_section",
        ]
    )
    .agg(
        records=("text", "size"),
    )
    .reset_index()
    .sort_values(
        [
            "document_id",
            "top_level_section",
        ]
    )
)

pd.set_option(
    "display.max_rows",
    None,
)

non_appendix_inventory

,document_id,top_level_section,records
0,cdc_sti_2021,Chlamydial Infections,29
1,cdc_sti_2021,Clinical Prevention Guidance,30
2,cdc_sti_2021,"Diseases Characterized by Genital, Anal, or Perianal Ulcers",62
3,cdc_sti_2021,Diseases Characterized by Urethritis and Cervicitis,28
4,cdc_sti_2021,"Diseases Characterized by Vulvovaginal Itching, Burning, Irritation, Odor, or Discharge",47
5,cdc_sti_2021,Ectoparasitic Infections,19
6,cdc_sti_2021,Epididymitis,9
7,cdc_sti_2021,Gonococcal Infections,43
8,cdc_sti_2021,HIV Infection,12
9,cdc_sti_2021,Human Papillomavirus Infections,45


### Фильтрация служебных основных разделов

Помимо приложений, часть основных разделов документов описывает создание guideline,
состав рабочей группы и будущие исследовательские направления.

Для baseline retrieval такие разделы исключаются, поскольку они не содержат информации,
непосредственно необходимой для ответа на клинические вопросы.

При этом сохраняются introduction/background, scope, clinical recommendations,
algorithms, special settings и другие разделы, потенциально содержащие клинический контекст.

In [99]:
NON_APPENDIX_DROP_RULES = [
    # VA/DoD
    "Guideline Development Team",
    "Summary of Guideline Development Methodology",
    "Research Priorities",

    # WHO
    "Method for developing the guideline",
    "Publication, implementation, evaluation and research gaps",

    # CDC
    "Methods",
]


def should_drop_non_appendix_section(title):
    title_lower = title.lower()

    return any(
        pattern.lower() in title_lower
        for pattern in NON_APPENDIX_DROP_RULES
    )

In [100]:
non_appendix_to_drop = (
    non_appendix_inventory[
        non_appendix_inventory[
            "top_level_section"
        ].apply(
            should_drop_non_appendix_section
        )
    ]
    .reset_index(drop=True)
)

non_appendix_to_drop

,document_id,top_level_section,records
0,cdc_sti_2021,Methods,2
1,va_dod_asthma_2025,V. Guideline Development Team,2
2,va_dod_asthma_2025,VI. Summary of Guideline Development Methodology,12
3,va_dod_asthma_2025,X. Research Priorities,6
4,va_dod_ckd_2025,V. Guideline Development Team,3
5,va_dod_ckd_2025,VI. Summary of Guideline Development Methodology,10
6,va_dod_ckd_2025,X. Research Priorities,7
7,va_dod_low_back_pain_2022,V. Guideline Development Team,3
8,va_dod_low_back_pain_2022,VI. Summary of Guideline Development Methodology,10
9,va_dod_low_back_pain_2022,X. Research Priorities,15


In [101]:
drop_non_appendix_mask = (
    ~retrieval_records_df[
        "top_level_section"
    ].str.startswith(
        "Appendix",
        na=False,
    )
    & retrieval_records_df[
        "top_level_section"
    ].apply(
        should_drop_non_appendix_section
    )
)

retrieval_records_df = (
    retrieval_records_df.loc[
        ~drop_non_appendix_mask
    ]
    .reset_index(drop=True)
)

drop_non_appendix_mask.sum()

np.int64(166)

In [ ]:
retrieval_summary = pd.DataFrame(
    {
        "stage": [
            "parsed corpus",
            "after appendix filtering",
            "final retrieval corpus",
        ],
        "records": [
            len(corpus_records_df),
            records_after_appendix_filtering,
            len(retrieval_records_df),
        ],
    }
)

retrieval_summary

,stage,records
0,parsed corpus,1751
1,after appendix filtering,1382
2,final retrieval corpus,1251


In [103]:
retrieval_records_df.groupby(
    "document_id"
).size().reset_index(
    name="records"
)

,document_id,records
0,cdc_sti_2021,597
1,va_dod_asthma_2025,100
2,va_dod_ckd_2025,133
3,va_dod_low_back_pain_2022,86
4,va_dod_major_depression_2022,98
5,va_dod_pregnancy_2023,119
6,va_dod_type2_diabetes_2023,89
7,who_hypertension_2021,29


### Итоговый retrieval corpus

После структурного parsing и фильтрации служебных разделов retrieval corpus содержит 1251 record из 8 клинических документов.

Из корпуса исключены:
- библиографические разделы `References`;
- front matter;
- методология разработки guideline;
- списки участников;
- поисковые стратегии;
- служебные abbreviation lists;
- historical recommendation categorization;
- другие явно неклинические приложения.

Клинические рекомендации, algorithms, diagnostic and monitoring information,
pharmacotherapy, dosing и клинически полезные appendices сохранены.

Исходный parsed corpus при этом остаётся неизменным; фильтрация применяется только к retrieval corpus.

## 14. Chunking

Dense retrieval работает не с целыми документами, а с отдельными фрагментами текста — chunks.

Слишком крупный chunk может содержать несколько разных тем и давать менее точный embedding.
Слишком маленький chunk теряет контекст, необходимый для интерпретации рекомендации.

Поэтому размер chunk нельзя выбирать произвольно. Сначала анализируем длину уже полученных structured records и определяем, требуется ли дополнительное разбиение.

In [104]:
retrieval_records_df = retrieval_records_df.copy()

retrieval_records_df["char_count"] = (
    retrieval_records_df["text"].str.len()
)

retrieval_records_df["word_count"] = (
    retrieval_records_df["text"]
    .str.split()
    .str.len()
)

retrieval_records_df[
    [
        "char_count",
        "word_count",
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

,char_count,word_count
count,1251.000000,1251.000000
mean,1675.223821,245.045564
std,1223.099497,180.087002
min,10.000000,2.000000
50%,1365.000000,197.000000
75%,2837.500000,413.500000
90%,3442.000000,505.000000
95%,3692.500000,546.000000
99%,4302.500000,636.000000
max,5522.000000,819.000000


In [105]:
retrieval_records_df.nlargest(
    20,
    "word_count",
)[
    [
        "document_id",
        "section_path",
        "pdf_page",
        "word_count",
        "char_count",
        "text",
    ]
]

document_id  \
524               cdc_sti_2021   
498               cdc_sti_2021   
415               cdc_sti_2021   
103               cdc_sti_2021   
35                cdc_sti_2021   
40                cdc_sti_2021   
611               cdc_sti_2021   
855            va_dod_ckd_2025   
511               cdc_sti_2021   
58                cdc_sti_2021   
461               cdc_sti_2021   
14       who_hypertension_2021   
60                cdc_sti_2021   
918  va_dod_low_back_pain_2022   
895  va_dod_low_back_pain_2022   
13       who_hypertension_2021   
18       who_hypertension_2021   
775            va_dod_ckd_2025   
20       who_hypertension_2021   
892  va_dod_low_back_pain_2022   

                                                                                                                                  section_path  \
524                   Human Papillomavirus Infections > Cervical Cancer > Follow-Up of Abnormal Cytology and Human Papillomavirus Test Results   
498                                                                             Human Papillomavirus Infections > Anogenital Warts > Treatment   
415  Diseases Characterized by Vulvovaginal Itching, Burning, Irritation, Odor, or Discharge > Bacterial Vaginosis > Diagnostic Considerations   
103                          STI Detection Among Special Populations > Women Who Have Sex with Women and Women Who Have Sex with Women and Men   
35                                                                        Clinical Prevention Guidance > STI and HIV Infection Risk Assessment   
40                                                      Clinical Prevention Guidance > Primary Prevention Methods > Condoms > External Condoms   
611                                              Sexual Assault and Abuse and STIs > Adolescents and Adults > Risk for Acquiring HIV Infection   
855                                                                                              Appendix Q. Gadolinium and Iodinated Contrast   
511                                                              Human Papillomavirus Infections > Cervical Cancer > Screening Recommendations   
58                                                                                             Clinical Prevention Guidance > Partner Services   
461                                                                                    Pelvic Inflammatory Disease > Diagnostic Considerations   
14                                                                                                 3 Recommendations > 3.5 Combination therapy   
60                                                                 Clinical Prevention Guidance > Partner Services > Expedited Partner Therapy   
918                                                                                                   IX. Recommendations > D. Pharmacotherapy   
895                                                                                   IX. Recommendations > B. Patient Education and Self-care   
13                                                                                                 3 Recommendations > 3.5 Combination therapy   
18                                                                                          3 Recommendations > 3.7 Frequency of re-assessment   
775                                                                                                                        IX. Recommendations   
20                                                           3 Recommendations > 3.8 Administration of treatment by nonphysician professionals   
892                                                                                IX. Recommendations > A. Evaluation and Diagnostic Approach   

     pdf_page  word_count  char_count  \
524       113         819        5380   
498       105         789        5522   
415        86         737        4963   
103        22         722        4873   
35          5         713       

In [106]:
pd.set_option(
    "display.max_colwidth",
    300,
)

In [107]:
retrieval_records_df.groupby(
    "source"
)["word_count"].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
    ]
)

,count,mean,std,min,50%,90%,95%,99%,max
source,,,,,,,,,
CDC,597.0,164.437186,147.296925,6.0,122.0,369.4,465.2,693.56,819.0
VA/DoD,625.0,315.358400,175.437675,2.0,346.0,528.0,557.8,594.00,688.0
WHO,29.0,389.103448,172.327832,48.0,382.0,609.2,618.8,637.12,643.0


### Вывод по размеру structured records

Structured record нельзя напрямую считать retrieval chunk.

Распределение длины неоднородно:

- медиана — около 201 слова;
- 25% records длиннее 419 слов;
- 10% длиннее 508 слов;
- максимальный record содержит 819 слов;
- VA/DoD и WHO в среднем существенно длиннее CDC.

Это означает, что часть records можно оставить целиком, а длинные records необходимо дополнительно разбивать.

При этом размер chunk должен задаваться в токенах embedding-модели, а не в словах, поскольку именно tokenizer embedding-модели определяет фактическую длину входа и риск truncation.

Поэтому сначала фиксируем embedding model для baseline dense retrieval, а затем подбираем token-based chunking.

In [ ]:
from transformers import AutoTokenizer

EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

embedding_tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL_NAME
)

In [109]:
def count_embedding_tokens(text):
    return len(
        embedding_tokenizer.encode(
            text,
            add_special_tokens=False,
        )
    )


retrieval_records_df["token_count"] = (
    retrieval_records_df["text"]
    .apply(count_embedding_tokens)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (784 > 512). Running this sequence through the model will result in indexing errors


In [110]:
retrieval_records_df[
    "token_count"
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

count    1251.000000
mean      373.112710
std       280.499356
min         3.000000
50%       314.000000
75%       607.500000
90%       753.000000
95%       836.500000
99%       965.000000
max      2391.000000
Name: token_count, dtype: float64

In [111]:
(
    retrieval_records_df[
        "token_count"
    ] > 512
).value_counts()

token_count
False    846
True     405
Name: count, dtype: int64

In [112]:
long_record_share = (
    retrieval_records_df[
        "token_count"
    ]
    .gt(512)
    .mean()
)

long_record_share

np.float64(0.3237410071942446)

### Стратегия chunking

Анализ длины показал, что structured records нельзя напрямую использовать как retrieval chunks:

- медиана составляет 314 токенов;
- 32.4% records превышают 512 токенов;
- максимальный record содержит 2391 токена.

Для baseline используется token-based chunking с целевым размером 384 токена и overlap 64 токена.

Chunking выполняется внутри одного structured record, поэтому chunk никогда не смешивает разные `section_path` и сохраняет точную page-level provenance.

В embedding input дополнительно включаются название документа и `section_path`, чтобы короткие или локальные фрагменты сохраняли смысловой контекст.

In [113]:
TARGET_CHUNK_TOKENS = 384
CHUNK_OVERLAP_TOKENS = 64

embedding_tokenizer.model_max_length

512

In [114]:
def build_embedding_prefix(row):
    return (
        f'Document: {row["document_title"]}\n'
        f'Section: {row["section_path"]}\n\n'
    )


def split_record_into_chunks(
    row,
    tokenizer,
    target_tokens=TARGET_CHUNK_TOKENS,
    overlap_tokens=CHUNK_OVERLAP_TOKENS,
):
    prefix = build_embedding_prefix(row)

    prefix_token_count = len(
        tokenizer.encode(
            prefix,
            add_special_tokens=False,
        )
    )

    body_budget = (
        target_tokens
        - prefix_token_count
    )

    if body_budget <= overlap_tokens:
        raise ValueError(
            "Metadata prefix is too long "
            "for the selected chunk size."
        )

    encoded = tokenizer(
        row["text"],
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )

    offsets = encoded["offset_mapping"]
    n_tokens = len(offsets)

    chunks = []

    if n_tokens == 0:
        return chunks

    start_token = 0
    chunk_index = 0

    while start_token < n_tokens:
        end_token = min(
            start_token + body_budget,
            n_tokens,
        )

        char_start = offsets[start_token][0]
        char_end = offsets[end_token - 1][1]

        chunk_text = (
            row["text"][char_start:char_end]
            .strip()
        )

        embedding_text = (
            prefix + chunk_text
        )

        chunks.append(
            {
                "record_id": row["record_id"],
                "chunk_id": (
                    f'{row["record_id"]}_chunk_{chunk_index:02d}'
                ),
                "document_id": row["document_id"],
                "source": row["source"],
                "document_title": row["document_title"],
                "year": row["year"],
                "section_path": row["section_path"],
                "page": row["page"],
                "pdf_page": row["pdf_page"],
                "chunk_index": chunk_index,
                "text": chunk_text,
                "embedding_text": embedding_text,
            }
        )

        if end_token == n_tokens:
            break

        start_token = (
            end_token
            - overlap_tokens
        )

        chunk_index += 1

    return chunks

### Формирование chunks

Разбиваем каждый structured record на один или несколько retrieval chunks.

Chunking выполняется независимо внутри каждого record, поэтому chunks не смешивают разные документы, страницы или `section_path`.

После разбиения проверяем общее количество полученных chunks.

In [115]:
retrieval_records_df = (
    retrieval_records_df
    .reset_index(drop=True)
    .copy()
)

retrieval_records_df["record_id"] = [
    f"record_{i:05d}"
    for i in range(len(retrieval_records_df))
]

In [116]:
chunks = []

for _, row in retrieval_records_df.iterrows():
    chunks.extend(
        split_record_into_chunks(
            row,
            embedding_tokenizer,
        )
    )

chunks_df = pd.DataFrame(chunks)

len(chunks_df)

2135

### Проверка длины полного embedding input

Embedding-модель получает не только текст chunk, но и metadata-prefix с названием документа и `section_path`.

Поэтому считаем длину именно полного `embedding_text`, который будет передаваться embedding-модели.

Цель проверки — убедиться, что выбранный chunk size действительно укладывается в заданный token budget.

In [117]:
chunks_df["embedding_token_count"] = (
    chunks_df["embedding_text"]
    .apply(
        lambda text: len(
            embedding_tokenizer.encode(
                text,
                add_special_tokens=True,
            )
        )
    )
)

chunks_df[
    "embedding_token_count"
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

count    2135.000000
mean      279.805152
std       114.235610
min        29.000000
50%       324.000000
75%       386.000000
90%       386.000000
95%       386.000000
99%       386.000000
max       387.000000
Name: embedding_token_count, dtype: float64

### Проверка отсутствия truncation

Проверяем, что ни один `embedding_text` не превышает максимальную длину входа tokenizer.

Если результат — пустой DataFrame, embedding-модель сможет обработать все chunks без автоматического обрезания текста.

In [118]:
chunks_df[
    chunks_df["embedding_token_count"]
    > embedding_tokenizer.model_max_length
]

,record_id,chunk_id,document_id,source,document_title,year,section_path,page,pdf_page,chunk_index,text,embedding_text,embedding_token_count


### Проверка длины текста внутри chunks

Отдельно считаем длину только содержимого chunk без metadata-prefix.

Это позволяет проверить качество самого разбиения и обнаружить слишком короткие chunks, которые могут возникать как остаток в конце длинного record.

In [119]:
chunks_df["text_token_count"] = (
    chunks_df["text"]
    .apply(
        lambda text: len(
            embedding_tokenizer.encode(
                text,
                add_special_tokens=False,
            )
        )
    )
)

chunks_df[
    "text_token_count"
].describe(
    percentiles=[
        0.01,
        0.05,
        0.50,
        0.95,
        0.99,
    ]
)

count    2135.000000
mean      245.130211
std       114.788787
min         3.000000
1%         15.680000
5%         52.700000
50%       283.000000
95%       361.000000
99%       364.660000
max       370.000000
Name: text_token_count, dtype: float64

**Наблюдение:**

Большинство chunks имеют содержательный размер, однако минимальная длина составляет всего несколько токенов.

Такие короткие chunks могут быть двух типов:

1. короткий исходный record — это допустимо;
2. искусственный хвост, появившийся после разбиения длинного record — такой chunk желательно устранить.

Поэтому отдельно проверяем короткие chunks с `chunk_index > 0`.

### Проверка коротких хвостовых chunks

Не удаляем все короткие chunks автоматически: короткий исходный record может быть содержательно полноценным.

Проблемой считаем прежде всего короткий chunk с `chunk_index > 0`, поскольку он возник в результате splitting и может представлять собой небольшой остаток предыдущего record.

In [120]:
SHORT_CHUNK_THRESHOLD = 64

short_chunk_summary = pd.DataFrame(
    {
        "short_first_chunks": [
            (
                (chunks_df["text_token_count"] < SHORT_CHUNK_THRESHOLD)
                & (chunks_df["chunk_index"] == 0)
            ).sum()
        ],
        "short_split_tails": [
            (
                (chunks_df["text_token_count"] < SHORT_CHUNK_THRESHOLD)
                & (chunks_df["chunk_index"] > 0)
            ).sum()
        ],
    }
)

short_chunk_summary

,short_first_chunks,short_split_tails
0,144,0


### Просмотр коротких хвостов

Выводим только короткие chunks, появившиеся после splitting.

Проверяем, действительно ли это искусственные остатки текста. Если таких chunks заметное количество, корректируем алгоритм chunking так, чтобы маленький хвост объединялся с предыдущим окном.

In [121]:
chunks_df[
    (chunks_df["text_token_count"] < SHORT_CHUNK_THRESHOLD)
    & (chunks_df["chunk_index"] > 0)
][
    [
        "document_id",
        "section_path",
        "pdf_page",
        "chunk_index",
        "text_token_count",
        "text",
    ]
]

,document_id,section_path,pdf_page,chunk_index,text_token_count,text


### Результат проверки chunking

Token-based chunking прошёл основные sanity-checks:

- сформировано 2135 chunks из 1251 structured records;
- ни один `embedding_text` не превышает максимальную длину tokenizer;
- максимальная длина полного embedding input составляет 388 токенов;
- короткие chunks встречаются только среди исходно коротких records;
- искусственных коротких хвостов после splitting не обнаружено (`short_split_tails = 0`);
- chunks не пересекают границы `section_path` и сохраняют page-level provenance.

Для baseline фиксируем параметры:

- target chunk size: 384 tokens;
- overlap: 64 tokens.

На этом этапе стратегия chunking считается зафиксированной.

Сейчас у chunks есть chunk_index, но он начинается заново для каждого record. Для retrieval evaluation и отладки нужен уникальный chunk_id.

### Идентификаторы records и chunks

Для воспроизводимой retrieval evaluation каждому исходному structured record присваивается уникальный `record_id`, а каждому chunk — уникальный `chunk_id`.

Это позволяет:

- однозначно связать найденный chunk с исходным record;
- отслеживать несколько chunks, полученных из одного record;
- анализировать retrieval errors;
- сравнивать результаты разных конфигураций chunking и retrieval.

Проверяем, что `chunk_id` уникален, а число уникальных `record_id` в chunks совпадает с числом исходных records.

In [122]:
retrieval_records_df = (
    retrieval_records_df
    .reset_index(drop=True)
    .copy()
)

retrieval_records_df["record_id"] = [
    f"record_{i:05d}"
    for i in range(len(retrieval_records_df))
]

In [123]:
chunks_df["chunk_id"].is_unique

True

In [124]:
chunks_df["record_id"].nunique(), len(retrieval_records_df)

(1251, 1251)

**Результат.**

Все `chunk_id` уникальны, а все 1251 исходных records представлены в `chunks_df`.

Идентификаторы сформированы корректно.

## 15. Dense embeddings

Каждый retrieval chunk преобразуется в dense vector с помощью
`BAAI/bge-base-en-v1.5`.

Семантически похожие тексты должны иметь близкие векторы в embedding space.
В дальнейшем query будет преобразовываться тем же encoder, после чего
релевантные chunks будут находиться по близости между query embedding и
document embeddings.

Для baseline embeddings L2-нормализуются. Поэтому cosine similarity между
двумя векторами эквивалентна их dot product, что удобно для последующего
vector search.

In [125]:
import numpy as np
import torch

from sentence_transformers import SentenceTransformer

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

device

'cuda'

In [126]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device,
)

embedding_model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7656.16it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [ ]:
embedding_model.get_sentence_embedding_dimension()

### Построение document embeddings

Для индексации кодируем `embedding_text`, а не только чистый текст chunk.

`embedding_text` содержит название документа и `section_path`, поэтому
embedding получает дополнительный контекст о том, к какому guideline и
клиническому разделу относится локальный фрагмент.

Document embeddings считаются один раз при построении knowledge base.

In [128]:
document_embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

Batches: 100%|██████████| 67/67 [00:22<00:00,  2.98it/s]


In [129]:
document_embeddings.shape

(2135, 768)

### Проверка embeddings

Проверяем три базовых свойства:

- число embeddings совпадает с числом chunks;
- в embeddings нет `NaN` или бесконечных значений;
- после L2-нормализации длина каждого вектора приблизительно равна 1.

In [130]:
embedding_checks = {
    "n_chunks": len(chunks_df),
    "n_embeddings": document_embeddings.shape[0],
    "embedding_dim": document_embeddings.shape[1],
    "has_nan": np.isnan(document_embeddings).any(),
    "has_inf": np.isinf(document_embeddings).any(),
}

embedding_checks

{'n_chunks': 2135,
 'n_embeddings': 2135,
 'embedding_dim': 768,
 'has_nan': np.False_,
 'has_inf': np.False_}

In [131]:
embedding_norms = np.linalg.norm(
    document_embeddings,
    axis=1,
)

pd.Series(
    embedding_norms,
    name="embedding_norm",
).describe()

count    2.135000e+03
mean     1.000000e+00
std      3.157785e-08
min      9.999999e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: embedding_norm, dtype: float64

In [132]:
assert document_embeddings.shape[0] == len(chunks_df)
assert not np.isnan(document_embeddings).any()
assert not np.isinf(document_embeddings).any()
assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-5,
)

print("Embedding sanity checks passed.")

Embedding sanity checks passed.


## 16. Проверка работы dense retrieval

Перед построением vector index проверяем, что dense retrieval действительно находит
клинически релевантные chunks.

Поскольку embeddings были L2-нормализованы, близость между query и chunk можно
считать через dot product, который в данном случае эквивалентен cosine similarity.

Для BGE запрос кодируется с отдельной retrieval-инструкцией, а chunks документа
кодируются без неё.

На этом этапе мы пока не измеряем итоговое качество retrieval количественно.
Цель — проверить базовую корректность поиска:

- находится ли нужный документ;
- находится ли нужный клинический раздел;
- содержит ли найденный chunk информацию, связанную с запросом.

In [133]:
BGE_QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)

In [134]:
def encode_query(query):
    query_text = (
        BGE_QUERY_INSTRUCTION
        + query
    )

    query_embedding = embedding_model.encode(
        query_text,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    return query_embedding

In [135]:
def dense_search(
    query,
    top_k=5,
):
    query_embedding = encode_query(
        query
    )

    scores = (
        document_embeddings
        @ query_embedding
    )

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = (
        chunks_df
        .iloc[top_indices]
        .copy()
    )

    results.insert(
        0,
        "score",
        scores[top_indices],
    )

    return results[
        [
            "score",
            "chunk_id",
            "document_id",
            "section_path",
            "page",
            "pdf_page",
            "text",
        ]
    ]

### Проверка на одном клиническом запросе

In [136]:
query = (
    "How should hyperkalemia be managed "
    "in patients with chronic kidney disease?"
)

results = dense_search(
    query,
    top_k=5,
)

pd.set_option(
    "display.max_colwidth",
    300,
)

results

,score,chunk_id,document_id,section_path,page,pdf_page,text
1286,0.798135,record_00834_chunk_01,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > D. Dietary Considerations,147,147,"for the Evaluation and Management of Chronic Kidney Disease (3) recommends, “Provide advice to limit the intake of foods rich in bioavailable potassium (e.g., processed foods) for people with CKD G3-G5 who have a history of hyperkalemia.” The involvement of a renal dietitian can be helpful, as c..."
1283,0.793694,record_00832_chunk_01,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,146,146,upper limit of normal range They acknowledge that it is not known whether EKG changes are sensitive in the prediction of potentially lethal arrhythmia. The KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease (3) recommends the following steps to man...
1282,0.789073,record_00832_chunk_00,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,146,146,"The aggressiveness of treatment of hyperkalemia depends on the degree of elevation and the presence or absence of electrocardiogram (EKG) findings. Observationally, the risk of death from a given level of hyperkalemia is lower in more advanced CKD, suggesting that there are adaptive mechanisms t..."
1281,0.768300,record_00831_chunk_00,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > B. Definition,146,146,"An acute episode of hyperkalemia is a potassium result above the upper limit of normal that is not known to be chronic. However, there is no consensus on the magnitude, duration, and frequency of elevated potassium values that define chronicity.(277)"
1284,0.762154,record_00833_chunk_00,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,147,147,• Optimize serum bicarbonate levels • Consider potassium-exchange agents Third-line (Last resort): • Reduce or discontinue RAASi/MRA; restart in future if patient condition allows Providers may consider reducing or stopping ACEI/ARB when eGFR drops below a given eGFR threshold or when hyperkalem...


In [137]:
sanity_queries = [
    "How should asthma severity and control be assessed?",
    "What treatment is recommended for chlamydial infection?",
    "What blood pressure treatment is recommended for adults with hypertension?",
    "What pharmacologic treatments are recommended for major depressive disorder?",
    "How should glycemic control be monitored in type 2 diabetes?",
]

In [138]:
for query in sanity_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)

    results = dense_search(
        query,
        top_k=3,
    )

    for _, row in results.iterrows():
        print(
            f'\nscore={row["score"]:.3f}'
            f'\n{row["document_id"]}'
            f'\n{row["section_path"]}'
            f'\n{row["text"][:300]}...'
        )


QUERY: How should asthma severity and control be assessed?

score=0.765
va_dod_asthma_2025
Appendix C: Assessments of Asthma Severity and Control > A. Assessment of Asthma Severity
Table C-1. Assessment of Asthma Severitya, b, c, d Asthma Severity Description of Asthma Control (assess after trial of 2-3 months of treatment) Mild Controlled on low-intensity treatment (ex: prn low dose ICS-formoterol or low dose ICS + prn rapid-onset LABA/SABA) Moderate Controlled with low or me...

score=0.750
va_dod_asthma_2025
II. Background > B. Classification of Asthma Severity and Control
the initial evaluation of a newly diagnosed patient. This table was carried forward from the 2019 VA/DOD Asthma CPG. The Algorithm within this CPG refers to Appendix C for the initial management of newly diagnosed patients. Decision points in the algorithm are determined by the CPG’s key recommendat...

score=0.749
va_dod_asthma_2025
II. Background > B. Classification of Asthma Severity and Control
Asthma severit

### Результат первой проверки dense retrieval

Dense retrieval корректно находит тематически релевантные chunks из нужных
clinical guidelines для нескольких разных медицинских тем.

В ходе ручной проверки запроса о glycemic control при type 2 diabetes была
обнаружена ошибка VA/DoD parser: часть текста ошибочно относилась к subsection
`B. Telehealth` вместо `C. Diabetes Mellitus`. После исправления составных
subsection headings metadata стала корректной.

При этом один медицинский вопрос может иметь релевантную информацию сразу
в нескольких разделах документа. Например, запрос о monitoring glycemic control
находит как рекомендации по HbA1c variability и continuous glucose monitoring,
так и appendix `Glycemic Control Targets and Monitoring`.

Поэтому качество retrieval нельзя оценивать только по совпадению с одним
заранее выбранным `section_path`. Для дальнейшей количественной оценки нужен
отдельный evaluation-набор с разметкой всех допустимых релевантных разделов.

## 17. Количественная оценка dense retrieval

Ручные примеры показывают, что retrieval работает, но не позволяют объективно сравнивать разные настройки.

Для оценки retrieval нужен набор запросов с заранее известным релевантным содержимым.

Используем только `dev`-часть данных. `test` остаётся замороженным и не используется для выбора embedding model, chunk size, overlap, `top_k` или других параметров.

Для каждого evaluation query фиксируем как минимум релевантный `document_id` и, где возможно, релевантный `section_path`.

In [139]:
retrieval_eval = [
    {
        "query": (
            "How should hyperkalemia be managed "
            "in patients with chronic kidney disease?"
        ),
        "relevant_document_id": "va_dod_ckd_2025",
        "relevant_section_contains": "Management of Hyperkalemia",
    },
    {
        "query": (
            "How should asthma severity and control be assessed?"
        ),
        "relevant_document_id": "va_dod_asthma_2025",
        "relevant_section_contains": "Assessments of Asthma Severity and Control",
    },
    {
        "query": (
            "What treatment is recommended "
            "for chlamydial infection in adults?"
        ),
        "relevant_document_id": "cdc_sti_2021",
        "relevant_section_contains": (
            "Chlamydial Infection Among Adolescents and Adults > Treatment"
        ),
    },
    {
        "query": (
            "What blood pressure target is recommended "
            "for adults with hypertension?"
        ),
        "relevant_document_id": "who_hypertension_2021",
        "relevant_section_contains": "Target blood pressure",
    },
    {
        "query": (
            "What pharmacologic treatments are recommended "
            "for major depressive disorder?"
        ),
        "relevant_document_id": "va_dod_major_depression_2022",
        "relevant_section_contains": "Recommendations",
    },
    {
        "query": (
            "How should glycemic control be monitored "
            "in type 2 diabetes?"
        ),
        "relevant_document_id": "va_dod_type2_diabetes_2023",
        "relevant_section_contains": "Glycemic Control Targets and Monitoring",
    },
]

retrieval_eval_df = pd.DataFrame(
    retrieval_eval
)

retrieval_eval_df

,query,relevant_document_id,relevant_section_contains
0,How should hyperkalemia be managed in patients with chronic kidney disease?,va_dod_ckd_2025,Management of Hyperkalemia
1,How should asthma severity and control be assessed?,va_dod_asthma_2025,Assessments of Asthma Severity and Control
2,What treatment is recommended for chlamydial infection in adults?,cdc_sti_2021,Chlamydial Infection Among Adolescents and Adults > Treatment
3,What blood pressure target is recommended for adults with hypertension?,who_hypertension_2021,Target blood pressure
4,What pharmacologic treatments are recommended for major depressive disorder?,va_dod_major_depression_2022,Recommendations
5,How should glycemic control be monitored in type 2 diabetes?,va_dod_type2_diabetes_2023,Glycemic Control Targets and Monitoring


In [140]:
def get_relevant_rank(
    query,
    relevant_document_id,
    relevant_section_contains,
    top_k=10,
):
    results = dense_search(
        query,
        top_k=top_k,
    )

    mask = (
        results["document_id"].eq(
            relevant_document_id
        )
        & results["section_path"].str.contains(
            relevant_section_contains,
            case=False,
            regex=False,
            na=False,
        )
    )

    matching_positions = np.flatnonzero(
        mask.to_numpy()
    )

    if len(matching_positions) == 0:
        return None

    return int(
        matching_positions[0] + 1
    )

In [141]:
eval_results = []

for _, row in retrieval_eval_df.iterrows():
    rank = get_relevant_rank(
        query=row["query"],
        relevant_document_id=(
            row["relevant_document_id"]
        ),
        relevant_section_contains=(
            row["relevant_section_contains"]
        ),
        top_k=10,
    )

    eval_results.append(
        {
            **row.to_dict(),
            "relevant_rank": rank,
        }
    )

retrieval_debug_results_df = pd.DataFrame(
    eval_results
)

retrieval_debug_results_df

,query,relevant_document_id,relevant_section_contains,relevant_rank
0,How should hyperkalemia be managed in patients with chronic kidney disease?,va_dod_ckd_2025,Management of Hyperkalemia,1
1,How should asthma severity and control be assessed?,va_dod_asthma_2025,Assessments of Asthma Severity and Control,1
2,What treatment is recommended for chlamydial infection in adults?,cdc_sti_2021,Chlamydial Infection Among Adolescents and Adults > Treatment,1
3,What blood pressure target is recommended for adults with hypertension?,who_hypertension_2021,Target blood pressure,1
4,What pharmacologic treatments are recommended for major depressive disorder?,va_dod_major_depression_2022,Recommendations,1
5,How should glycemic control be monitored in type 2 diabetes?,va_dod_type2_diabetes_2023,Glycemic Control Targets and Monitoring,6


### Результат debug-проверки

На 5 из 6 проверочных запросов релевантный раздел находится на первом месте.

Для запроса о monitoring glycemic control при type 2 diabetes нужный раздел
`Glycemic Control Targets and Monitoring` находится только на 6-м месте.

Это показывает, что dense retrieval в целом работает, но ranking не всегда оптимален.

Эти запросы уже использовались при разработке pipeline, поэтому они рассматриваются
только как debug-набор и не используются как независимая оценка качества.

In [142]:
def hit_at_k(ranks, k):
    return np.mean(
        [
            rank is not None and rank <= k
            for rank in ranks
        ]
    )


def mean_reciprocal_rank(ranks):
    reciprocal_ranks = [
        0.0 if rank is None else 1.0 / rank
        for rank in ranks
    ]

    return np.mean(reciprocal_ranks)


debug_ranks = (
    retrieval_debug_results_df[
        "relevant_rank"
    ].tolist()
)

debug_metrics = {
    "Hit@1": hit_at_k(debug_ranks, 1),
    "Hit@3": hit_at_k(debug_ranks, 3),
    "Hit@5": hit_at_k(debug_ranks, 5),
    "Hit@10": hit_at_k(debug_ranks, 10),
    "MRR": mean_reciprocal_rank(debug_ranks),
}

debug_metrics

{'Hit@1': np.float64(0.8333333333333334),
 'Hit@3': np.float64(0.8333333333333334),
 'Hit@5': np.float64(0.8333333333333334),
 'Hit@10': np.float64(1.0),
 'MRR': np.float64(0.8611111111111112)}

### Разбор запроса с неидеальным ranking

Запрос о monitoring glycemic control является полезным примером ошибки ranking.

Проверяем top-10, чтобы понять, какие тематически близкие chunks вытесняют
наиболее специфичный раздел.

In [143]:
t2d_query = (
    "How should glycemic control be monitored "
    "in type 2 diabetes?"
)

t2d_results = dense_search(
    t2d_query,
    top_k=10,
)

t2d_results[
    [
        "score",
        "document_id",
        "section_path",
        "text",
    ]
]

,score,document_id,section_path,text
2029,0.735175,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"the following recommendation: For adults with type 2 diabetes mellitus, we suggest using high glycemic variability over time (e.g., fluctuation in HbA1c or fasting blood glucose) as a prognostic indicator for risk of hypoglycemia, morbidity, and mortality."
2104,0.724610,va_dod_type2_diabetes_2023,IX. Recommendations > E. Pharmacotherapy,"Recommendation 25. In adults with type 2 diabetes mellitus, especially those 65 years and older, we suggest prioritizing drug classes other than insulin, sulfonylureas, or meglitinides to minimize the risk of hypoglycemia, if glycemic control can be achieved with other treatments. (Weak for | Re..."
2046,0.720311,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"Discussion Although evidence is increasing that continuous glucose monitoring (CGM) can improve glucose control and might reduce hypoglycemia in T1DM patients, whether CGM improves outcomes in patients with T2DM is unclear. However, the systematic evidence review found that results differed depe..."
1994,0.719733,va_dod_type2_diabetes_2023,IX. Recommendations,"’s appraisal of the risk benefit ratio, patient characteristics, presence or absence of type 2 diabetes mellitus complications, comorbidities, and life expectancy. 10. We suggest an HbA1c range of 7.0–8.5% for most patients, if it can be safely achieved. Weak for Not reviewed, Amended Glycemic M..."
1993,0.716697,va_dod_type2_diabetes_2023,IX. Recommendations,"Topic Sub-topic # Recommendation Strengtha Categoryb Neither for nor against Reviewed, New­added 4. There is insufficient evidence to recommend for or against routine screening or using a specific tool to screen for or diagnose diabetes distress. 5. Weak for Reviewed, New­added In adults with ty..."
2115,0.715480,va_dod_type2_diabetes_2023,Appendix B: Glycemic Control Targets and Monitoring,"in some patients based on other factors, balancing safety and tolerability of therapy. n Major comorbidity is present, but is not end-stage, and management is achievable. o Major comorbidity is present and is either end-stage or management is significantly challenging, including mental health co..."
2028,0.710746,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,causal effect on adverse events and whether prospectively reducing fasting blood glucose or HbA1c variability will favorably impact outcomes are also unknown. The Work Group systematically reviewed evidence related to this recommendation and considered the assessment of the evidence put forth in...
2027,0.709554,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"events.(75, 76, 83) The increased risk for hypoglycemia events in the setting of higher variability generally ranged from 50–300%. Notably, the median duration of follow-up ranged from 8 months to nearly 9 years, so the number of measures of fasting glucose or HbA1c differed substantially across..."
2022,0.707995,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"be burdensome for providers with limited time. Most screening tools were developed to target older patients, so their accuracy in younger patients is unclear. The Work Group systematically searched for evidence and did not identify any studies that met inclusion criteria regarding routine screen..."
2050,0.705465,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"it is categorized as Reviewed, New-added. The Work Group’s confidence in the quality of the evidence was moderate. The body of evidence had some limitations, including a small sample size spread among relatively few studies.(92–95) We found moderate evidence that rtCGM led to decreased hypoglyce..."


### Проверка структуры T2D recommendations

В результатах поиска несколько фрагментов о glycemic variability, HbA1c и CGM
имеют `section_path = IX. Recommendations > B. Telehealth`.

Поскольку guideline содержит последующие тематические подразделы recommendations,
проверяем, корректно ли parser распознал переход от `B. Telehealth`
к следующим subsection headings.

Это важно проверить до количественной оценки retrieval, поскольку ошибочный
`section_path` сделает разметку релевантных разделов ненадёжной.

In [144]:
t2d_path = (
    GUIDELINES_DIR
    / "va_dod"
    / "va_dod_type2_diabetes_2023.pdf"
)

t2d_pages = extract_document_pages(
    t2d_path
)

for pdf_page in range(30, 36):
    page_data = t2d_pages[
        pdf_page - 1
    ]

    lines = clean_va_dod_lines(
        page_data,
        get_va_dod_margin_noise(
            t2d_path
        ),
    )

    lines = merge_va_dod_heading_lines(
        lines
    )

    print(
        "\n",
        "=" * 80,
        f"\npdf_page = {pdf_page}",
    )

    for line in lines:
        if (
            line.get("heading_level") is not None
            or "Diabetes Mellitus" in line["text"]
            or "Telehealth" in line["text"]
        ):
            print(
                "level:",
                line.get("heading_level"),
                "| text:",
                line["text"],
                "| size:",
                line["font_size"],
                "| fonts:",
                line["fonts"],
                "| bbox:",
                line["bbox"],
            )


pdf_page = 30

pdf_page = 31
level: 2 | text: B. Telehealth | size: 13.0 | fonts: ['Arial-BoldMT'] | bbox: (72.0, 367.3, 165.1, 380.3)

pdf_page = 32

pdf_page = 33
level: 2 | text: C. Diabetes Mellitus | size: 13.0 | fonts: ['Arial-BoldMT'] | bbox: (72.0, 74.2, 207.0, 87.2)

pdf_page = 34

pdf_page = 35


### Проверка переноса subsection между страницами

Заголовок `Diabetes Mellitus` имеет формат subsection, но parser не распознаёт его,
поскольку в строке отсутствует буквенный префикс `C.`.

Проверяем конец предыдущей страницы и начало следующей, чтобы понять,
не был ли заголовок разделён между страницами PDF.

In [145]:
for pdf_page in [32, 33]:
    print("\n", "=" * 80)
    print("pdf_page =", pdf_page)

    page_data = t2d_pages[pdf_page - 1]

    lines = clean_va_dod_lines(
        page_data,
        get_va_dod_margin_noise(t2d_path),
    )

    for i, line in enumerate(lines):
        if (
            line["font_size"] >= 12.5
            and is_va_dod_bold(line)
        ):
            print(
                i,
                "|",
                line["text"],
                "| size:",
                line["font_size"],
                "| bbox:",
                line["bbox"],
            )


pdf_page = 32

pdf_page = 33
0 | C. | size: 13.0 | bbox: (72.0, 74.2, 88.6, 87.2)
1 | Diabetes Mellitus | size: 13.0 | bbox: (100.8, 74.2, 207.0, 87.2)


### Проверка объединения составного заголовка

В PDF subsection heading `C. Diabetes Mellitus` разбит на два текстовых блока,
расположенных на одной строке.

Проверяем отдельно работу `merge_va_dod_inline_labels()` и
`merge_va_dod_heading_lines()`.

In [146]:
page_data = t2d_pages[33 - 1]

lines = clean_va_dod_lines(
    page_data,
    get_va_dod_margin_noise(t2d_path),
)

print("BEFORE:")
for line in lines[:5]:
    print(
        line["text"],
        "| size:", line["font_size"],
        "| fonts:", line["fonts"],
        "| bbox:", line["bbox"],
    )


inline_merged = merge_va_dod_inline_labels(
    lines
)

print("\nAFTER merge_va_dod_inline_labels:")
for line in inline_merged[:5]:
    print(
        line["text"],
        "| size:", line["font_size"],
        "| fonts:", line["fonts"],
        "| bbox:", line["bbox"],
    )


heading_merged = merge_va_dod_heading_lines(
    lines
)

print("\nAFTER merge_va_dod_heading_lines:")
for line in heading_merged[:5]:
    print(
        line["text"],
        "| level:", line.get("heading_level"),
        "| size:", line["font_size"],
        "| bbox:", line["bbox"],
    )

BEFORE:
C. | size: 13.0 | fonts: ['Arial-BoldMT'] | bbox: (72.0, 74.2, 88.6, 87.2)
Diabetes Mellitus | size: 13.0 | fonts: ['Arial-BoldMT'] | bbox: (100.8, 74.2, 207.0, 87.2)
Recommendation | size: 12.0 | fonts: ['Arial-BoldItalicMT'] | bbox: (72.0, 93.9, 172.7, 105.9)
4. There is insufficient evidence to recommend for or against routine screening or | size: 12.0 | fonts: ['ArialMT'] | bbox: (90.0, 112.7, 532.3, 124.7)
using a specific tool to screen for or diagnose diabetes distress. | size: 12.0 | fonts: ['ArialMT'] | bbox: (111.6, 127.8, 452.6, 139.8)

AFTER merge_va_dod_inline_labels:
C. Diabetes Mellitus | size: 13.0 | fonts: ['Arial-BoldMT'] | bbox: (72.0, 74.2, 207.0, 87.2)
Recommendation | size: 12.0 | fonts: ['Arial-BoldItalicMT'] | bbox: (72.0, 93.9, 172.7, 105.9)
4. There is insufficient evidence to recommend for or against routine screening or | size: 12.0 | fonts: ['ArialMT'] | bbox: (90.0, 112.7, 532.3, 124.7)
using a specific tool to screen for or diagnose diabetes distr

In [147]:
heading_merged = (
    merge_va_dod_heading_lines(
        lines
    )
)

for line in heading_merged[:5]:
    print(
        line["text"],
        "| level:",
        line.get("heading_level"),
        "| size:",
        line["font_size"],
    )

C. Diabetes Mellitus | level: 2 | size: 13.0
Recommendation | level: None | size: 12.0
4. There is insufficient evidence to recommend for or against routine screening or | level: None | size: 12.0
using a specific tool to screen for or diagnose diabetes distress. | level: None | size: 12.0
(Neither for nor against | Reviewed, New-added) | level: None | size: 12.0


### Проверка составных subsection headings

В некоторых VA/DoD PDF буквенный префикс subsection и название раздела
хранятся как отдельные текстовые блоки на одной строке.

Например:

`C.` + `Diabetes Mellitus`

Parser объединяет такие блоки до определения уровня заголовка.

In [148]:
t2d_test_lines = merge_va_dod_heading_lines(
    clean_va_dod_lines(
        t2d_pages[33 - 1],
        get_va_dod_margin_noise(t2d_path),
    )
)

[
    (
        line["text"],
        line.get("heading_level"),
    )
    for line in t2d_test_lines
    if "Diabetes Mellitus" in line["text"]
]

[('C. Diabetes Mellitus', 2)]

In [149]:
va_dod_records_df[
    (
        va_dod_records_df["document_id"]
        == "va_dod_type2_diabetes_2023"
    )
    & (
        va_dod_records_df["section_path"]
        .str.contains(
            "Diabetes Mellitus",
            case=False,
            regex=False,
            na=False,
        )
    )
][
    [
        "section_path",
        "pdf_page",
        "text",
    ]
].head(10)

,section_path,pdf_page,text
929,II. Background > A. Description of Type 2 Diabetes Mellitus,5,"Diabetes mellitus (DM) is a disease caused by an absolute or relative insulin deficiency resulting in hyperglycemia. Type 1 DM (T1DM) is due to deficient insulin production and secretion and can present across the lifespan, with older patients often having a more indolent presentation that has b..."
930,II. Background > A. Description of Type 2 Diabetes Mellitus,6,skeletal muscle. Prediabetes refers to the development of dysglycemia that does not reach the threshold for a diagnosis of diabetes. Gestational diabetes mellitus (GDM) is diabetes diagnosed in the second or third trimester of pregnancy that is typically not clinically overt. A variety of other ...
931,II. Background > A. Description of Type 2 Diabetes Mellitus,7,"· Women with polycystic ovary syndrome(3) · History of GDM(3) or history of delivering babies weighing >9 pounds (about 4 kg) · Other clinical conditions associated with insulin resistance (e.g., severe obesity, acanthosis nigricans)(3) · Physical inactivity/sedentary lifestyle(3) · Patients wit..."
932,II. Background > A. Description of Type 2 Diabetes Mellitus,8,"An oral glucose tolerance test (OGTT) is not commonly used to diagnose DM. Although both the ADA and the American Association of Clinical Endocrinology guidelines include the OGTT as a diagnostic criterion for T2DM, it is cumbersome and needs better reproducibility, making it less useful for rou..."
976,IX. Recommendations > C. Diabetes Mellitus,33,"Recommendation 4. There is insufficient evidence to recommend for or against routine screening or using a specific tool to screen for or diagnose diabetes distress. (Neither for nor against | Reviewed, New-added) Discussion The Work Group found no evidence in the systematic evidence review relat..."
977,IX. Recommendations > C. Diabetes Mellitus,34,"progressive disease such as non-alcoholic steatohepatitis (NASH), a more severe form of fatty liver disease associated with a greater risk of advanced fibrosis and cirrhosis. Approximately 70% of patients with T2DM and NAFLD are diagnosed with biopsy-proven NASH, which is two to three times the ..."
978,IX. Recommendations > C. Diabetes Mellitus,35,general population might be extrapolated to patients with T2DM and used as justification for their application in clinical practice. Other noninvasive methods that require no imaging may be used to assess advanced liver fibrosis in patients with T2DM and co-occurring NAFLD. These methods include...
979,IX. Recommendations > C. Diabetes Mellitus,36,"inclined to reserve testing until signs and symptoms appear. Other considerations, such as acceptability and feasibility, are unlikely to have a significant impact. The Work Group systematically reviewed evidence related to this recommendation. Therefore, it is categorized as Reviewed, New-added..."
980,IX. Recommendations > C. Diabetes Mellitus,37,"be burdensome for providers with limited time. Most screening tools were developed to target older patients, so their accuracy in younger patients is unclear. The Work Group systematically searched for evidence and did not identify any studies that met inclusion criteria regarding routine screen..."
981,IX. Recommendations > C. Diabetes Mellitus,38,"The Work Group considered the assessment of the evidence put forth in the 2017 VA/ DoD DM CPG. Therefore, it is categorized as Not reviewed, Amended.(70–72) The Work Group’s confidence in the quality of the evidence was moderate. The body of evidence had some limitations, including inability to ..."


In [ ]:
va_dod_path_by_document_id = {
    "va_dod_asthma_2025": (
        GUIDELINES_DIR
        / "va_dod"
        / "va_dod_asthma_2025.pdf"
    ),
    "va_dod_ckd_2025": (
        GUIDELINES_DIR
        / "va_dod"
        / "va_dod_ckd_2025.pdf"
    ),
    "va_dod_low_back_pain_2022": (
        GUIDELINES_DIR
        / "va_dod"
        / "va_dod_low_back_pain_2022.pdf"
    ),
    "va_dod_major_depression_2022": (
        GUIDELINES_DIR
        / "va_dod"
        / "va_dod_major_depression_2022.pdf"
    ),
    "va_dod_pregnancy_2023": (
        GUIDELINES_DIR
        / "va_dod"
        / "va_dod_pregnancy_2023.pdf"
    ),
    "va_dod_type2_diabetes_2023": (
        GUIDELINES_DIR
        / "va_dod"
        / "va_dod_type2_diabetes_2023.pdf"
    ),
}

In [ ]:
body_start_pages = (
    va_dod_records_df
    .groupby("document_id")["pdf_page"]
    .min()
    .to_dict()
)

unmerged_heading_candidates = []

for document_id, path in va_dod_path_by_document_id.items():
    pages = extract_document_pages(path)
    margin_noise = get_va_dod_margin_noise(path)

    body_start = body_start_pages[document_id]

    for page_data in pages:
        if page_data["pdf_page"] < body_start:
            continue

        lines = clean_va_dod_lines(
            page_data,
            margin_noise,
        )

        lines = merge_va_dod_inline_labels(
            lines
        )

        for line in lines:
            if (
                re.fullmatch(
                    r"[A-Z]\.",
                    line["text"].strip(),
                )
                and is_va_dod_bold(line)
                and line["font_size"] >= 11.8
                and line["bbox"][0] < 120
            ):
                unmerged_heading_candidates.append(
                    {
                        "document_id": document_id,
                        "pdf_page": page_data["pdf_page"],
                        "text": line["text"],
                        "font_size": line["font_size"],
                        "bbox": line["bbox"],
                    }
                )

pd.DataFrame(unmerged_heading_candidates)

""


In [152]:
t2d_query = (
    "How should glycemic control be monitored "
    "in type 2 diabetes?"
)

dense_search(
    t2d_query,
    top_k=10,
)[
    [
        "score",
        "document_id",
        "section_path",
        "text",
    ]
]

,score,document_id,section_path,text
2029,0.735175,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"the following recommendation: For adults with type 2 diabetes mellitus, we suggest using high glycemic variability over time (e.g., fluctuation in HbA1c or fasting blood glucose) as a prognostic indicator for risk of hypoglycemia, morbidity, and mortality."
2104,0.724610,va_dod_type2_diabetes_2023,IX. Recommendations > E. Pharmacotherapy,"Recommendation 25. In adults with type 2 diabetes mellitus, especially those 65 years and older, we suggest prioritizing drug classes other than insulin, sulfonylureas, or meglitinides to minimize the risk of hypoglycemia, if glycemic control can be achieved with other treatments. (Weak for | Re..."
2046,0.720311,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"Discussion Although evidence is increasing that continuous glucose monitoring (CGM) can improve glucose control and might reduce hypoglycemia in T1DM patients, whether CGM improves outcomes in patients with T2DM is unclear. However, the systematic evidence review found that results differed depe..."
1994,0.719733,va_dod_type2_diabetes_2023,IX. Recommendations,"’s appraisal of the risk benefit ratio, patient characteristics, presence or absence of type 2 diabetes mellitus complications, comorbidities, and life expectancy. 10. We suggest an HbA1c range of 7.0–8.5% for most patients, if it can be safely achieved. Weak for Not reviewed, Amended Glycemic M..."
1993,0.716697,va_dod_type2_diabetes_2023,IX. Recommendations,"Topic Sub-topic # Recommendation Strengtha Categoryb Neither for nor against Reviewed, New­added 4. There is insufficient evidence to recommend for or against routine screening or using a specific tool to screen for or diagnose diabetes distress. 5. Weak for Reviewed, New­added In adults with ty..."
2115,0.715480,va_dod_type2_diabetes_2023,Appendix B: Glycemic Control Targets and Monitoring,"in some patients based on other factors, balancing safety and tolerability of therapy. n Major comorbidity is present, but is not end-stage, and management is achievable. o Major comorbidity is present and is either end-stage or management is significantly challenging, including mental health co..."
2028,0.710746,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,causal effect on adverse events and whether prospectively reducing fasting blood glucose or HbA1c variability will favorably impact outcomes are also unknown. The Work Group systematically reviewed evidence related to this recommendation and considered the assessment of the evidence put forth in...
2027,0.709554,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"events.(75, 76, 83) The increased risk for hypoglycemia events in the setting of higher variability generally ranged from 50–300%. Notably, the median duration of follow-up ranged from 8 months to nearly 9 years, so the number of measures of fasting glucose or HbA1c differed substantially across..."
2022,0.707995,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"be burdensome for providers with limited time. Most screening tools were developed to target older patients, so their accuracy in younger patients is unclear. The Work Group systematically searched for evidence and did not identify any studies that met inclusion criteria regarding routine screen..."
2050,0.705465,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,"it is categorized as Reviewed, New-added. The Work Group’s confidence in the quality of the evidence was moderate. The body of evidence had some limitations, including a small sample size spread among relatively few studies.(92–95) We found moderate evidence that rtCGM led to decreased hypoglyce..."


**Вывод debug-проверки.**

Dense retrieval технически работает и возвращает тематически релевантные
chunks из правильных clinical guidelines.

В ходе проверки также была обнаружена и исправлена ошибка VA/DoD parser,
связанная с составными subsection headings.

Debug-примеры не используются для итоговой оценки качества retrieval,
поскольку их результаты уже анализировались в процессе разработки.

## 18. Подготовка evaluation-набора для retrieval

Debug-примеры использовались во время разработки и поэтому не подходят
для независимой оценки качества retrieval.

Для дальнейшей оценки используем новые вопросы только из `dev`.
`test` остаётся полностью замороженным.

Поскольку knowledge base покрывает ограниченный набор clinical guidelines,
не каждый вопрос из исходного датасета обязан иметь ответ в KB.

Поэтому сначала формируем набор кандидатов из `dev`, а затем вручную
определяем:

- можно ли ответить на вопрос с помощью текущей knowledge base;
- какой документ содержит релевантную информацию;
- какие разделы документа можно считать релевантными.

In [153]:
MATERIALS_DIR = Path("../materials")

dev_df = pd.read_csv(
    MATERIALS_DIR / "dev.csv"
)

print("dev shape:", dev_df.shape)
print("columns:", dev_df.columns.tolist())

dev_df.head()

dev shape: (1000, 2)
columns: ['input', 'output']


,input,output
0,my husband age is 29 yrs.height is 5.10ft.his sperm analysis report showsquantity,"Hello Thanks for writing to your husbands semen analysis report suggests infection. Infection is indicated by the presence of WBC in semen. Normally there shouldn't be any WBC in semen. Infection may be due to prostatitis, UTI etc. You need antibiotics for infection. You need few more investigat..."
1,"My HBeAg is non-reactive but my Anti-HBc is reactive. My liver is normal in size and it shows homogeneous echopattern. The intrahepatic ducts are not dilated and negatve for focal solid or sytic masses. Do I have hepa B and if I have, what is the condition of my health?",Hi dear thanks for asking question.... Noted you have anti HBC positive. You have not mentioned whether it is IG M or IG G type. Usually anti HBC is first antibody appear in Blood from current or past infection. IG M replaced by IG G and IG G type anti HE c remain for lie long. You might develop...
2,"hello Dr, The entire left side of my face has been swollen for nearly a week, my gums in that area is bubbling up, soft but scary feeling. This is very painful, I thought it would go down after a few days. But days pass and no kind of improvement has occured. Should I go to a doctor at a emergen...","Hi, in my opinion u should see a dentist at the earliest and not ignore it, as per your symptoms it seems to be an infection which needs to be investigated for the reason for infection and treating it with antibiotics and analgesics and treating the reason as well, if u want me to help me more u..."
3,"Hi, I have slight pain in my lower right abdomen when sneezing or coughing for a couple of weeks now. Does not hurt when I lift something, even very heavy (slightly feel it if try to do abdominal crunches). I am a 49 years old male, quite fit, exersize regularly. No nausea, no temperature or hig...","HI ! Good afternoon. I am Chat Doctor answering your query. If I were your doctor, I would clinically examine your abdomen to rule out chances of any inguinal hernia in its early stages when it becomes painful due to stretching of its covering. Also, I would like to see if there is any other int..."
4,I am a physician investigating a workers compensation claim in which the client claims he was bitten by a tic while at a work retreat in Georgia in May 2015. My question is whether Lyme disease is common in Georgia and whether it can cause Bell s Palsey.,"Hello, Yes, Lyme disease can cause Bells palsy. Caused by the bacterium Cordelia burgdorferi and transmitted to humans through the bite of infected black legged ticks, the typical symptoms of Lyme disease include fever, headache, fatigue, and a characteristic skin rash called Erythema migrant. I..."


### Исключение debug-вопросов

`debug` является подмножеством `dev` и уже использовался во время разработки pipeline.

Чтобы вопросы, которые мы уже видели, не попали в независимую retrieval evaluation,
исключаем их из `dev`.

Далее нам нужны только вопросы, потенциально относящиеся к тем клиническим темам,
которые представлены в текущей knowledge base.

In [154]:
debug_df = pd.read_csv(
    MATERIALS_DIR / "debug.csv"
)

dev_retrieval_df = (
    dev_df[
        ~dev_df["input"].isin(
            debug_df["input"]
        )
    ]
    .reset_index(drop=True)
    .copy()
)

print("dev:", len(dev_df))
print("debug:", len(debug_df))
print(
    "dev after debug exclusion:",
    len(dev_retrieval_df),
)

dev: 1000
debug: 30
dev after debug exclusion: 970


### Поиск вопросов, потенциально покрываемых knowledge base

Knowledge base содержит восемь clinical guidelines по ограниченному набору тем.

Поэтому случайная выборка из всего `dev` будет содержать много вопросов,
для которых в KB в принципе нет подходящего источника.

Используем простой keyword-фильтр только для предварительного отбора кандидатов
на ручную разметку.

Этот фильтр не определяет релевантность и не участвует в расчёте retrieval-метрик.
Окончательное решение о том, покрывается ли вопрос knowledge base, принимается
вручную по содержанию guideline.

In [155]:
KB_TOPIC_PATTERNS = {
    "asthma": (
        r"\basthma\b|"
        r"\bwheez\w*\b|"
        r"\bbronchial asthma\b"
    ),

    "ckd": (
        r"\bchronic kidney disease\b|"
        r"\bckd\b|"
        r"\brenal insufficien\w*\b|"
        r"\bkidney disease\b|"
        r"\begfr\b|"
        r"\bhyperkal\w*\b"
    ),

    "low_back_pain": (
        r"\blow back pain\b|"
        r"\blower back pain\b|"
        r"\blumbar pain\b|"
        r"\bsciatica\b"
    ),

    "major_depression": (
        r"\bmajor depressive\b|"
        r"\bdepress\w*\b|"
        r"\bantidepress\w*\b|"
        r"\bphq[- ]?9\b"
    ),

    "pregnancy": (
        r"\bpregnan\w*\b|"
        r"\bprenatal\b|"
        r"\bantenatal\b|"
        r"\bgestation\w*\b"
    ),

    "type2_diabetes": (
        r"\btype 2 diabetes\b|"
        r"\bt2dm\b|"
        r"\bdiabet\w*\b|"
        r"\bhba1c\b|"
        r"\ba1c\b"
    ),

    "hypertension": (
        r"\bhypertension\b|"
        r"\bhigh blood pressure\b"
    ),

    "sti": (
        r"\bsexually transmitted\b|"
        r"\bsti\b|"
        r"\bstd\b|"
        r"\bchlamyd\w*\b|"
        r"\bgonorrh\w*\b|"
        r"\bsyphilis\b|"
        r"\btrichomon\w*\b|"
        r"\bgenital herpes\b"
    ),
}

In [156]:
def detect_kb_topics(text):
    topics = []

    for topic, pattern in KB_TOPIC_PATTERNS.items():
        if re.search(
            pattern,
            str(text),
            flags=re.IGNORECASE,
        ):
            topics.append(topic)

    return topics

In [157]:
dev_retrieval_df["kb_topics"] = (
    dev_retrieval_df["input"]
    .apply(detect_kb_topics)
)

retrieval_candidates_df = (
    dev_retrieval_df[
        dev_retrieval_df["kb_topics"]
        .str.len()
        .gt(0)
    ]
    .reset_index(drop=True)
    .copy()
)

retrieval_candidates_df["candidate_id"] = [
    f"candidate_{i:04d}"
    for i in range(
        len(retrieval_candidates_df)
    )
]

print(
    "Candidate questions:",
    len(retrieval_candidates_df),
)

Candidate questions: 162


In [158]:
topic_counts = (
    retrieval_candidates_df[
        ["candidate_id", "kb_topics"]
    ]
    .explode("kb_topics")
    .groupby("kb_topics")
    .size()
    .sort_values(ascending=False)
    .rename("questions")
)

topic_counts

kb_topics
pregnancy           63
asthma              27
type2_diabetes      21
hypertension        18
low_back_pain       18
major_depression    14
sti                  7
ckd                  2
Name: questions, dtype: int64

### Формирование набора для ручной разметки

Количество потенциально подходящих вопросов заметно различается между темами.
Поэтому простая случайная выборка из всех кандидатов могла бы быть сильно
смещена в сторону наиболее частых тем, например pregnancy.

Для первого retrieval evaluation формируем стратифицированную выборку:

- до 5 вопросов на каждую клиническую тему;
- если доступно меньше 5 вопросов, используем все;
- выбор выполняется до запуска retrieval, чтобы результаты поиска не влияли
  на состав evaluation-набора.

Колонка `output` при разметке не используется, поскольку ответы исходного
датасета не рассматриваются как эталон клинической корректности.

In [159]:
EVAL_PER_TOPIC = 5
EVAL_RANDOM_STATE = 42

candidate_topics_df = (
    retrieval_candidates_df[
        [
            "candidate_id",
            "input",
            "kb_topics",
        ]
    ]
    .explode("kb_topics")
    .rename(
        columns={
            "kb_topics": "kb_topic"
        }
    )
    .reset_index(drop=True)
)

In [ ]:
retrieval_eval_candidates_df = (
    candidate_topics_df
    .groupby(
        "kb_topic",
        group_keys=False,
    )
    .apply(
        lambda group: group.sample(
            n=min(
                EVAL_PER_TOPIC,
                len(group),
            ),
            random_state=EVAL_RANDOM_STATE,
        )
    )
    .reset_index(drop=True)
)

In [161]:
retrieval_eval_candidates_df[
    "candidate_id"
].duplicated().sum()

np.int64(1)

In [162]:
retrieval_eval_candidates_df.groupby(
    "kb_topic"
).size()

kb_topic
asthma              5
ckd                 2
hypertension        5
low_back_pain       5
major_depression    5
pregnancy           5
sti                 5
type2_diabetes      5
dtype: int64

In [163]:
pd.set_option(
    "display.max_colwidth",
    500,
)

retrieval_eval_candidates_df[
    [
        "candidate_id",
        "kb_topic",
        "input",
    ]
]

,candidate_id,kb_topic,input
0,candidate_0063,asthma,"My 9 year old grandson who has a history of asthma but has not been on any medication nor had a reaction in at least 3 months had Eosinophils,absolute of 544 and AST of 33. doctor didnt seem at all concerned. Should there be any follow up or need for concern?"
1,candidate_0097,asthma,"ihave severe asthma and have been 5 times in the last 9 weeks and been given lots of steroids, so i am on insulin now. i am so wore ouy and can hardly get around. thw wheezing has stopped. i have high numbers of latic acid in the muscles. please tell me what to do."
2,candidate_0071,asthma,hi there i am 24 an a female im a mum of 4 last sunday i had this extreme pain on both sides of my spine which to this day i still have it i was subscribed codiene for the pain which didnt doanything i cant sleep at night as lying down hurt any movement to my back bring tears i cant touch my back also breathing coughing hurts... an i have now another problem that happend 2 nights ago an i had a very sharp pain in my heart which made my left arm go dead an all my finger tips went numb an felt...
3,candidate_0135,asthma,"I experience respiratory distress symptoms (SOB, decreased O2, voice changes) when I am exposed to perfumes and a few other strong oders and flying dust/dirt. What could this be a symptoms of?I am 37 years old, weigh 145, and have no hx of asthma or any other respiratory issues."
4,candidate_0004,asthma,Had a cold for 3 weeks. Went away. 2 weeks later came back. Doctor treated me with 5 day Z Pak (250 mg) & a corticosteroid pill (4 mg). Still sick. Bad cough. Cough up white foamy stuff. Using Mucinex & Robitussin. Lots of clear discharge from nose. Wheezing.
5,candidate_0067,ckd,hi i am 30 years old and my husband an i have been trying to conceive for 19 cycles now with no success. I was diagnosed with medullary sponge kidney 5 years ago and am wondering if this is causing us to not conceive. We have both been tested by fertility specialists and he was by urologist and we are both fine. Except this kidney disease and frequent uti s that i keep having. Your help and advise is greatly appreciated.
6,candidate_0029,ckd,"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be tested further"
7,candidate_0001,hypertension,"hi in 2010 I was diagnosed with the right right bundle branch block also with a sinus tachycardia and a heart murmur I recently went to the hospital and was diagnosed with chest wall pain and Mild tricuspid regurgitation and mild tricuspid pulmonic regurgitation, my question is what do I expect from this point on? Should I be concerned? Also diagnosed with a specified essential hypertension and ectopic atrial rhythm other specified cardiac dysrhythmias. Tachycardia unspecified? ? What does t..."
8,candidate_0003,hypertension,"I had chest pains, back pains going up to my throat and neck, tropamine in my blood. Heart rate 90 bpm. High blood pressure. An ambulance took me to hospital, 4 days of monitoring and an angiogram and echo. They do not know what it was.Still feel washed out and tired with a pain in my chest like I have been kicked. I am a fit 58 year old woman. Was diagnosed with BC in 2010 had chemo, radiotherapy, Herceptin.... and am on Tamoxifen at the moment."
9,candidate_0066,hypertension,I have hypertension currently 160 / 95 and has been for some weeks. I exercise regularly run 5k 3 times per week. I recently had gout but it has gone and I am back into regular exercise. I keep having pains in the base of my foot to the point it is difficult to walk and go extremely cold. I currently have 2 fleeces on.


### Удаление повторов между темами

Один медицинский вопрос может содержать признаки нескольких тем knowledge base.

Поэтому после стратифицированного отбора один и тот же `candidate_id`
может встретиться несколько раз.

Для ручной разметки каждый вопрос должен присутствовать только один раз.
При этом сохраняем полный список тем, найденных на предыдущем этапе.

In [164]:
selected_candidate_ids = (
    retrieval_eval_candidates_df[
        "candidate_id"
    ]
    .unique()
)

retrieval_eval_labeling_df = (
    retrieval_candidates_df[
        retrieval_candidates_df[
            "candidate_id"
        ].isin(
            selected_candidate_ids
        )
    ][
        [
            "candidate_id",
            "kb_topics",
            "input",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

print(
    "Rows before deduplication:",
    len(retrieval_eval_candidates_df),
)

print(
    "Unique evaluation questions:",
    len(retrieval_eval_labeling_df),
)

Rows before deduplication: 37
Unique evaluation questions: 36


### Ручная проверка покрытия knowledge base

Keyword-фильтр определяет только потенциальную тематическую связь вопроса
с knowledge base.

Наличие ключевого слова ещё не означает, что текущие guidelines содержат
информацию, достаточную для ответа на вопрос.

Поэтому каждый кандидат вручную классифицируется как:

- `covered` — вопрос содержательно покрывается текущей knowledge base;
- `not_covered` — тема упоминается, но нужной информации в KB нет;
- `uncertain` — требуется дополнительная проверка guideline.

Для вопросов со статусом `covered` дополнительно указываются релевантный документ
и один или несколько релевантных разделов.

In [165]:
retrieval_eval_labeling_df[
    "coverage"
] = ""

retrieval_eval_labeling_df[
    "relevant_document_id"
] = ""

retrieval_eval_labeling_df[
    "relevant_sections"
] = ""

retrieval_eval_labeling_df[
    "notes"
] = ""

retrieval_eval_labeling_df

,candidate_id,kb_topics,input,coverage,relevant_document_id,relevant_sections,notes
0,candidate_0000,"[low_back_pain, pregnancy]","Hi! I recently had been tested for pregnancy due to low back pain, headaches, nausea and no appetite. It came back negative. But now, I m constantly sick. I have lower abdomen pains and sharp stabbing center abdomen pains. I have also been very shaky with my hands. And I have no appetite. I have headaches that will last for days on end if I don t take ibuprofen every 6-8 hours. I m not sure what it is. Please help?!",,,,
1,candidate_0001,[hypertension],"hi in 2010 I was diagnosed with the right right bundle branch block also with a sinus tachycardia and a heart murmur I recently went to the hospital and was diagnosed with chest wall pain and Mild tricuspid regurgitation and mild tricuspid pulmonic regurgitation, my question is what do I expect from this point on? Should I be concerned? Also diagnosed with a specified essential hypertension and ectopic atrial rhythm other specified cardiac dysrhythmias. Tachycardia unspecified? ? What does t...",,,,
2,candidate_0003,[hypertension],"I had chest pains, back pains going up to my throat and neck, tropamine in my blood. Heart rate 90 bpm. High blood pressure. An ambulance took me to hospital, 4 days of monitoring and an angiogram and echo. They do not know what it was.Still feel washed out and tired with a pain in my chest like I have been kicked. I am a fit 58 year old woman. Was diagnosed with BC in 2010 had chemo, radiotherapy, Herceptin.... and am on Tamoxifen at the moment.",,,,
3,candidate_0004,[asthma],Had a cold for 3 weeks. Went away. 2 weeks later came back. Doctor treated me with 5 day Z Pak (250 mg) & a corticosteroid pill (4 mg). Still sick. Bad cough. Cough up white foamy stuff. Using Mucinex & Robitussin. Lots of clear discharge from nose. Wheezing.,,,,
4,candidate_0005,[major_depression],Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depr...,,,,
5,candidate_0008,[type2_diabetes],"Hi. In the morning, I rarely have an issue, but in the evening (by 3-5pm and after) all of my arm and leg veins are bulging. There is also change in color from pink to red and swollen feet/toes. I wake up with numb limbs throughout the night very frequently but have not been diagnosed diabetic, vein disorders nothing. I will drink more water but don t know if it would make it worse. I recently quit smoking after 15years, if that helps form an opinion. Thank you. Jeanie.",,,,
6,candidate_0010,[type2_diabetes],Hi My father is diabetic and had a stroke couple of years ago. Now he has developed sinus in his big toe of right leg. I consulted ortho specialists and they suggested surgery on the big toe. Is there alternative treatement as the blood circulation is low on the right leg and healing might take longer,,,,
7,candidate_0011,[sti],"I m 14 and I m sexually active, but I went 7 months without sex, and I received a hand job, then the shaft of my penis starting getting a rash and got a cut on it, it s red and spread to my balls, it has bumps and it hurts.. I don t think it s an std but you never know, it s been two months now and it s only gotten worse. My balls are red and have bumbs and do does my shaft, and my penis did but they disappeared, I m uncircumcised if that helps. It s only on the skin, nothing is on the head ...",,,,
8,candidate_0012,[pregnancy],"HI, I am 30 years old and my husband is the same age. We have been trying to conceive for almost two years. I had a missed miscarriage last year in September and had a sept

In [166]:
coverage_labels = {
    "candidate_0000": "not_covered",
    "candidate_0001": "not_covered",
    "candidate_0003": "not_covered",
    "candidate_0004": "uncertain",
    "candidate_0005": "uncertain",
    "candidate_0008": "not_covered",
    "candidate_0010": "uncertain",
    "candidate_0011": "uncertain",
    "candidate_0012": "not_covered",
    "candidate_0014": "covered",
    "candidate_0023": "not_covered",
    "candidate_0029": "covered",
    "candidate_0030": "not_covered",
    "candidate_0035": "uncertain",
    "candidate_0048": "covered",
    "candidate_0058": "not_covered",
    "candidate_0063": "not_covered",
    "candidate_0066": "not_covered",
    "candidate_0067": "not_covered",
    "candidate_0070": "not_covered",
    "candidate_0071": "not_covered",
    "candidate_0076": "not_covered",
    "candidate_0078": "covered",
    "candidate_0091": "not_covered",
    "candidate_0097": "uncertain",
    "candidate_0105": "uncertain",
    "candidate_0109": "uncertain",
    "candidate_0116": "not_covered",
    "candidate_0131": "not_covered",
    "candidate_0135": "uncertain",
    "candidate_0139": "not_covered",
    "candidate_0143": "not_covered",
    "candidate_0144": "not_covered",
    "candidate_0145": "uncertain",
    "candidate_0151": "uncertain",
    "candidate_0160": "not_covered",
}

In [167]:
retrieval_eval_labeling_df["coverage"] = (
    retrieval_eval_labeling_df[
        "candidate_id"
    ].map(coverage_labels)
)

retrieval_eval_labeling_df[
    "coverage"
].value_counts()

coverage
not_covered    21
uncertain      11
covered         4
Name: count, dtype: int64

In [168]:
topic_coverage_df = (
    retrieval_eval_labeling_df[
        [
            "candidate_id",
            "kb_topics",
            "coverage",
        ]
    ]
    .explode("kb_topics")
)

topic_coverage_df.groupby(
    [
        "kb_topics",
        "coverage",
    ]
).size()

kb_topics         coverage   
asthma            not_covered    2
                  uncertain      3
ckd               covered        1
                  not_covered    1
hypertension      not_covered    4
                  uncertain      1
low_back_pain     covered        3
                  not_covered    2
major_depression  not_covered    2
                  uncertain      3
pregnancy         not_covered    7
sti               not_covered    3
                  uncertain      2
type2_diabetes    not_covered    3
                  uncertain      2
dtype: int64

In [169]:
uncertain_questions_df = (
    retrieval_eval_labeling_df[
        retrieval_eval_labeling_df[
            "coverage"
        ].eq("uncertain")
    ][
        [
            "candidate_id",
            "kb_topics",
            "input",
        ]
    ]
    .reset_index(drop=True)
)

pd.set_option(
    "display.max_colwidth",
    None,
)

uncertain_questions_df

,candidate_id,kb_topics,input
0,candidate_0004,[asthma],Had a cold for 3 weeks. Went away. 2 weeks later came back. Doctor treated me with 5 day Z Pak (250 mg) & a corticosteroid pill (4 mg). Still sick. Bad cough. Cough up white foamy stuff. Using Mucinex & Robitussin. Lots of clear discharge from nose. Wheezing.
1,candidate_0005,[major_depression],Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?
2,candidate_0010,[type2_diabetes],Hi My father is diabetic and had a stroke couple of years ago. Now he has developed sinus in his big toe of right leg. I consulted ortho specialists and they suggested surgery on the big toe. Is there alternative treatement as the blood circulation is low on the right leg and healing might take longer
3,candidate_0011,[sti],"I m 14 and I m sexually active, but I went 7 months without sex, and I received a hand job, then the shaft of my penis starting getting a rash and got a cut on it, it s red and spread to my balls, it has bumps and it hurts.. I don t think it s an std but you never know, it s been two months now and it s only gotten worse. My balls are red and have bumbs and do does my shaft, and my penis did but they disappeared, I m uncircumcised if that helps. It s only on the skin, nothing is on the head of my penis, I m scared to go to the doctor. And puss come out of my balls and shaft and it s yellow, please help me. Thank you."
4,candidate_0035,[hypertension],"my blood pressure readings today have been 106/69, 119/62, and 108/64 are any of these low? yesterday my reading was 80/50 at the drs ofc. im currently on high blood pressure meds. and lost 25lbs. dr. my take me off blood pressure med. he said to monitor it til tomorrow."
5,candidate_0097,[asthma],"ihave severe asthma and have been 5 times in the last 9 weeks and been given lots of steroids, so i am on insulin now. i am so wore ouy and can hardly get around. thw wheezing has stopped. i have high numbers of latic acid in the muscles. please tell me what to do."
6,candidate_0105,[major_depression],"What mental disorder do I have? I m really depressed so it s hard for me to type. I ve just had to re write this so I m very angry, but i ll just put symptoms. Symptoms Used to hear voices (first of my guardian angel and then from god- lasted from January- till Christmas day 2012- only very rarely talk to him agin) - Have many beliefs (voices explained to me about the universe and how he created it from experiments on planets in the solar systems he had made then he let it all die out so that he could use all that power to create a world big enough to plant and create a cycle of many creations and many other stories) - laugh at innopropiate things- laugh at sad things or serious things (that inside i know is not funny, but I can t help it) such as people crying, people shouting or being shouted at, people saying there loved one had died or something sad had happened to them. Don t laugh at funny things, like people laughing so badly at a joke, that I don t think is funny, and I m the only one who doesn t get the joke and I look miserable. - sleeping patterns- cant get to sleep until at least 3am and don t get up until at least 11am. - scared of people- walking past people or seeing people that aren t my sister, my mother, sometimes my father or me makes me disturbed, petrified, paranoid and a panic attack feeling. - staying in bed all the time- always in my bed away from everyone either sleeping, hiding under bed sheets or planning a new life (obses

In [170]:
resolved_coverage = {
    "candidate_0004": "not_covered",   # cough/wheezing after infection; слишком широкий respiratory differential
    "candidate_0005": "covered",       # depression + необходимость помощи + non-pharmacologic treatment
    "candidate_0010": "not_covered",   # diabetic foot lesion + surgery/vascular issue
    "candidate_0011": "not_covered",   # genital rash/pus без установленной STI; широкий differential
    "candidate_0035": "not_covered",   # hypotension на antihypertensives, вопрос об отмене препарата
    "candidate_0097": "covered",       # severe asthma, repeated steroid courses, poor control
    "candidate_0105": "not_covered",   # psychosis/hallucinations/suicidality, вопрос о psychiatric diagnosis
    "candidate_0109": "not_covered",   # acute severe hyperglycemia / possible emergency
    "candidate_0135": "covered",       # симптомы, провоцируемые irritants; asthma diagnosis/assessment
    "candidate_0145": "covered",       # Ureaplasma / STI-related question
    "candidate_0151": "not_covered",   # possible antidepressant-induced mania; immediate medication decision
}

for candidate_id, coverage in resolved_coverage.items():
    retrieval_eval_labeling_df.loc[
        retrieval_eval_labeling_df["candidate_id"].eq(candidate_id),
        "coverage",
    ] = coverage

In [171]:
retrieval_eval_labeling_df[
    "coverage"
].value_counts()

coverage
not_covered    28
covered         8
Name: count, dtype: int64

### Разметка covered-вопросов

Для независимой оценки retrieval оставляем только вопросы, которые однозначно
покрываются текущей knowledge base.

Для каждого такого вопроса вручную определяем:

- релевантный clinical guideline;
- один или несколько разделов, содержащих информацию для ответа.

Разметка выполняется по структуре knowledge base до просмотра результатов dense retrieval.

In [172]:
covered_dev_eval_df = (
    retrieval_eval_labeling_df[
        retrieval_eval_labeling_df[
            "coverage"
        ].eq("covered")
    ]
    .reset_index(drop=True)
    .copy()
)

covered_dev_eval_df[
    [
        "candidate_id",
        "kb_topics",
        "input",
    ]
]

,candidate_id,kb_topics,input
0,candidate_0005,[major_depression],Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?
1,candidate_0014,[low_back_pain],"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to determine the cause of this?."
2,candidate_0029,[ckd],"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be tested further"
3,candidate_0048,[low_back_pain],hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but no cure now my doctor suggest me duzella 30 mg so how much its useful for me pls suggest me some good medicine iam fearing tht any side affects will come because my marriage is thier in a month and iam having high uric acid 9.0
4,candidate_0078,[low_back_pain],"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable."
5,candidate_0097,[asthma],"ihave severe asthma and have been 5 times in the last 9 weeks and been given lots of steroids, so i am on insulin now. i am so wore ouy and can hardly get around. thw wheezing has stopped. i have high numbers of latic acid in the muscles. please tell me what to do."
6,candidate_0135,[asthma],"I experience respiratory distress symptoms (SOB, decreased O2, voice changes) when I am exposed to perfumes and a few other strong oders and flying dust/dirt. What could this be a symptoms of?I am 37 years old, weigh 145, and have no hx of asthma or any other respiratory issues."
7,candidate_0145,[sti],"Hi, may I answer your health queries right now ? Hello i am a health 29 yr old female with two children and my boyfriend is a divorcee with no children however he recently said he was diagnosed with ureaplasma, while he has been overseas in Bahrain in the military and we havent been sexually active since Oct 2010. And he thinks it may have been from me however, i have never had any abnormalities in my pap smears in all my years (since 16). I would like to know what exactly causes ureaplasma in males..ive heard several things, it can be from a female carrier (who is infertile like his ex-wife) or from anal.But i am just not sure. I just want to get some clear answers...could I have given this to him Ive had BV since weve been sexually active... Is he the cause for it? What causes ureaplasma? Is it linked to HPV? Do I have reason to believe hes been unfaithful since he got to Bahrain if this is an STD or could he have contracted this from his ex-wife since the organism has been known to lie dormant for a while ....thank u for your time"


In [173]:
relevant_documents = {
    "candidate_0005": "va_dod_major_depression_2022",
    "candidate_0014": "va_dod_low_back_pain_2022",
    "candidate_0029": "va_dod_ckd_2025",
    "candidate_0048": "va_dod_low_back_pain_2022",
    "candidate_0078": "va_dod_low_back_pain_2022",
    "candidate_0097": "va_dod_asthma_2025",
    "candidate_0135": "va_dod_asthma_2025",
    "candidate_0145": "cdc_sti_2021",
}

covered_dev_eval_df[
    "relevant_document_id"
] = (
    covered_dev_eval_df[
        "candidate_id"
    ].map(relevant_documents)
)

In [174]:
def show_document_sections(document_id):
    return (
        retrieval_records_df[
            retrieval_records_df[
                "document_id"
            ].eq(document_id)
        ][
            "section_path"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

In [175]:
for section in show_document_sections(
    "va_dod_asthma_2025"
):
    print(section)

Appendix C: Assessments of Asthma Severity and Control > A. Assessment of Asthma Severity
Appendix C: Assessments of Asthma Severity and Control > B. Assessment of Asthma Control
Appendix C: Assessments of Asthma Severity and Control > C. Indications for Specialist Referral
Appendix C: Assessments of Asthma Severity and Control > D. Identifying Alternative Diagnoses
Appendix D: Details of a Comprehensive History and Physical Exam > A. Details of a Comprehensive History
Appendix D: Details of a Comprehensive History and Physical Exam > B. Details of a Comprehensive Physical Exam
Appendix F: Example Asthma Action Plan Templates
Appendix F: Example Asthma Action Plan Templates > A. National Heart, Lung, and Blood Institute Asthma Action Plan Example Template
Appendix F: Example Asthma Action Plan Templates > B. Department of Defense Asthma Action Plan Template
Appendix F: Example Asthma Action Plan Templates > C. CDC Asthma Action Plan Templates
Appendix G: Additional Information on Pharm

In [176]:
asthma_relevant_sections = {
    "candidate_0097": [
        (
            "Appendix C: Assessments of Asthma Severity and Control "
            "> B. Assessment of Asthma Control"
        ),
        (
            "Appendix C: Assessments of Asthma Severity and Control "
            "> C. Indications for Specialist Referral"
        ),
        "IX. Recommendations > B. Treatment and Management",
    ],

    "candidate_0135": [
        "IX. Recommendations > A. Diagnosis and Assessment",
        (
            "Appendix C: Assessments of Asthma Severity and Control "
            "> D. Identifying Alternative Diagnoses"
        ),
        (
            "Appendix D: Details of a Comprehensive History and Physical Exam "
            "> A. Details of a Comprehensive History"
        ),
        (
            "Appendix K: Alternative Text Descriptions of Algorithms "
            "> A. Module A: Assessment and Diagnosis of Asthma"
        ),
    ],
}

In [177]:
for candidate_id, sections in asthma_relevant_sections.items():
    covered_dev_eval_df.loc[
        covered_dev_eval_df["candidate_id"].eq(candidate_id),
        "relevant_sections",
    ] = pd.Series(
        [sections],
        index=covered_dev_eval_df.index[
            covered_dev_eval_df["candidate_id"].eq(candidate_id)
        ],
    )

In [178]:
for candidate_id, sections in asthma_relevant_sections.items():
    covered_dev_eval_df.loc[
        covered_dev_eval_df["candidate_id"].eq(candidate_id),
        "relevant_sections",
    ] = pd.Series(
        [sections],
        index=covered_dev_eval_df.index[
            covered_dev_eval_df["candidate_id"].eq(candidate_id)
        ],
    )

In [179]:
covered_dev_eval_df[
    covered_dev_eval_df["candidate_id"].isin(
        ["candidate_0097", "candidate_0135"]
    )
][
    [
        "candidate_id",
        "relevant_document_id",
        "relevant_sections",
    ]
]

,candidate_id,relevant_document_id,relevant_sections
5,candidate_0097,va_dod_asthma_2025,"[Appendix C: Assessments of Asthma Severity and Control > B. Assessment of Asthma Control, Appendix C: Assessments of Asthma Severity and Control > C. Indications for Specialist Referral, IX. Recommendations > B. Treatment and Management]"
6,candidate_0135,va_dod_asthma_2025,"[IX. Recommendations > A. Diagnosis and Assessment, Appendix C: Assessments of Asthma Severity and Control > D. Identifying Alternative Diagnoses, Appendix D: Details of a Comprehensive History and Physical Exam > A. Details of a Comprehensive History, Appendix K: Alternative Text Descriptions of Algorithms > A. Module A: Assessment and Diagnosis of Asthma]"


In [180]:
for section in show_document_sections(
    "va_dod_major_depression_2022"
):
    print(section)

Appendix G: Alternative Text Descriptions of Algorithm
Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > A. Purpose
Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > B. Scoring the PHQ-9 (223)
Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > C. Using the PHQ-9 in Measurement-Based Care
Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > D. Example of Using the PHQ-9 in Clinical Practice
Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > E. Additional Clinical Considerations
Appendix J: Pharmacotherapy
Appendix K: Definitions > A. Major Depressive Disorder
Appendix K: Definitions > B. Treatments
I. Introduction
II. Background > A. Description of Major Depressive Disorder (MDD)
II. Background > B. Epidemiology and Impact on the General Population
II. Background > C. Major Depressive Disorder in the Department of Defense and the

In [181]:
mdd_relevant_sections = {
    "candidate_0005": [
        "VIII. Algorithm > A. Module A: Initial Assessment and Treatment",
        "IX. Recommendations > C. Treatment Setting",
        "IX. Recommendations > D. Treatment of Uncomplicated MDD",
        (
            "IX. Recommendations > H. Self-help, "
            "Complementary, and Alternative Treatments"
        ),
    ],
}

In [182]:
for candidate_id, sections in mdd_relevant_sections.items():
    covered_dev_eval_df.loc[
        covered_dev_eval_df["candidate_id"].eq(candidate_id),
        "relevant_sections",
    ] = pd.Series(
        [sections],
        index=covered_dev_eval_df.index[
            covered_dev_eval_df["candidate_id"].eq(candidate_id)
        ],
    )

In [183]:
covered_dev_eval_df[
    covered_dev_eval_df["candidate_id"].eq(
        "candidate_0005"
    )
][
    [
        "candidate_id",
        "relevant_document_id",
        "relevant_sections",
    ]
]

,candidate_id,relevant_document_id,relevant_sections
0,candidate_0005,va_dod_major_depression_2022,"[VIII. Algorithm > A. Module A: Initial Assessment and Treatment, IX. Recommendations > C. Treatment Setting, IX. Recommendations > D. Treatment of Uncomplicated MDD, IX. Recommendations > H. Self-help, Complementary, and Alternative Treatments]"


In [184]:
for section in show_document_sections(
    "va_dod_low_back_pain_2022"
):
    print(section)

Appendix E: Dosing for Select Pharmacologic Agentsa,b
Appendix F: Glossary
Appendix I: Alternative Text Descriptions of Algorithm
Appendix I: Alternative Text Descriptions of Algorithm > A. Module A: Initial Evaluation of Low Back Pain
Appendix I: Alternative Text Descriptions of Algorithm > B. Module B: Management of Low Back Pain
I. Introduction
II. Background > A. Description of Low Back Pain
II. Background > B. Epidemiology and Impact
III. Scope of this Guideline
III. Scope of this Guideline > A. Guideline Audience
III. Scope of this Guideline > B. Guideline Population
IV. Highlighted Features of this Guideline > A. Highlights in this Guideline Update
IV. Highlighted Features of this Guideline > B. Components of the Guideline
IX. Recommendations
IX. Recommendations > A. Evaluation and Diagnostic Approach
IX. Recommendations > B. Patient Education and Self-care
IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy
IX. Recommendations > D. Pharmacotherapy
IX. Recommenda

In [185]:
lbp_relevant_sections = {
    "candidate_0014": [
        "IX. Recommendations > A. Evaluation and Diagnostic Approach",
        "VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain",
        (
            "Appendix I: Alternative Text Descriptions of Algorithm "
            "> A. Module A: Initial Evaluation of Low Back Pain"
        ),
    ],

    "candidate_0048": [
        "IX. Recommendations > D. Pharmacotherapy",
        (
            "IX. Recommendations > C. "
            "Non-pharmacologic and Non-invasive Therapy"
        ),
        "VIII. Algorithm > B. Module B: Management of Low Back Pain",
        (
            "Appendix I: Alternative Text Descriptions of Algorithm "
            "> B. Module B: Management of Low Back Pain"
        ),
    ],

    "candidate_0078": [
        "IX. Recommendations > A. Evaluation and Diagnostic Approach",
        "VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain",
        "VIII. Algorithm > B. Module B: Management of Low Back Pain",
    ],
}

In [186]:
for candidate_id, sections in lbp_relevant_sections.items():
    covered_dev_eval_df.loc[
        covered_dev_eval_df["candidate_id"].eq(candidate_id),
        "relevant_sections",
    ] = pd.Series(
        [sections],
        index=covered_dev_eval_df.index[
            covered_dev_eval_df["candidate_id"].eq(candidate_id)
        ],
    )

In [187]:
covered_dev_eval_df[
    covered_dev_eval_df["candidate_id"].isin(
        [
            "candidate_0014",
            "candidate_0048",
            "candidate_0078",
        ]
    )
][
    [
        "candidate_id",
        "relevant_document_id",
        "relevant_sections",
    ]
]

,candidate_id,relevant_document_id,relevant_sections
1,candidate_0014,va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > A. Module A: Initial Evaluation of Low Back Pain]"
3,candidate_0048,va_dod_low_back_pain_2022,"[IX. Recommendations > D. Pharmacotherapy, IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy, VIII. Algorithm > B. Module B: Management of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > B. Module B: Management of Low Back Pain]"
4,candidate_0078,va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, VIII. Algorithm > B. Module B: Management of Low Back Pain]"


In [188]:
for section in show_document_sections(
    "va_dod_ckd_2025"
):
    print(section)

Appendix G. Alternative Text Descriptions of Algorithms
Appendix H. Management of CKD Table
Appendix I. Monitoring of CKD Table
Appendix J. Approaches for eGFR Calculation
Appendix K. Nephrotoxic Agents and Medication Dose Adjustments in CKD > A. Background
Appendix K. Nephrotoxic Agents and Medication Dose Adjustments in CKD > B. Nephrotoxic Medications
Appendix K. Nephrotoxic Agents and Medication Dose Adjustments in CKD > C. Medication Management in CKD
Appendix L. List of Pharmacotherapies
Appendix M. Management of Hyperkalemia > A. Background
Appendix M. Management of Hyperkalemia > B. Definition
Appendix M. Management of Hyperkalemia > C. Management
Appendix M. Management of Hyperkalemia > D. Dietary Considerations
Appendix M. Management of Hyperkalemia > E. Use of Potassium Binders
Appendix N. Chronic Pain Management in CKD > A. Overview and Conceptual Approach
Appendix N. Chronic Pain Management in CKD > B. Non-Pharmacologic Pain Management in Patients with CKD
Appendix N. Chro

### Проверка покрытия CKD-вопроса

Вопрос `candidate_0029` касается advanced CKD, dialysis, transplant и необходимости
дальнейшего обследования.

По названиям разделов нельзя уверенно определить, покрывает ли guideline эти вопросы.
Поэтому перед присвоением `covered` проверяем наличие соответствующей информации
непосредственно в тексте CKD records.

In [189]:
ckd_coverage_check = (
    retrieval_records_df[
        retrieval_records_df[
            "document_id"
        ].eq("va_dod_ckd_2025")
        &
        retrieval_records_df[
            "text"
        ].str.contains(
            (
                r"dialysis|"
                r"transplant|"
                r"kidney replacement|"
                r"renal replacement|"
                r"nephrolog"
            ),
            case=False,
            regex=True,
            na=False,
        )
    ][
        [
            "section_path",
            "pdf_page",
            "text",
        ]
    ]
)

pd.set_option(
    "display.max_colwidth",
    500,
)

ckd_coverage_check

,section_path,pdf_page,text
727,II. Background > A. Description of Chronic Kidney Disease,6,"Chronic kidney disease (CKD) is defined as abnormalities of kidney structure or function characterized by a glomerular filtration rate (GFR) of less than 60 mL/min/1.73m2 or a normal GFR with other markers of kidney disease such as proteinuria, hematuria, or abnormal imaging of the kidneys, present for greater than three months, with implications for health.(3) In 2002, the National Kidney Foundation (NKF) published treatment guidelines that classified five stages of CKD based on declining e..."
728,II. Background > A. Description of Chronic Kidney Disease,7,"stages of CKD and the relative risk of these complications are presented in Sidebar 9. The majority of patients with CKD are asymptomatic until CKD stage G5 when uremic symptoms develop, at which time kidney replacement therapy (KRT) may be recommended depending on patients’ co-occurring conditions and values. Interventions described in this CPG can slow the progression of CKD, improve cardiovascular outcomes, and reduce mortality. Testing within populations at high risk for CKD is suggested..."
730,II. Background > C. Chronic Kidney Disease in the Department of Veterans Affairs and the Department of Defense,7,"CKD prevalence is likely underestimated within the VA and DOD health systems. First, the presence of CKD could be an exclusion criterion for continued active service, thereby disincentivizing screening and non-disclosure, if known. Second, testing for CKD remains low, even among individuals with known risk factors such as diabetes mellitus. The National Health and Nutrition Examination Survey (NHANES) from 1999 to 2016 showed that the prevalence of CKD stages G1-G4 based on the presence of a..."
733,II. Background > D. Social Determinants of Health,9,"CKD within the VA between 2005-2016, followed for up to 10 years, found that Black Veterans were on average 7.8 years younger than White Veterans at CKD onset and had persistently >2-fold higher likelihood of requiring KRT, consistent with trends observed in the general U.S. population. A cohort study of VA patients with incident stage G3 and G4 CKD found that both Black and Hispanic patients experienced faster progression to stage G5 CKD compared with non-Hispanic White patients, despite hi..."
734,II. Background > D. Social Determinants of Health,10,"medical care – such as making informed referrals to specialists – enhances patient-centered care. This approach ensures that each patient receives appropriate treatment tailored to their specific needs and risks. Healthcare systems and primary care providers (PCPs) play a crucial role in facilitating early CKD diagnosis, managing CKD risk factors, and educating patients. Incorporating CKD detection into the workflow for preventive visits, automating lab monitoring, implementing care manageme..."
743,VII. Approach to Care in the Department of Veterans Affairs and the Department of Defense > B. Shared Decision-Making,18,"This CPG encourages providers to practice shared decision-making (SDM), a process in which providers, patients, and patient care partners (e.g., family, friends, caregivers) consider clinical evidence of benefits and risks as well as patient values and preferences to make decisions regarding the patient’s treatment.(56) Shared decision-making is emphasized in “Crossing the Quality Chasm”, an Institute of Medicine, now NAM, report in 2001 (57) and is a core component of a patient-centered, wh..."
753,VIII. Algorithm,27,"Module D. Pharmacologic Management of CKD in Patients Not on Dialysis The WG recommends stepwise addition of pharmacotherapies to slow progression of CKD and reduce MACE, noting that trials of SGLT2i, GLP-1 RA, and finerenone were each conducted on a background of ACEI/ARB therapy; however, the benefits of various combinations are unknown. * Strongest evidence for kidney protection with ACEI/ARB is in UACR>300 mg/g. ** In patients with HF, sacubitri

In [190]:
ckd_relevant_sections = {
    "candidate_0029": [
        "IX. Recommendations",
        "VIII. Algorithm",
        "Appendix G. Alternative Text Descriptions of Algorithms",
        "Appendix I. Monitoring of CKD Table",
    ],
}

In [191]:
for candidate_id, sections in ckd_relevant_sections.items():
    covered_dev_eval_df.loc[
        covered_dev_eval_df["candidate_id"].eq(candidate_id),
        "relevant_sections",
    ] = pd.Series(
        [sections],
        index=covered_dev_eval_df.index[
            covered_dev_eval_df["candidate_id"].eq(candidate_id)
        ],
    )

In [192]:
for section in show_document_sections(
    "cdc_sti_2021"
):
    print(section)

Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Diagnostic Considerations
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Follow-Up
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Management of Sex Partners
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Other Management Considerations
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Special Considerations > HIV Infection
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Special Considerations > Pregnancy
Chlamydial Infections > Chlamydial Infection Among Adolescents and Adults > Treatment
Chlamydial Infections > Chlamydial Infection Among Infants and Children
Chlamydial Infections > Chlamydial Infection Among Infants and Children > Diagnostic Considerations
Chlamydial Infections > Chlamydial Infectio

### Проверка покрытия вопроса про Ureaplasma

В CDC guideline нет отдельного раздела `Ureaplasma`.

Перед включением вопроса в evaluation проверяем, обсуждается ли Ureaplasma
внутри текста других разделов, например urethritis или cervicitis.

In [193]:
ureaplasma_check = (
    retrieval_records_df[
        retrieval_records_df["document_id"].eq(
            "cdc_sti_2021"
        )
        &
        retrieval_records_df["text"].str.contains(
            r"ureaplasma",
            case=False,
            regex=True,
            na=False,
        )
    ][
        [
            "section_path",
            "pdf_page",
            "text",
        ]
    ]
)

pd.set_option(
    "display.max_colwidth",
    500,
)

ureaplasma_check

,section_path,pdf_page,text
305,Diseases Characterized by Urethritis and Cervicitis > Urethritis > Etiology,62,"Multiple organisms can cause infectious urethritis. The presence of gram-negative intracellular diplococci (GNID) or purple intracellular diplococci (MB or GV) on urethral smear is indicative of presumed gonococcal infection, which is frequently accompanied by chlamydial infection. Nongonococcal urethritis (NGU), which is diagnosed when microscopy of urethral secretions indicate inflammation without GNID or MB or GV purple intracellular diplococci, is caused by C. trachomatis in 15%–40% of c..."
320,Diseases Characterized by Urethritis and Cervicitis > Cervicitis > Etiology,66,"C. trachomatis or N. gonorrhoeae is the most common etiology of cervicitis defined by diagnostic testing. Trichomoniasis, genital herpes (especially primary HSV-2 infection), or M. genitalium (761,765–768) also have been associated with cervicitis. However, in many cases of cervicitis, no organism is isolated, especially among women at relatively low risk for recent acquisition of these STIs (e.g., women aged >30 years) (769). Limited data indicate that BV and frequent douching might cause c..."


### Уточнение покрытия вопроса про Ureaplasma

Ureaplasma упоминается в разделах об этиологии urethritis и cervicitis.

Проверяем полный текст этих фрагментов, чтобы определить, содержит ли guideline
информацию, достаточную для ответа на основной вопрос пользователя.

In [194]:
for _, row in ureaplasma_check.iterrows():
    print("\n" + "=" * 100)
    print("SECTION:", row["section_path"])
    print("PDF PAGE:", row["pdf_page"])
    print()
    print(row["text"])


SECTION: Diseases Characterized by Urethritis and Cervicitis > Urethritis > Etiology
PDF PAGE: 62

Multiple organisms can cause infectious urethritis. The presence of gram-negative intracellular diplococci (GNID) or purple intracellular diplococci (MB or GV) on urethral smear is indicative of presumed gonococcal infection, which is frequently accompanied by chlamydial infection. Nongonococcal urethritis (NGU), which is diagnosed when microscopy of urethral secretions indicate inflammation without GNID or MB or GV purple intracellular diplococci, is caused by C. trachomatis in 15%–40% of cases; however, prevalence varies by age group, with a lower proportion of disease occurring among older men (699). Documentation of chlamydial infection as NGU etiology is essential because of the need for partner referral for evaluation and treatment to prevent complications of chlamydia, especially for female partners. Complications of C. trachomatis–associated NGU among males include epididymitis, 

In [195]:
retrieval_eval_labeling_df.loc[
    retrieval_eval_labeling_df["candidate_id"].eq("candidate_0145"),
    "coverage",
] = "not_covered"

retrieval_eval_labeling_df.loc[
    retrieval_eval_labeling_df["candidate_id"].eq("candidate_0145"),
    "notes",
] = (
    "Ureaplasma is mentioned in CDC STI guideline, "
    "but the KB does not sufficiently cover the main questions "
    "about transmission, source of infection, HPV/BV association, "
    "or inference of infidelity."
)

In [196]:
covered_dev_eval_df = (
    retrieval_eval_labeling_df[
        retrieval_eval_labeling_df["coverage"].eq("covered")
    ]
    .reset_index(drop=True)
    .copy()
)

covered_dev_eval_df[
    [
        "candidate_id",
        "kb_topics",
        "input",
    ]
]

,candidate_id,kb_topics,input
0,candidate_0005,[major_depression],Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depr...
1,candidate_0014,[low_back_pain],"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to determine the cause of this?."
2,candidate_0029,[ckd],"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be tested further"
3,candidate_0048,[low_back_pain],hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but no cure now my doctor suggest me duzella 30 mg so how much its useful for me pls suggest me some good medicine iam fearing tht any side affects will come because my marriage is thier in a month and ia...
4,candidate_0078,[low_back_pain],"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable."
5,candidate_0097,[asthma],"ihave severe asthma and have been 5 times in the last 9 weeks and been given lots of steroids, so i am on insulin now. i am so wore ouy and can hardly get around. thw wheezing has stopped. i have high numbers of latic acid in the muscles. please tell me what to do."
6,candidate_0135,[asthma],"I experience respiratory distress symptoms (SOB, decreased O2, voice changes) when I am exposed to perfumes and a few other strong oders and flying dust/dirt. What could this be a symptoms of?I am 37 years old, weigh 145, and have no hx of asthma or any other respiratory issues."


In [197]:
retrieval_eval_labeling_df["coverage"].value_counts()

coverage
not_covered    29
covered         7
Name: count, dtype: int64

### Guideline-grounded evaluation set

Реальные вопросы из `dev` плохо совпадают со scope текущей knowledge base:
после ручной проверки осталось только 7 однозначно покрываемых вопросов.

Поэтому дополнительно создаётся контролируемый retrieval benchmark
непосредственно по содержанию clinical guidelines.

Для каждого документа формулируются естественные вопросы, на которые
есть явный ответ в knowledge base.

Вопросы:

- не копируют формулировки заголовков дословно;
- создаются до запуска dense retrieval;
- имеют заранее определённые релевантные разделы;
- распределены по всем документам knowledge base.

Результаты на real-dev и guideline-grounded вопросах будут оцениваться отдельно.

In [198]:
guideline_eval_rows = []

In [199]:
for section in show_document_sections(
    "who_hypertension_2021"
):
    print(section)

1 Introduction
3 Recommendations > 3.1 Blood pressure threshold for initiation of pharmacological treatment
3 Recommendations > 3.2 Laboratory testing before and during pharmacological treatment
3 Recommendations > 3.3 Cardiovascular disease risk assessment as guide to initiation of
3 Recommendations > 3.4 Drug classes to be used as first-line agents
3 Recommendations > 3.5 Combination therapy
3 Recommendations > 3.6 Target blood pressure
3 Recommendations > 3.7 Frequency of re-assessment
3 Recommendations > 3.8 Administration of treatment by nonphysician professionals
4 Special settings > 4.1 Hypertension in disaster, humanitarian and emergency settings
4 Special settings > 4.2 COVID-19 and hypertension
4 Special settings > 4.3 Pregnancy and hypertension
6 Implementation tools > 6.1 Guideline recommendations
6 Implementation tools > 6.2 Drug- and dose-specific protocols


In [200]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "who_01",
            "question_source": "guideline",
            "query": (
                "At what blood pressure level should pharmacological "
                "treatment for hypertension be started?"
            ),
            "relevant_document_id": "who_hypertension_2021",
            "relevant_sections": [
                (
                    "3 Recommendations > "
                    "3.1 Blood pressure threshold for initiation "
                    "of pharmacological treatment"
                )
            ],
        },
        {
            "eval_id": "who_02",
            "question_source": "guideline",
            "query": (
                "What laboratory tests should be considered before "
                "starting antihypertensive medication and during treatment?"
            ),
            "relevant_document_id": "who_hypertension_2021",
            "relevant_sections": [
                (
                    "3 Recommendations > "
                    "3.2 Laboratory testing before and during "
                    "pharmacological treatment"
                )
            ],
        },
        {
            "eval_id": "who_03",
            "question_source": "guideline",
            "query": (
                "Which classes of medication are recommended as "
                "first-line treatment for hypertension?"
            ),
            "relevant_document_id": "who_hypertension_2021",
            "relevant_sections": [
                (
                    "3 Recommendations > "
                    "3.4 Drug classes to be used as first-line agents"
                )
            ],
        },
        {
            "eval_id": "who_04",
            "question_source": "guideline",
            "query": (
                "When should combination drug therapy be used "
                "for treating hypertension?"
            ),
            "relevant_document_id": "who_hypertension_2021",
            "relevant_sections": [
                "3 Recommendations > 3.5 Combination therapy"
            ],
        },
        {
            "eval_id": "who_05",
            "question_source": "guideline",
            "query": (
                "What blood pressure target should treatment aim for "
                "in adults receiving therapy for hypertension?"
            ),
            "relevant_document_id": "who_hypertension_2021",
            "relevant_sections": [
                "3 Recommendations > 3.6 Target blood pressure"
            ],
        },
    ]
)

In [201]:
pd.DataFrame(guideline_eval_rows)[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

,eval_id,query,relevant_document_id,relevant_sections
0,who_01,At what blood pressure level should pharmacological treatment for hypertension be started?,who_hypertension_2021,[3 Recommendations > 3.1 Blood pressure threshold for initiation of pharmacological treatment]
1,who_02,What laboratory tests should be considered before starting antihypertensive medication and during treatment?,who_hypertension_2021,[3 Recommendations > 3.2 Laboratory testing before and during pharmacological treatment]
2,who_03,Which classes of medication are recommended as first-line treatment for hypertension?,who_hypertension_2021,[3 Recommendations > 3.4 Drug classes to be used as first-line agents]
3,who_04,When should combination drug therapy be used for treating hypertension?,who_hypertension_2021,[3 Recommendations > 3.5 Combination therapy]
4,who_05,What blood pressure target should treatment aim for in adults receiving therapy for hypertension?,who_hypertension_2021,[3 Recommendations > 3.6 Target blood pressure]


In [202]:
for section in show_document_sections(
    "va_dod_type2_diabetes_2023"
):
    print(section)

Appendix B: Glycemic Control Targets and Monitoring
Appendix C: Pharmacotherapy
Appendix I: Alternative Text Descriptions of Algorithm
I. Introduction
I. Introduction ........................................................................................................... 5
II. Background > A. Description of Type 2 Diabetes Mellitus
II. Background > B. Epidemiology and Impact on the General Population
III. Scope of This Guideline
III. Scope of This Guideline > A. Guideline Audience
III. Scope of This Guideline > B. Guideline Population
IV. Highlighted Features of this Guideline > A. Highlights in this Guideline Update
IV. Highlighted Features of this Guideline > B. Components of the Guideline
IV. Highlighted Features of this Guideline > C. Racial and Ethnic Demographic Terminology in This Guideline
IX. Recommendations
IX. Recommendations > A. Prediabetes
IX. Recommendations > B. Telehealth
IX. Recommendations > C. Diabetes Mellitus
IX. Recommendations > D. Non-Pharmacotherapy
IX. Rec

In [203]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "t2d_01",
            "question_source": "guideline",
            "query": (
                "How should glycemic control be monitored in adults "
                "with type 2 diabetes?"
            ),
            "relevant_document_id": "va_dod_type2_diabetes_2023",
            "relevant_sections": [
                "Appendix B: Glycemic Control Targets and Monitoring",
            ],
        },
        {
            "eval_id": "t2d_02",
            "question_source": "guideline",
            "query": (
                "What lifestyle and other non-drug interventions are "
                "recommended for adults with type 2 diabetes?"
            ),
            "relevant_document_id": "va_dod_type2_diabetes_2023",
            "relevant_sections": [
                "IX. Recommendations > D. Non-Pharmacotherapy",
            ],
        },
        {
            "eval_id": "t2d_03",
            "question_source": "guideline",
            "query": (
                "What should be considered when choosing medications "
                "for treatment of type 2 diabetes?"
            ),
            "relevant_document_id": "va_dod_type2_diabetes_2023",
            "relevant_sections": [
                "IX. Recommendations > E. Pharmacotherapy",
                "Appendix C: Pharmacotherapy",
            ],
        },
        {
            "eval_id": "t2d_04",
            "question_source": "guideline",
            "query": (
                "What management strategies are recommended for adults "
                "with prediabetes to reduce progression to type 2 diabetes?"
            ),
            "relevant_document_id": "va_dod_type2_diabetes_2023",
            "relevant_sections": [
                "IX. Recommendations > A. Prediabetes",
            ],
        },
        {
            "eval_id": "t2d_05",
            "question_source": "guideline",
            "query": (
                "What education and support should patients receive "
                "to help them manage type 2 diabetes themselves?"
            ),
            "relevant_document_id": "va_dod_type2_diabetes_2023",
            "relevant_sections": [
                "VIII. Algorithm > B. Module B: Self-Management Education and Support",
            ],
        },
    ]
)

In [204]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

guideline_eval_df[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

Guideline questions: 10


,eval_id,query,relevant_document_id,relevant_sections
0,who_01,At what blood pressure level should pharmacological treatment for hypertension be started?,who_hypertension_2021,[3 Recommendations > 3.1 Blood pressure threshold for initiation of pharmacological treatment]
1,who_02,What laboratory tests should be considered before starting antihypertensive medication and during treatment?,who_hypertension_2021,[3 Recommendations > 3.2 Laboratory testing before and during pharmacological treatment]
2,who_03,Which classes of medication are recommended as first-line treatment for hypertension?,who_hypertension_2021,[3 Recommendations > 3.4 Drug classes to be used as first-line agents]
3,who_04,When should combination drug therapy be used for treating hypertension?,who_hypertension_2021,[3 Recommendations > 3.5 Combination therapy]
4,who_05,What blood pressure target should treatment aim for in adults receiving therapy for hypertension?,who_hypertension_2021,[3 Recommendations > 3.6 Target blood pressure]
5,t2d_01,How should glycemic control be monitored in adults with type 2 diabetes?,va_dod_type2_diabetes_2023,[Appendix B: Glycemic Control Targets and Monitoring]
6,t2d_02,What lifestyle and other non-drug interventions are recommended for adults with type 2 diabetes?,va_dod_type2_diabetes_2023,[IX. Recommendations > D. Non-Pharmacotherapy]
7,t2d_03,What should be considered when choosing medications for treatment of type 2 diabetes?,va_dod_type2_diabetes_2023,"[IX. Recommendations > E. Pharmacotherapy, Appendix C: Pharmacotherapy]"
8,t2d_04,What management strategies are recommended for adults with prediabetes to reduce progression to type 2 diabetes?,va_dod_type2_diabetes_2023,[IX. Recommendations > A. Prediabetes]
9,t2d_05,What education and support should patients receive to help them manage type 2 diabetes themselves?,va_dod_type2_diabetes_2023,[VIII. Algorithm > B. Module B: Self-Management Education and Support]


In [205]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "ckd_01",
            "question_source": "guideline",
            "query": (
                "How should chronic kidney disease be evaluated, "
                "staged, and monitored over time?"
            ),
            "relevant_document_id": "va_dod_ckd_2025",
            "relevant_sections": [
                "VIII. Algorithm",
                "Appendix G. Alternative Text Descriptions of Algorithms",
                "Appendix I. Monitoring of CKD Table",
            ],
        },
        {
            "eval_id": "ckd_02",
            "question_source": "guideline",
            "query": (
                "When should a patient with chronic kidney disease "
                "be referred to a nephrology specialist?"
            ),
            "relevant_document_id": "va_dod_ckd_2025",
            "relevant_sections": [
                "IX. Recommendations",
                "Appendix G. Alternative Text Descriptions of Algorithms",
            ],
        },
        {
            "eval_id": "ckd_03",
            "question_source": "guideline",
            "query": (
                "How should the decision between dialysis and "
                "conservative management be made in advanced CKD?"
            ),
            "relevant_document_id": "va_dod_ckd_2025",
            "relevant_sections": [
                "IX. Recommendations",
            ],
        },
        {
            "eval_id": "ckd_04",
            "question_source": "guideline",
            "query": (
                "How should hyperkalemia be managed in a patient "
                "with chronic kidney disease?"
            ),
            "relevant_document_id": "va_dod_ckd_2025",
            "relevant_sections": [
                "Appendix M. Management of Hyperkalemia > C. Management",
                (
                    "Appendix M. Management of Hyperkalemia "
                    "> D. Dietary Considerations"
                ),
                (
                    "Appendix M. Management of Hyperkalemia "
                    "> E. Use of Potassium Binders"
                ),
            ],
        },
        {
            "eval_id": "ckd_05",
            "question_source": "guideline",
            "query": (
                "What medication-related precautions should be considered "
                "in patients with chronic kidney disease?"
            ),
            "relevant_document_id": "va_dod_ckd_2025",
            "relevant_sections": [
                (
                    "Appendix K. Nephrotoxic Agents and Medication "
                    "Dose Adjustments in CKD > B. Nephrotoxic Medications"
                ),
                (
                    "Appendix K. Nephrotoxic Agents and Medication "
                    "Dose Adjustments in CKD > C. Medication Management in CKD"
                ),
            ],
        },
    ]
)

In [206]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

guideline_eval_df.tail(5)[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

Guideline questions: 15


,eval_id,query,relevant_document_id,relevant_sections
10,ckd_01,"How should chronic kidney disease be evaluated, staged, and monitored over time?",va_dod_ckd_2025,"[VIII. Algorithm, Appendix G. Alternative Text Descriptions of Algorithms, Appendix I. Monitoring of CKD Table]"
11,ckd_02,When should a patient with chronic kidney disease be referred to a nephrology specialist?,va_dod_ckd_2025,"[IX. Recommendations, Appendix G. Alternative Text Descriptions of Algorithms]"
12,ckd_03,How should the decision between dialysis and conservative management be made in advanced CKD?,va_dod_ckd_2025,[IX. Recommendations]
13,ckd_04,How should hyperkalemia be managed in a patient with chronic kidney disease?,va_dod_ckd_2025,"[Appendix M. Management of Hyperkalemia > C. Management, Appendix M. Management of Hyperkalemia > D. Dietary Considerations, Appendix M. Management of Hyperkalemia > E. Use of Potassium Binders]"
14,ckd_05,What medication-related precautions should be considered in patients with chronic kidney disease?,va_dod_ckd_2025,"[Appendix K. Nephrotoxic Agents and Medication Dose Adjustments in CKD > B. Nephrotoxic Medications, Appendix K. Nephrotoxic Agents and Medication Dose Adjustments in CKD > C. Medication Management in CKD]"


In [207]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "asthma_01",
            "question_source": "guideline",
            "query": (
                "How should asthma be diagnosed and assessed "
                "in a patient with respiratory symptoms?"
            ),
            "relevant_document_id": "va_dod_asthma_2025",
            "relevant_sections": [
                "IX. Recommendations > A. Diagnosis and Assessment",
                (
                    "Appendix K: Alternative Text Descriptions of Algorithms "
                    "> A. Module A: Assessment and Diagnosis of Asthma"
                ),
            ],
        },
        {
            "eval_id": "asthma_02",
            "question_source": "guideline",
            "query": (
                "How can the level of asthma control be assessed "
                "in a patient who already has asthma?"
            ),
            "relevant_document_id": "va_dod_asthma_2025",
            "relevant_sections": [
                (
                    "Appendix C: Assessments of Asthma Severity and Control "
                    "> B. Assessment of Asthma Control"
                ),
                "II. Background > B. Classification of Asthma Severity and Control",
            ],
        },
        {
            "eval_id": "asthma_03",
            "question_source": "guideline",
            "query": (
                "When should a patient with asthma be referred "
                "to a specialist?"
            ),
            "relevant_document_id": "va_dod_asthma_2025",
            "relevant_sections": [
                (
                    "Appendix C: Assessments of Asthma Severity and Control "
                    "> C. Indications for Specialist Referral"
                ),
            ],
        },
        {
            "eval_id": "asthma_04",
            "question_source": "guideline",
            "query": (
                "What treatment and management approaches are recommended "
                "for adults with asthma?"
            ),
            "relevant_document_id": "va_dod_asthma_2025",
            "relevant_sections": [
                "IX. Recommendations > B. Treatment and Management",
                (
                    "Appendix K: Alternative Text Descriptions of Algorithms "
                    "> B. Module B: Initiation of Therapy"
                ),
            ],
        },
        {
            "eval_id": "asthma_05",
            "question_source": "guideline",
            "query": (
                "What alternative diagnoses should be considered "
                "when symptoms resemble asthma?"
            ),
            "relevant_document_id": "va_dod_asthma_2025",
            "relevant_sections": [
                (
                    "Appendix C: Assessments of Asthma Severity and Control "
                    "> D. Identifying Alternative Diagnoses"
                ),
                (
                    "Appendix D: Details of a Comprehensive History "
                    "and Physical Exam > A. Details of a Comprehensive History"
                ),
            ],
        },
    ]
)

In [208]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

guideline_eval_df.tail(5)[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

Guideline questions: 20


,eval_id,query,relevant_document_id,relevant_sections
15,asthma_01,How should asthma be diagnosed and assessed in a patient with respiratory symptoms?,va_dod_asthma_2025,"[IX. Recommendations > A. Diagnosis and Assessment, Appendix K: Alternative Text Descriptions of Algorithms > A. Module A: Assessment and Diagnosis of Asthma]"
16,asthma_02,How can the level of asthma control be assessed in a patient who already has asthma?,va_dod_asthma_2025,"[Appendix C: Assessments of Asthma Severity and Control > B. Assessment of Asthma Control, II. Background > B. Classification of Asthma Severity and Control]"
17,asthma_03,When should a patient with asthma be referred to a specialist?,va_dod_asthma_2025,[Appendix C: Assessments of Asthma Severity and Control > C. Indications for Specialist Referral]
18,asthma_04,What treatment and management approaches are recommended for adults with asthma?,va_dod_asthma_2025,"[IX. Recommendations > B. Treatment and Management, Appendix K: Alternative Text Descriptions of Algorithms > B. Module B: Initiation of Therapy]"
19,asthma_05,What alternative diagnoses should be considered when symptoms resemble asthma?,va_dod_asthma_2025,"[Appendix C: Assessments of Asthma Severity and Control > D. Identifying Alternative Diagnoses, Appendix D: Details of a Comprehensive History and Physical Exam > A. Details of a Comprehensive History]"


In [209]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "lbp_01",
            "question_source": "guideline",
            "query": (
                "How should a patient with low back pain be evaluated "
                "to determine whether further diagnostic testing is needed?"
            ),
            "relevant_document_id": "va_dod_low_back_pain_2022",
            "relevant_sections": [
                "IX. Recommendations > A. Evaluation and Diagnostic Approach",
                "VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain",
                (
                    "Appendix I: Alternative Text Descriptions of Algorithm "
                    "> A. Module A: Initial Evaluation of Low Back Pain"
                ),
            ],
        },
        {
            "eval_id": "lbp_02",
            "question_source": "guideline",
            "query": (
                "What self-care and education should be recommended "
                "to patients with low back pain?"
            ),
            "relevant_document_id": "va_dod_low_back_pain_2022",
            "relevant_sections": [
                "IX. Recommendations > B. Patient Education and Self-care",
            ],
        },
        {
            "eval_id": "lbp_03",
            "question_source": "guideline",
            "query": (
                "Which non-drug and non-invasive treatments are recommended "
                "for managing low back pain?"
            ),
            "relevant_document_id": "va_dod_low_back_pain_2022",
            "relevant_sections": [
                (
                    "IX. Recommendations > C. "
                    "Non-pharmacologic and Non-invasive Therapy"
                ),
                "VIII. Algorithm > B. Module B: Management of Low Back Pain",
            ],
        },
        {
            "eval_id": "lbp_04",
            "question_source": "guideline",
            "query": (
                "What medications can be considered for the treatment "
                "of low back pain?"
            ),
            "relevant_document_id": "va_dod_low_back_pain_2022",
            "relevant_sections": [
                "IX. Recommendations > D. Pharmacotherapy",
                "**Appendix E: Dosing for Select Pharmacologic Agentsa,b**",
            ],
        },
        {
            "eval_id": "lbp_05",
            "question_source": "guideline",
            "query": (
                "What non-surgical invasive treatments may be considered "
                "for patients with persistent low back pain?"
            ),
            "relevant_document_id": "va_dod_low_back_pain_2022",
            "relevant_sections": [
                "IX. Recommendations > F. Non-surgical Invasive Therapy",
            ],
        },
    ]
)

In [210]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

guideline_eval_df.tail(5)[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

Guideline questions: 25


,eval_id,query,relevant_document_id,relevant_sections
20,lbp_01,How should a patient with low back pain be evaluated to determine whether further diagnostic testing is needed?,va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > A. Module A: Initial Evaluation of Low Back Pain]"
21,lbp_02,What self-care and education should be recommended to patients with low back pain?,va_dod_low_back_pain_2022,[IX. Recommendations > B. Patient Education and Self-care]
22,lbp_03,Which non-drug and non-invasive treatments are recommended for managing low back pain?,va_dod_low_back_pain_2022,"[IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy, VIII. Algorithm > B. Module B: Management of Low Back Pain]"
23,lbp_04,What medications can be considered for the treatment of low back pain?,va_dod_low_back_pain_2022,"[IX. Recommendations > D. Pharmacotherapy, **Appendix E: Dosing for Select Pharmacologic Agentsa,b**]"
24,lbp_05,What non-surgical invasive treatments may be considered for patients with persistent low back pain?,va_dod_low_back_pain_2022,[IX. Recommendations > F. Non-surgical Invasive Therapy]


In [211]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "mdd_01",
            "question_source": "guideline",
            "query": (
                "How should a patient be initially assessed "
                "when major depressive disorder is suspected?"
            ),
            "relevant_document_id": "va_dod_major_depression_2022",
            "relevant_sections": [
                "VIII. Algorithm > A. Module A: Initial Assessment and Treatment",
                "IX. Recommendations > A. Screening",
            ],
        },
        {
            "eval_id": "mdd_02",
            "question_source": "guideline",
            "query": (
                "How should treatment outcomes be monitored "
                "in a patient being treated for major depression?"
            ),
            "relevant_document_id": "va_dod_major_depression_2022",
            "relevant_sections": [
                "IX. Recommendations > B. Monitoring Outcomes",
                (
                    "Appendix I: Quick Guide to the Patient Health Questionnaire "
                    "in Clinical Practice > C. Using the PHQ-9 "
                    "in Measurement-Based Care"
                ),
            ],
        },
        {
            "eval_id": "mdd_03",
            "question_source": "guideline",
            "query": (
                "What treatments are recommended for uncomplicated "
                "major depressive disorder?"
            ),
            "relevant_document_id": "va_dod_major_depression_2022",
            "relevant_sections": [
                "IX. Recommendations > D. Treatment of Uncomplicated MDD",
            ],
        },
        {
            "eval_id": "mdd_04",
            "question_source": "guideline",
            "query": (
                "What treatment options should be considered when depression "
                "is severe or has not responded adequately to initial treatment?"
            ),
            "relevant_document_id": "va_dod_major_depression_2022",
            "relevant_sections": [
                (
                    "IX. Recommendations > E. Treatment of MDD that is Severe "
                    "or has a Partial or Limited Response to Initial Treatment"
                ),
                "VIII. Algorithm > B. Module B: Advanced Care Management",
            ],
        },
        {
            "eval_id": "mdd_05",
            "question_source": "guideline",
            "query": (
                "What non-medication self-help or complementary approaches "
                "may be considered for major depression?"
            ),
            "relevant_document_id": "va_dod_major_depression_2022",
            "relevant_sections": [
                (
                    "IX. Recommendations > H. Self-help, "
                    "Complementary, and Alternative Treatments"
                ),
            ],
        },
    ]
)

In [212]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

guideline_eval_df.tail(5)[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

Guideline questions: 30


,eval_id,query,relevant_document_id,relevant_sections
25,mdd_01,How should a patient be initially assessed when major depressive disorder is suspected?,va_dod_major_depression_2022,"[VIII. Algorithm > A. Module A: Initial Assessment and Treatment, IX. Recommendations > A. Screening]"
26,mdd_02,How should treatment outcomes be monitored in a patient being treated for major depression?,va_dod_major_depression_2022,"[IX. Recommendations > B. Monitoring Outcomes, Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > C. Using the PHQ-9 in Measurement-Based Care]"
27,mdd_03,What treatments are recommended for uncomplicated major depressive disorder?,va_dod_major_depression_2022,[IX. Recommendations > D. Treatment of Uncomplicated MDD]
28,mdd_04,What treatment options should be considered when depression is severe or has not responded adequately to initial treatment?,va_dod_major_depression_2022,"[IX. Recommendations > E. Treatment of MDD that is Severe or has a Partial or Limited Response to Initial Treatment, VIII. Algorithm > B. Module B: Advanced Care Management]"
29,mdd_05,What non-medication self-help or complementary approaches may be considered for major depression?,va_dod_major_depression_2022,"[IX. Recommendations > H. Self-help, Complementary, and Alternative Treatments]"


In [213]:
for section in show_document_sections(
    "va_dod_pregnancy_2023"
):
    print(section)

I. Introduction
I. Introduction ........................................................................................................... 6 II. Background ........................................................................................................... 6
II. Background > A. Description of Pregnancy
II. Background > B. Current Trends in Pregnancy Epidemiology in the United States General Population
II. Background > C. Pregnancy in the Department of Defense and the Department of Veterans Affairs Populations
III. Scope of This Guideline
III. Scope of This Guideline > A. Guideline Audience
III. Scope of This Guideline > B. Guideline Population
IV. Highlighted Features of This Guideline > A. Highlights in This Guideline
IV. Highlighted Features of This Guideline > B. Components of This Guideline
IV. Highlighted Features of This Guideline > C. Racial and Ethnic Demographic Terminology in This Guideline
IV. Highlighted Features of This Guideline > D. Sex Terminology in This Guidel

In [214]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "pregnancy_01",
            "question_source": "guideline",
            "query": (
                "What should be assessed and discussed during "
                "the initial prenatal visit?"
            ),
            "relevant_document_id": "va_dod_pregnancy_2023",
            "relevant_sections": [
                "X. Routine Pregnancy Care > A. Initial Prenatal Visit",
            ],
        },
        {
            "eval_id": "pregnancy_02",
            "question_source": "guideline",
            "query": (
                "What care and monitoring should be provided "
                "during follow-up prenatal visits?"
            ),
            "relevant_document_id": "va_dod_pregnancy_2023",
            "relevant_sections": [
                "X. Routine Pregnancy Care > B. Subsequent Prenatal Visits",
            ],
        },
        {
            "eval_id": "pregnancy_03",
            "question_source": "guideline",
            "query": (
                "What care should be provided to a patient "
                "during the postpartum period?"
            ),
            "relevant_document_id": "va_dod_pregnancy_2023",
            "relevant_sections": [
                "X. Routine Pregnancy Care > C. Postpartum",
            ],
        },
        {
            "eval_id": "pregnancy_04",
            "question_source": "guideline",
            "query": (
                "What education should patients receive "
                "as part of routine pregnancy care?"
            ),
            "relevant_document_id": "va_dod_pregnancy_2023",
            "relevant_sections": [
                "X. Routine Pregnancy Care > D. Education",
            ],
        },
        {
            "eval_id": "pregnancy_05",
            "question_source": "guideline",
            "query": (
                "When should a pregnant patient be referred "
                "to an advanced prenatal care provider?"
            ),
            "relevant_document_id": "va_dod_pregnancy_2023",
            "relevant_sections": [
                "XI. Referral Indications > A. Advanced Prenatal Care Provider",
                "VIII. Recommendations > B. Complicated Obstetrics",
            ],
        },
    ]
)

In [215]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

guideline_eval_df.tail(5)[
    [
        "eval_id",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
]

Guideline questions: 35


,eval_id,query,relevant_document_id,relevant_sections
30,pregnancy_01,What should be assessed and discussed during the initial prenatal visit?,va_dod_pregnancy_2023,[X. Routine Pregnancy Care > A. Initial Prenatal Visit]
31,pregnancy_02,What care and monitoring should be provided during follow-up prenatal visits?,va_dod_pregnancy_2023,[X. Routine Pregnancy Care > B. Subsequent Prenatal Visits]
32,pregnancy_03,What care should be provided to a patient during the postpartum period?,va_dod_pregnancy_2023,[X. Routine Pregnancy Care > C. Postpartum]
33,pregnancy_04,What education should patients receive as part of routine pregnancy care?,va_dod_pregnancy_2023,[X. Routine Pregnancy Care > D. Education]
34,pregnancy_05,When should a pregnant patient be referred to an advanced prenatal care provider?,va_dod_pregnancy_2023,"[XI. Referral Indications > A. Advanced Prenatal Care Provider, VIII. Recommendations > B. Complicated Obstetrics]"


In [216]:
guideline_eval_rows.extend(
    [
        {
            "eval_id": "sti_01",
            "question_source": "guideline",
            "query": (
                "What treatment is recommended for chlamydial infection "
                "in adolescents and adults?"
            ),
            "relevant_document_id": "cdc_sti_2021",
            "relevant_sections": [
                (
                    "Chlamydial Infections > "
                    "Chlamydial Infection Among Adolescents and Adults > Treatment"
                ),
            ],
        },
        {
            "eval_id": "sti_02",
            "question_source": "guideline",
            "query": (
                "How should a first clinical episode of genital herpes "
                "be treated?"
            ),
            "relevant_document_id": "cdc_sti_2021",
            "relevant_sections": [
                (
                    "Diseases Characterized by Genital, Anal, or Perianal Ulcers "
                    "> Genital Herpes > Genital Herpes Management "
                    "> First Clinical Episode of Genital Herpes"
                ),
            ],
        },
        {
            "eval_id": "sti_03",
            "question_source": "guideline",
            "query": (
                "How should cervicitis be evaluated to determine "
                "the likely infectious cause?"
            ),
            "relevant_document_id": "cdc_sti_2021",
            "relevant_sections": [
                (
                    "Diseases Characterized by Urethritis and Cervicitis "
                    "> Cervicitis > Diagnostic Considerations"
                ),
                (
                    "Diseases Characterized by Urethritis and Cervicitis "
                    "> Cervicitis > Etiology"
                ),
            ],
        },
        {
            "eval_id": "sti_04",
            "question_source": "guideline",
            "query": (
                "What tests are used to diagnose syphilis?"
            ),
            "relevant_document_id": "cdc_sti_2021",
            "relevant_sections": [
                "Syphilis > Diagnostic Considerations",
                (
                    "Syphilis > Diagnostic Considerations "
                    "> Nontreponemal Tests and Traditional Algorithm"
                ),
                (
                    "Syphilis > Diagnostic Considerations "
                    "> Treponemal Tests and Reverse Sequence Algorithm"
                ),
            ],
        },
        {
            "eval_id": "sti_05",
            "question_source": "guideline",
            "query": (
                "What treatments are recommended for bacterial vaginosis?"
            ),
            "relevant_document_id": "cdc_sti_2021",
            "relevant_sections": [
                (
                    "Diseases Characterized by Vulvovaginal Itching, Burning, "
                    "Irritation, Odor, or Discharge > Bacterial Vaginosis > Treatment"
                ),
            ],
        },
    ]
)

In [217]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

print(
    "Guideline questions:",
    len(guideline_eval_df),
)

print(
    guideline_eval_df[
        "relevant_document_id"
    ].value_counts()
)

Guideline questions: 40
relevant_document_id
who_hypertension_2021           5
va_dod_type2_diabetes_2023      5
va_dod_ckd_2025                 5
va_dod_asthma_2025              5
va_dod_low_back_pain_2022       5
va_dod_major_depression_2022    5
va_dod_pregnancy_2023           5
cdc_sti_2021                    5
Name: count, dtype: int64


In [218]:
existing_sections = set(
    zip(
        retrieval_records_df["document_id"],
        retrieval_records_df["section_path"],
    )
)

missing_gold_sections = []

for _, row in guideline_eval_df.iterrows():
    for section in row["relevant_sections"]:
        if (
            row["relevant_document_id"],
            section,
        ) not in existing_sections:
            missing_gold_sections.append(
                {
                    "eval_id": row["eval_id"],
                    "document_id": row["relevant_document_id"],
                    "section": section,
                }
            )

missing_gold_sections_df = pd.DataFrame(
    missing_gold_sections
)

print(
    "Missing gold sections:",
    len(missing_gold_sections_df),
)

missing_gold_sections_df

Missing gold sections: 1


,eval_id,document_id,section
0,lbp_04,va_dod_low_back_pain_2022,"**Appendix E: Dosing for Select Pharmacologic Agentsa,b**"


In [219]:
for row in guideline_eval_rows:
    if row["eval_id"] == "lbp_04":
        row["relevant_sections"] = [
            "IX. Recommendations > D. Pharmacotherapy",
            "Appendix E: Dosing for Select Pharmacologic Agentsa,b",
        ]

In [220]:
guideline_eval_df = pd.DataFrame(
    guideline_eval_rows
)

In [221]:
existing_sections = set(
    zip(
        retrieval_records_df["document_id"],
        retrieval_records_df["section_path"],
    )
)

missing_gold_sections = []

for _, row in guideline_eval_df.iterrows():
    for section in row["relevant_sections"]:
        if (
            row["relevant_document_id"],
            section,
        ) not in existing_sections:
            missing_gold_sections.append(
                {
                    "eval_id": row["eval_id"],
                    "document_id": row["relevant_document_id"],
                    "section": section,
                }
            )

missing_gold_sections_df = pd.DataFrame(
    missing_gold_sections
)

print(
    "Missing gold sections:",
    len(missing_gold_sections_df),
)

missing_gold_sections_df

Missing gold sections: 0


""


### Финальная разметка real-dev вопросов

После ручной проверки coverage осталось 7 естественных пользовательских вопросов,
для которых текущая knowledge base действительно содержит релевантную информацию.

Gold-разметка фиксируется до запуска retrieval.

Релевантность на этом этапе определяется на уровне `section_path`:
retrieval считается успешным, если среди top-k chunks присутствует chunk
из одного из заранее размеченных релевантных разделов.

In [222]:
real_dev_gold = {
    "candidate_0005": {
        "relevant_document_id": "va_dod_major_depression_2022",
        "relevant_sections": [
            "VIII. Algorithm > A. Module A: Initial Assessment and Treatment",
            "IX. Recommendations > C. Treatment Setting",
            "IX. Recommendations > D. Treatment of Uncomplicated MDD",
            (
                "IX. Recommendations > H. Self-help, "
                "Complementary, and Alternative Treatments"
            ),
        ],
    },

    "candidate_0014": {
        "relevant_document_id": "va_dod_low_back_pain_2022",
        "relevant_sections": [
            "IX. Recommendations > A. Evaluation and Diagnostic Approach",
            "VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain",
            (
                "Appendix I: Alternative Text Descriptions of Algorithm "
                "> A. Module A: Initial Evaluation of Low Back Pain"
            ),
        ],
    },

    "candidate_0029": {
        "relevant_document_id": "va_dod_ckd_2025",
        "relevant_sections": [
            "IX. Recommendations",
            "VIII. Algorithm",
            "Appendix G. Alternative Text Descriptions of Algorithms",
            "Appendix I. Monitoring of CKD Table",
        ],
    },

    "candidate_0048": {
        "relevant_document_id": "va_dod_low_back_pain_2022",
        "relevant_sections": [
            "IX. Recommendations > D. Pharmacotherapy",
            (
                "IX. Recommendations > C. "
                "Non-pharmacologic and Non-invasive Therapy"
            ),
            "VIII. Algorithm > B. Module B: Management of Low Back Pain",
            (
                "Appendix I: Alternative Text Descriptions of Algorithm "
                "> B. Module B: Management of Low Back Pain"
            ),
        ],
    },

    "candidate_0078": {
        "relevant_document_id": "va_dod_low_back_pain_2022",
        "relevant_sections": [
            "IX. Recommendations > A. Evaluation and Diagnostic Approach",
            "VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain",
            "VIII. Algorithm > B. Module B: Management of Low Back Pain",
        ],
    },

    "candidate_0097": {
        "relevant_document_id": "va_dod_asthma_2025",
        "relevant_sections": [
            (
                "Appendix C: Assessments of Asthma Severity and Control "
                "> B. Assessment of Asthma Control"
            ),
            (
                "Appendix C: Assessments of Asthma Severity and Control "
                "> C. Indications for Specialist Referral"
            ),
            "IX. Recommendations > B. Treatment and Management",
        ],
    },

    "candidate_0135": {
        "relevant_document_id": "va_dod_asthma_2025",
        "relevant_sections": [
            "IX. Recommendations > A. Diagnosis and Assessment",
            (
                "Appendix C: Assessments of Asthma Severity and Control "
                "> D. Identifying Alternative Diagnoses"
            ),
            (
                "Appendix D: Details of a Comprehensive History and Physical Exam "
                "> A. Details of a Comprehensive History"
            ),
            (
                "Appendix K: Alternative Text Descriptions of Algorithms "
                "> A. Module A: Assessment and Diagnosis of Asthma"
            ),
        ],
    },
}

In [223]:
real_dev_eval_rows = []

for candidate_id, gold in real_dev_gold.items():
    source_row = retrieval_eval_labeling_df.loc[
        retrieval_eval_labeling_df["candidate_id"].eq(candidate_id)
    ].iloc[0]

    real_dev_eval_rows.append(
        {
            "eval_id": f'dev_{candidate_id.split("_")[-1]}',
            "candidate_id": candidate_id,
            "question_source": "real_dev",
            "query": source_row["input"],
            "relevant_document_id": gold["relevant_document_id"],
            "relevant_sections": gold["relevant_sections"],
        }
    )

real_dev_eval_df = pd.DataFrame(
    real_dev_eval_rows
)

real_dev_eval_df[
    [
        "eval_id",
        "candidate_id",
        "relevant_document_id",
        "relevant_sections",
    ]
]

,eval_id,candidate_id,relevant_document_id,relevant_sections
0,dev_0005,candidate_0005,va_dod_major_depression_2022,"[VIII. Algorithm > A. Module A: Initial Assessment and Treatment, IX. Recommendations > C. Treatment Setting, IX. Recommendations > D. Treatment of Uncomplicated MDD, IX. Recommendations > H. Self-help, Complementary, and Alternative Treatments]"
1,dev_0014,candidate_0014,va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > A. Module A: Initial Evaluation of Low Back Pain]"
2,dev_0029,candidate_0029,va_dod_ckd_2025,"[IX. Recommendations, VIII. Algorithm, Appendix G. Alternative Text Descriptions of Algorithms, Appendix I. Monitoring of CKD Table]"
3,dev_0048,candidate_0048,va_dod_low_back_pain_2022,"[IX. Recommendations > D. Pharmacotherapy, IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy, VIII. Algorithm > B. Module B: Management of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > B. Module B: Management of Low Back Pain]"
4,dev_0078,candidate_0078,va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, VIII. Algorithm > B. Module B: Management of Low Back Pain]"
5,dev_0097,candidate_0097,va_dod_asthma_2025,"[Appendix C: Assessments of Asthma Severity and Control > B. Assessment of Asthma Control, Appendix C: Assessments of Asthma Severity and Control > C. Indications for Specialist Referral, IX. Recommendations > B. Treatment and Management]"
6,dev_0135,candidate_0135,va_dod_asthma_2025,"[IX. Recommendations > A. Diagnosis and Assessment, Appendix C: Assessments of Asthma Severity and Control > D. Identifying Alternative Diagnoses, Appendix D: Details of a Comprehensive History and Physical Exam > A. Details of a Comprehensive History, Appendix K: Alternative Text Descriptions of Algorithms > A. Module A: Assessment and Diagnosis of Asthma]"


In [224]:
existing_sections = set(
    zip(
        retrieval_records_df["document_id"],
        retrieval_records_df["section_path"],
    )
)

missing_real_dev_sections = []

for _, row in real_dev_eval_df.iterrows():
    for section in row["relevant_sections"]:
        if (
            row["relevant_document_id"],
            section,
        ) not in existing_sections:
            missing_real_dev_sections.append(
                {
                    "eval_id": row["eval_id"],
                    "document_id": row["relevant_document_id"],
                    "section": section,
                }
            )

missing_real_dev_sections_df = pd.DataFrame(
    missing_real_dev_sections
)

print(
    "Real-dev questions:",
    len(real_dev_eval_df),
)

print(
    "Missing real-dev gold sections:",
    len(missing_real_dev_sections_df),
)

missing_real_dev_sections_df

Real-dev questions: 7
Missing real-dev gold sections: 0


""


### Финальный retrieval evaluation set

Итоговый benchmark состоит из двух независимых частей:

- `real_dev` — 7 естественных пользовательских вопросов из `dev`, которые
  вручную признаны покрываемыми текущей knowledge base;
- `guideline` — 40 контролируемых вопросов, по 5 на каждый документ KB.

Всего: 47 вопросов.

Gold-релевантность задана на уровне `section_path` и была определена
до просмотра результатов dense retrieval.

После сохранения этого набора gold-разметка не изменяется на основании
результатов retriever.

In [225]:
guideline_eval_df = guideline_eval_df.copy()
real_dev_eval_df = real_dev_eval_df.copy()

guideline_eval_df["candidate_id"] = None

retrieval_eval_df = pd.concat(
    [
        real_dev_eval_df,
        guideline_eval_df,
    ],
    ignore_index=True,
    sort=False,
)

retrieval_eval_df = retrieval_eval_df[
    [
        "eval_id",
        "candidate_id",
        "question_source",
        "query",
        "relevant_document_id",
        "relevant_sections",
    ]
].copy()

retrieval_eval_df

,eval_id,candidate_id,question_source,query,relevant_document_id,relevant_sections
0,dev_0005,candidate_0005,real_dev,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depr...,va_dod_major_depression_2022,"[VIII. Algorithm > A. Module A: Initial Assessment and Treatment, IX. Recommendations > C. Treatment Setting, IX. Recommendations > D. Treatment of Uncomplicated MDD, IX. Recommendations > H. Self-help, Complementary, and Alternative Treatments]"
1,dev_0014,candidate_0014,real_dev,"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to determine the cause of this?.",va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > A. Module A: Initial Evaluation of Low Back Pain]"
2,dev_0029,candidate_0029,real_dev,"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be tested further",va_dod_ckd_2025,"[IX. Recommendations, VIII. Algorithm, Appendix G. Alternative Text Descriptions of Algorithms, Appendix I. Monitoring of CKD Table]"
3,dev_0048,candidate_0048,real_dev,hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but no cure now my doctor suggest me duzella 30 mg so how much its useful for me pls suggest me some good medicine iam fearing tht any side affects will come because my marriage is thier in a month and ia...,va_dod_low_back_pain_2022,"[IX. Recommendations > D. Pharmacotherapy, IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy, VIII. Algorithm > B. Module B: Management of Low Back Pain, Appendix I: Alternative Text Descriptions of Algorithm > B. Module B: Management of Low Back Pain]"
4,dev_0078,candidate_0078,real_dev,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable.",va_dod_low_back_pain_2022,"[IX. Recommendations > A. Evaluation and Diagnostic Approach, VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain, VIII. Algorithm > B. Module B: Management of Low Back Pain]"
5,dev_0097,candidate_0097,real_dev,"ihave severe asthma and have been 5 times in the last 9 weeks and been given lots of steroids, so i am on insulin now. i am so wore ouy and can hardly get around. thw wheezing has stopped. i have high numbers of latic acid in the muscles. please tell me what to do.",va_dod_asthma_2025,"[Appendix C: Assessments of Asthma Severity and Control > B. Assessment of Asthma Control, Appendix C: Assessments of Asthma Severity and Control > C. Indications for Specialist Referral, IX. Recommendations > B. Treatment and Manage

In [226]:
print("Total questions:", len(retrieval_eval_df))

print(
    "\nBy source:"
)
print(
    retrieval_eval_df[
        "question_source"
    ].value_counts()
)

print(
    "\nUnique eval_id:",
    retrieval_eval_df["eval_id"].is_unique,
)

print(
    "Unique queries:",
    retrieval_eval_df["query"].is_unique,
)

print(
    "Missing queries:",
    retrieval_eval_df["query"].isna().sum(),
)

print(
    "Missing gold documents:",
    retrieval_eval_df[
        "relevant_document_id"
    ].isna().sum(),
)

print(
    "Empty relevant_sections:",
    retrieval_eval_df[
        "relevant_sections"
    ].apply(len).eq(0).sum(),
)

Total questions: 47

By source:
question_source
guideline    40
real_dev      7
Name: count, dtype: int64

Unique eval_id: True
Unique queries: True
Missing queries: 0
Missing gold documents: 0
Empty relevant_sections: 0


In [227]:
existing_sections = set(
    zip(
        retrieval_records_df["document_id"],
        retrieval_records_df["section_path"],
    )
)

missing_sections = []

for _, row in retrieval_eval_df.iterrows():
    for section in row["relevant_sections"]:
        if (
            row["relevant_document_id"],
            section,
        ) not in existing_sections:
            missing_sections.append(
                {
                    "eval_id": row["eval_id"],
                    "document_id": row["relevant_document_id"],
                    "section": section,
                }
            )

missing_sections_df = pd.DataFrame(
    missing_sections
)

print(
    "Missing gold sections:",
    len(missing_sections_df),
)

missing_sections_df

Missing gold sections: 0


""


In [228]:
RETRIEVAL_EVAL_PATH = (
    MATERIALS_DIR
    / "retrieval_eval_v1.jsonl"
)

retrieval_eval_df.to_json(
    RETRIEVAL_EVAL_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(
    "Saved:",
    RETRIEVAL_EVAL_PATH,
)

Saved: ..\materials\retrieval_eval_v1.jsonl


In [229]:
retrieval_eval_frozen_df = pd.read_json(
    RETRIEVAL_EVAL_PATH,
    lines=True,
)

print(
    retrieval_eval_frozen_df.shape
)

print(
    type(
        retrieval_eval_frozen_df.loc[
            0,
            "relevant_sections",
        ]
    )
)

(47, 6)
<class 'list'>


In [230]:
retrieval_eval_frozen_df = pd.read_json(
    RETRIEVAL_EVAL_PATH,
    lines=True,
)

### Оценка dense retrieval на frozen benchmark

Для каждого вопроса получаем top-10 chunks текущим dense retriever.

Chunk считается релевантным, если одновременно:

- `document_id` совпадает с gold-документом;
- `section_path` входит в заранее размеченный список релевантных разделов.

Считаем:

- Hit@1
- Hit@3
- Hit@5
- MRR@10

Результаты считаются отдельно для `real_dev` и `guideline`,
поскольку эти две части benchmark имеют разную сложность и происхождение.

In [231]:
def evaluate_retrieval_query(row, top_k=10):
    results = dense_search(
        row["query"],
        top_k=top_k,
    ).reset_index(drop=True).copy()

    gold_document = row["relevant_document_id"]
    gold_sections = set(row["relevant_sections"])

    results["is_relevant"] = (
        results["document_id"].eq(gold_document)
        &
        results["section_path"].isin(gold_sections)
    )

    relevant_positions = np.flatnonzero(
        results["is_relevant"].to_numpy()
    )

    if len(relevant_positions) > 0:
        first_relevant_rank = int(
            relevant_positions[0] + 1
        )
    else:
        first_relevant_rank = None

    return {
        "eval_id": row["eval_id"],
        "candidate_id": row["candidate_id"],
        "question_source": row["question_source"],
        "query": row["query"],

        "first_relevant_rank": first_relevant_rank,

        "hit_at_1": (
            first_relevant_rank is not None
            and first_relevant_rank <= 1
        ),
        "hit_at_3": (
            first_relevant_rank is not None
            and first_relevant_rank <= 3
        ),
        "hit_at_5": (
            first_relevant_rank is not None
            and first_relevant_rank <= 5
        ),

        "rr_at_10": (
            1 / first_relevant_rank
            if first_relevant_rank is not None
            else 0.0
        ),

        "top1_document_id": (
            results.iloc[0]["document_id"]
            if len(results) > 0
            else None
        ),
        "top1_section_path": (
            results.iloc[0]["section_path"]
            if len(results) > 0
            else None
        ),
        "top1_score": (
            float(results.iloc[0]["score"])
            if len(results) > 0
            else None
        ),
    }

In [232]:
retrieval_eval_results = []

for _, row in retrieval_eval_frozen_df.iterrows():
    retrieval_eval_results.append(
        evaluate_retrieval_query(
            row,
            top_k=10,
        )
    )

retrieval_eval_results_df = pd.DataFrame(
    retrieval_eval_results
)

retrieval_eval_results_df.head()

,eval_id,candidate_id,question_source,query,first_relevant_rank,hit_at_1,hit_at_3,hit_at_5,rr_at_10,top1_document_id,top1_section_path,top1_score
0,dev_0005,candidate_0005,real_dev,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depr...,8.0,False,False,False,0.125,va_dod_major_depression_2022,II. Background > A. Description of Major Depressive Disorder (MDD),0.628073
1,dev_0014,candidate_0014,real_dev,"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to determine the cause of this?.",1.0,True,True,True,1.000,va_dod_low_back_pain_2022,IX. Recommendations > A. Evaluation and Diagnostic Approach,0.668335
2,dev_0029,candidate_0029,real_dev,"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be tested further",1.0,True,True,True,1.000,va_dod_ckd_2025,IX. Recommendations,0.723086
3,dev_0048,candidate_0048,real_dev,hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but no cure now my doctor suggest me duzella 30 mg so how much its useful for me pls suggest me some good medicine iam fearing tht any side affects will come because my marriage is thier in a month and ia...,1.0,True,True,True,1.000,va_dod_low_back_pain_2022,IX. Recommendations > D. Pharmacotherapy,0.692542
4,dev_0078,candidate_0078,real_dev,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable.",2.0,False,True,True,0.500,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,0.646642


In [233]:
def summarize_retrieval_metrics(df):
    return pd.Series(
        {
            "n": len(df),
            "Hit@1": df["hit_at_1"].mean(),
            "Hit@3": df["hit_at_3"].mean(),
            "Hit@5": df["hit_at_5"].mean(),
            "MRR@10": df["rr_at_10"].mean(),
        }
    )

In [234]:
overall_metrics = summarize_retrieval_metrics(
    retrieval_eval_results_df
)

overall_metrics

n         47.000000
Hit@1      0.829787
Hit@3      0.936170
Hit@5      0.936170
MRR@10     0.882092
dtype: float64

In [ ]:
metrics_by_source = (
    retrieval_eval_results_df
    .groupby("question_source")
    .apply(summarize_retrieval_metrics)
)

metrics_by_source

In [236]:
retrieval_failures_df = (
    retrieval_eval_results_df[
        ~retrieval_eval_results_df["hit_at_5"]
    ][
        [
            "eval_id",
            "question_source",
            "query",
            "first_relevant_rank",
            "top1_document_id",
            "top1_section_path",
            "top1_score",
        ]
    ]
    .reset_index(drop=True)
)

print(
    "Failures at Hit@5:",
    len(retrieval_failures_df),
)

retrieval_failures_df

Failures at Hit@5: 3


,eval_id,question_source,query,first_relevant_rank,top1_document_id,top1_section_path,top1_score
0,dev_0005,real_dev,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depr...,8.0,va_dod_major_depression_2022,II. Background > A. Description of Major Depressive Disorder (MDD),0.628073
1,t2d_01,guideline,How should glycemic control be monitored in adults with type 2 diabetes?,NaN,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,0.768070
2,asthma_01,guideline,How should asthma be diagnosed and assessed in a patient with respiratory symptoms?,NaN,va_dod_asthma_2025,VIII. Algorithm,0.734461


In [237]:
retrieval_eval_results_df[
    [
        "eval_id",
        "question_source",
        "first_relevant_rank",
    ]
].sort_values(
    [
        "question_source",
        "first_relevant_rank",
    ],
    na_position="last",
)

,eval_id,question_source,first_relevant_rank
7,who_01,guideline,1.0
8,who_02,guideline,1.0
9,who_03,guideline,1.0
10,who_04,guideline,1.0
11,who_05,guideline,1.0
13,t2d_02,guideline,1.0
14,t2d_03,guideline,1.0
15,t2d_04,guideline,1.0
16,t2d_05,guideline,1.0
18,ckd_02,guideline,1.0


### Анализ ошибок retrieval

Ошибки `Hit@5` разбираются вручную.

Для каждого случая проверяется:

1. действительно ли top-ranked chunks нерелевантны;
2. не является ли gold-разметка неполной;
3. найден ли правильный документ, но неверно ранжирован конкретный раздел.

Gold не изменяется автоматически по результатам retriever.

In [238]:
def inspect_retrieval_case(eval_id, top_k=10):
    row = retrieval_eval_frozen_df[
        retrieval_eval_frozen_df["eval_id"].eq(eval_id)
    ].iloc[0]

    results = dense_search(
        row["query"],
        top_k=top_k,
    ).reset_index(drop=True).copy()

    gold_sections = set(
        row["relevant_sections"]
    )

    results["rank"] = (
        np.arange(len(results)) + 1
    )

    results["gold_match"] = (
        results["document_id"].eq(
            row["relevant_document_id"]
        )
        &
        results["section_path"].isin(
            gold_sections
        )
    )

    print("=" * 100)
    print("EVAL ID:", eval_id)
    print("\nQUERY:")
    print(row["query"])

    print("\nGOLD DOCUMENT:")
    print(row["relevant_document_id"])

    print("\nGOLD SECTIONS:")
    for section in row["relevant_sections"]:
        print("-", section)

    return results[
        [
            "rank",
            "score",
            "document_id",
            "section_path",
            "page",
            "gold_match",
            "text",
        ]
    ]

In [239]:
t2d_01_results = inspect_retrieval_case(
    "t2d_01"
)

pd.set_option(
    "display.max_colwidth",
    500,
)

t2d_01_results

EVAL ID: t2d_01

QUERY:
How should glycemic control be monitored in adults with type 2 diabetes?

GOLD DOCUMENT:
va_dod_type2_diabetes_2023

GOLD SECTIONS:
- Appendix B: Glycemic Control Targets and Monitoring


,rank,score,document_id,section_path,page,gold_match,text
0,1,0.768070,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,39,False,"the following recommendation: For adults with type 2 diabetes mellitus, we suggest using high glycemic variability over time (e.g., fluctuation in HbA1c or fasting blood glucose) as a prognostic indicator for risk of hypoglycemia, morbidity, and mortality."
1,2,0.761366,va_dod_type2_diabetes_2023,IX. Recommendations > E. Pharmacotherapy,69,False,"Recommendation 25. In adults with type 2 diabetes mellitus, especially those 65 years and older, we suggest prioritizing drug classes other than insulin, sulfonylureas, or meglitinides to minimize the risk of hypoglycemia, if glycemic control can be achieved with other treatments. (Weak for | Reviewed, New-added) 26. In adults with type 2 diabetes mellitus who have co-occurring cognitive impairment or risk of falls, there is insufficient evidence to recommend for or against specific treatmen..."
2,3,0.744628,va_dod_type2_diabetes_2023,IX. Recommendations,25,False,"Topic Sub-topic # Recommendation Strengtha Categoryb Neither for nor against Reviewed, New­added 4. There is insufficient evidence to recommend for or against routine screening or using a specific tool to screen for or diagnose diabetes distress. 5. Weak for Reviewed, New­added In adults with type 2 diabetes mellitus and co­occurring non­alcoholic fatty liver disease, we suggest clinicians should assess for fibrosis using a non­invasive tool (e.g., Fibrosis­4). Screening for Comorbidities Ne..."
3,4,0.739464,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,37,False,"be burdensome for providers with limited time. Most screening tools were developed to target older patients, so their accuracy in younger patients is unclear. The Work Group systematically searched for evidence and did not identify any studies that met inclusion criteria regarding routine screening for fall risks and cognitive decline in adult patients with T2DM. Therefore, this recommendation is categorized as Reviewed, New-added. The Work Group’s confidence in the quality of the evidence w..."
4,5,0.732190,va_dod_type2_diabetes_2023,IX. Recommendations,25,False,"’s appraisal of the risk benefit ratio, patient characteristics, presence or absence of type 2 diabetes mellitus complications, comorbidities, and life expectancy. 10. We suggest an HbA1c range of 7.0–8.5% for most patients, if it can be safely achieved. Weak for Not reviewed, Amended Glycemic Management 11. Weak for Reviewed, New­added In insulin­treated adults with type 2 diabetes mellitus who are not achieving glycemic goals, we suggest real­time continuous glucose monitoring to decrease ..."
5,6,0.725484,va_dod_type2_diabetes_2023,IX. Recommendations > C. Diabetes Mellitus,46,False,"Discussion Although evidence is increasing that continuous glucose monitoring (CGM) can improve glucose control and might reduce hypoglycemia in T1DM patients, whether CGM improves outcomes in patients with T2DM is unclear. However, the systematic evidence review found that results differed depending on the type of CGM. For real-time CGM (rtCGM) (i.e., glucose readings automatically and continuously pushed to the user’s receiver or smartphone), we found moderate quality evidence that rtCGM u..."
6,7,0.721560,va_dod_type2_diabetes_2023,IX. Recommendations,26,False,"feedback, hypnosis, guided imagery, massage therapy, yoga, or tai chi to improve outcomes. 19. Strong for Reviewed, New­added For adults with type 2 diabetes mellitus with atherosclerotic cardiovascular disease, we recommend glucagon­like peptide­1 receptor agonists or sodium­glucose cotransporter­2 inhibitors with proven cardiovascular benefits to decrease the risk of major adverse cardiovascular events. 20. Weak for Reviewed, New­added For adults with type 2 diabetes mellitus at high risk ..."
7,8,0.720276,va_dod_type2_diabetes_2023,I. Introduction,5,False,"Recommendation Strength). This CP

In [240]:
asthma_01_results = inspect_retrieval_case(
    "asthma_01"
)

asthma_01_results

EVAL ID: asthma_01

QUERY:
How should asthma be diagnosed and assessed in a patient with respiratory symptoms?

GOLD DOCUMENT:
va_dod_asthma_2025

GOLD SECTIONS:
- IX. Recommendations > A. Diagnosis and Assessment
- Appendix K: Alternative Text Descriptions of Algorithms > A. Module A: Assessment and Diagnosis of Asthma


,rank,score,document_id,section_path,page,gold_match,text
0,1,0.734461,va_dod_asthma_2025,VIII. Algorithm,21,False,Module A. Assessment and Diagnosis of Asthma 1 P with symptoms and signs compatible with asthma (see Si a A) 2 4 ollow up as Is the patient Treat exacerbation Yes appropriate acutely ill No Is there a confident clinical diagnosis of No Treat alternative Yes diagnosis asthma (see Si a and Is there an alternative diagnosis A i C) No 8 Yes Yes Is the patient capable of spirometry and is it Obtain spirometry readily available 10 No Is spirometry compatible Yes with asthma (consistent with obstru...
1,2,0.733807,va_dod_asthma_2025,II. Background > A. Description of Asthma,7,False,"weak. In some cases, asthma exacerbations can be severe and potentially life threatening. Airway inflammation and bronchial hyperreactivity are considered the primary underlying pathologic processes. Asthma is characterized by airway obstruction that is usually at least partially reversible. Despite these unifying characteristics, asthma is a very heterogeneous condition. There is significant variability in presenting symptoms, degree of airway obstruction, level of impairment, responsivenes..."
2,3,0.718827,va_dod_asthma_2025,IX. Recommendations > B. Treatment and Management,57,False,"of patients with stable asthma was included in the 2009 and 2019 VA/DOD Asthma CPGs, these recommendations were based on guidance from other organizations. Current literature does not support routine (e.g., quarterly) spirometry for stable patients with asthma in the general population. However, there may be specific requirements that need to be considered for active-duty members of the military. While there are no obvious harms associated with spirometry, there may be added burden and many ..."
3,4,0.716059,va_dod_asthma_2025,IX. Recommendations > B. Treatment and Management,57,False,"to mitigate concerns related to resources and stigma. Thus, the Work Group decided upon a Weak for recommendation. d. Monitoring and Follow-up Recommendation 18. We suggest against utilizing spirometry for routine monitoring of patients with stable asthma. (Weak against | Not reviewed, Not changed) Discussion The diagnosis of asthma is a clinical diagnosis based on history, physical examination, and findings suggestive of airway hyperactivity. While objective measurements of airway reactivit..."
4,5,0.714595,va_dod_asthma_2025,Appendix C: Assessments of Asthma Severity and Control > C. Indications for Specialist Referral,88,False,"Patients may benefit from specialist referral to pulmonology, allergy/immunology, ENT and others, for assistance in asthma management in the following circumstances: • Patient has ever had a life-threatening asthma exacerbation • Patients needing advanced therapies, such as biologics, roflumilast, or a chronic macrolide antibiotic. Also, any patients requiring more than two courses of oral corticosteroids in one year or had an exacerbation requiring hospitalization • Other conditions that co..."
5,6,0.706535,va_dod_asthma_2025,Appendix D: Details of a Comprehensive History and Physical Exam > A. Details of a Comprehensive History,93,False,"• The history should focus on the characterization of symptoms related to airway obstruction or airway hyper-responsiveness, note that patients usually, but not always, present with two or more symptoms:  Cough  Wheezing  Shortness of breath  Chest tightness  Sputum production • The pattern of symptoms should be characterized:  Onset  Duration  Frequency  Diurnal variation  Seasonality • Precipitating and aggravating factors should be explored:  Viral infections  Exercise  Envir..."
6,7,0.706210,va_dod_asthma_2025,VIII. Algorithm,25,False,"Sidebar E: Asthma Education and Self-Management Support Patients and caregivers should be informed of the diagnosis of asthma. Their current understanding of asthma and treatment adherence should be assessed, they should be provided evidence-based education and material

In [241]:
dev_0005_results = inspect_retrieval_case(
    "dev_0005"
)

dev_0005_results

EVAL ID: dev_0005

QUERY:
Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?

GOLD DOCUMENT:
va_dod_major_depression_2022

GOLD SECTIONS:
- VIII. Algorithm > A. Module A: Initial Assessment and Treatment
- IX. Recommendations > C. Treatment Setting
- IX. Recommendations > D. Treatment of Uncomplicated MDD
- IX. Recommendations > H. Self-help, Complementary, and Alternative Treatments


,rank,score,document_id,section_path,page,gold_match,text
0,1,0.628073,va_dod_major_depression_2022,II. Background > A. Description of Major Depressive Disorder (MDD),7,False,"the most prevalent and disabling form of depression. In addition to the immediate symptoms of depression, MDD precipitates overall poor quality of life (QoL), decreased productivity, increased obesity and sedentary behavior, and increased risk of mortality from suicide and other causes. Social difficulties potentially emerge from the condition, including stigma, loss of employment, and relationship conflict. Approaches used in the literature to categorize the severity of MDD are discussed in..."
1,2,0.626393,va_dod_major_depression_2022,Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > B. Scoring the PHQ-9 (223),128,False,"Table I-3). Note: The diagnoses of MDD requires ruling out a history of a manic episode (Bipolar Disorder) and a physical disorder, medication or other drug as the biological cause of the depressive symptoms. In the context of bereavement or other significant loss, symptoms consistent with a major depression can occur, and the diagnosis of MDD is considered if there is indication the symptoms are distinguished from normal response to loss given the individual’s history, cultural norms, and t..."
2,3,0.622863,va_dod_major_depression_2022,IX. Recommendations,23,False,"of the cardinal symptoms, and severe MDD had 8 – 9 cardinal symptoms. Regarding chronic depression, also termed persistent depressive disorder (or dysthymia) in DSM-5, symptoms must be present for most days over two years. Typically, symptoms do not remit for greater than two months at a time. Appendix K also describes depression subsets in detail. Topic # Recommendation Strengtha Categoryb 1. We suggest that all patients not currently receiving treatment for depression be screened for depre..."
3,4,0.622756,va_dod_major_depression_2022,IX. Recommendations,24,False,": · Severe (e.g., PHQ-9 >20) · Persistent major depressive disorder (duration greater than two years) · Recurrent (with two or more episodes) Weak for Reviewed, 16. Amended For patients with MDD who have demonstrated partial or no response to an adequate trial of initial pharmacotherapy, we suggest (not rank ordered): · Switching to another antidepressant (including TCAs, MAOIs, or those in Recommendation 12) · Switching to psychotherapy · Augmenting with a psychotherapy · Augmenting with a ..."
4,5,0.622244,va_dod_major_depression_2022,Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > E. Additional Clinical Considerations,131,False,"After completing a provisional diagnosis with the PHQ-9, additional clinical considerations exist that may shape management and treatment options.(224) · Has a psychosocial stressor(s) triggered current symptoms? · What is the duration of the current disturbance, and has the patient received any treatment for it? · To what extent are the symptoms of the patient impairing their capacity to complete daily work and life duties and responsibilities? · Is there a history of similar episodes, and ..."
5,6,0.621646,va_dod_major_depression_2022,IX. Recommendations > G. Recommendations for Specific Populations,58,False,"symptoms.(177) However, that meta-analysis and an RCT by Cohen et al. (2010) showed significantly better recovery and improvement of symptoms for patients in couples-focused therapy compared to patients on a waitlist control or no treatment.(178) The Work Group was unable to make recommendations related to couples-focused therapy as adjunct or augmentation strategy because no studies comparing couples-focused therapy with combined treatment were found. Current findings were limited by small ..."
6,7,0.617259,va_dod_major_depression_2022,IX. Recommendations > G. Recommendations for Specific Populations,57,False,"Gould et al. (2012) (n=1,712) also supported the use of CBT for older adults, although the quality of

### Анализ ошибок strict section-level evaluation

Все три случая, классифицированные как `Hit@5 = False`, были проверены вручную.

Проверка показала, что они не являются однозначными ошибками dense retrieval:

- `asthma_01`: top-1 chunk содержит алгоритм assessment and diagnosis of asthma;
- `t2d_01`: top-ranked chunks содержат информацию о glycemic variability,
  HbA1c и continuous glucose monitoring;
- `dev_0005`: top-ranked chunks содержат информацию о differential assessment
  при bereavement, исключении bipolar disorder и дополнительных clinical
  considerations; заранее размеченный non-medication section появляется на rank 8.

Таким образом, strict matching по заранее выбранным `section_path`
может давать false negatives, поскольку одна и та же клиническая информация
представлена в нескольких разделах guideline.

Поэтому section-level Hit@k используется как консервативная метрика,
а ошибки дополнительно анализируются вручную.

In [242]:
failure_audit_df = pd.DataFrame(
    [
        {
            "eval_id": "dev_0005",
            "strict_hit_at_5": False,
            "audit_result": "gold_incomplete",
            "best_answer_bearing_rank": 2,
            "notes": (
                "Rank 2 discusses ruling out manic episode and distinguishing "
                "MDD from normal bereavement; rank 5 contains additional "
                "clinical assessment considerations; predefined self-help "
                "section appears at rank 8."
            ),
        },
        {
            "eval_id": "t2d_01",
            "strict_hit_at_5": False,
            "audit_result": "gold_incomplete",
            "best_answer_bearing_rank": 1,
            "notes": (
                "Top-ranked recommendations contain information on glycemic "
                "variability, HbA1c and CGM, although gold contained only "
                "Appendix B."
            ),
        },
        {
            "eval_id": "asthma_01",
            "strict_hit_at_5": False,
            "audit_result": "gold_incomplete",
            "best_answer_bearing_rank": 1,
            "notes": (
                "Top-1 chunk is the Assessment and Diagnosis of Asthma "
                "algorithm and directly addresses the query."
            ),
        },
    ]
)

failure_audit_df

,eval_id,strict_hit_at_5,audit_result,best_answer_bearing_rank,notes
0,dev_0005,False,gold_incomplete,2,Rank 2 discusses ruling out manic episode and distinguishing MDD from normal bereavement; rank 5 contains additional clinical assessment considerations; predefined self-help section appears at rank 8.
1,t2d_01,False,gold_incomplete,1,"Top-ranked recommendations contain information on glycemic variability, HbA1c and CGM, although gold contained only Appendix B."
2,asthma_01,False,gold_incomplete,1,Top-1 chunk is the Assessment and Diagnosis of Asthma algorithm and directly addresses the query.


In [243]:
def evaluate_document_retrieval(row, top_k=10):
    results = dense_search(
        row["query"],
        top_k=top_k,
    ).reset_index(drop=True)

    matches = (
        results["document_id"]
        .eq(row["relevant_document_id"])
        .to_numpy()
    )

    positions = np.flatnonzero(matches)

    first_document_rank = (
        int(positions[0] + 1)
        if len(positions) > 0
        else None
    )

    return {
        "eval_id": row["eval_id"],
        "question_source": row["question_source"],
        "first_document_rank": first_document_rank,
        "doc_hit_at_1": (
            first_document_rank is not None
            and first_document_rank <= 1
        ),
        "doc_hit_at_3": (
            first_document_rank is not None
            and first_document_rank <= 3
        ),
        "doc_hit_at_5": (
            first_document_rank is not None
            and first_document_rank <= 5
        ),
    }

In [244]:
document_eval_results_df = pd.DataFrame(
    [
        evaluate_document_retrieval(row)
        for _, row in retrieval_eval_frozen_df.iterrows()
    ]
)

document_metrics_by_source = (
    document_eval_results_df
    .groupby("question_source")
    .agg(
        n=("eval_id", "size"),
        Doc_Hit_at_1=("doc_hit_at_1", "mean"),
        Doc_Hit_at_3=("doc_hit_at_3", "mean"),
        Doc_Hit_at_5=("doc_hit_at_5", "mean"),
    )
)

document_metrics_by_source

,n,Doc_Hit_at_1,Doc_Hit_at_3,Doc_Hit_at_5
question_source,,,,
guideline,40,1.0,1.0,1.0
real_dev,7,1.0,1.0,1.0


In [245]:
pd.Series(
    {
        "n": len(document_eval_results_df),
        "Doc Hit@1": document_eval_results_df["doc_hit_at_1"].mean(),
        "Doc Hit@3": document_eval_results_df["doc_hit_at_3"].mean(),
        "Doc Hit@5": document_eval_results_df["doc_hit_at_5"].mean(),
    }
)

n            47.0
Doc Hit@1     1.0
Doc Hit@3     1.0
Doc Hit@5     1.0
dtype: float64

In [246]:
guideline_results_with_docs = (
    retrieval_eval_results_df[
        retrieval_eval_results_df["question_source"].eq("guideline")
    ]
    .merge(
        retrieval_eval_frozen_df[
            [
                "eval_id",
                "relevant_document_id",
            ]
        ],
        on="eval_id",
        how="left",
    )
)

metrics_by_document = (
    guideline_results_with_docs
    .groupby("relevant_document_id")
    .agg(
        n=("eval_id", "size"),
        Hit_at_1=("hit_at_1", "mean"),
        Hit_at_3=("hit_at_3", "mean"),
        Hit_at_5=("hit_at_5", "mean"),
        MRR_at_10=("rr_at_10", "mean"),
    )
    .sort_index()
)

metrics_by_document

,n,Hit_at_1,Hit_at_3,Hit_at_5,MRR_at_10
relevant_document_id,,,,,
cdc_sti_2021,5,1.0,1.0,1.0,1.000000
va_dod_asthma_2025,5,0.8,0.8,0.8,0.800000
va_dod_ckd_2025,5,0.8,1.0,1.0,0.900000
va_dod_low_back_pain_2022,5,0.6,1.0,1.0,0.766667
va_dod_major_depression_2022,5,0.8,1.0,1.0,0.900000
va_dod_pregnancy_2023,5,1.0,1.0,1.0,1.000000
va_dod_type2_diabetes_2023,5,0.8,0.8,0.8,0.800000
who_hypertension_2021,5,1.0,1.0,1.0,1.000000


In [247]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

retrieval_eval_results_df.to_csv(
    RESULTS_DIR / "retrieval_dense_eval_v1.csv",
    index=False,
)

failure_audit_df.to_csv(
    RESULTS_DIR / "retrieval_dense_failure_audit_v1.csv",
    index=False,
)

metrics_by_document.to_csv(
    RESULTS_DIR / "retrieval_dense_metrics_by_document_v1.csv"
)

### Вывод по dense retrieval

Для оценки retrieval был зафиксирован benchmark из 47 вопросов:

- 40 guideline-grounded вопросов, по 5 на каждый документ;
- 7 естественных вопросов из `dev`, вручную признанных покрываемыми KB.

На strict section-level evaluation dense retrieval показал:

- Hit@1 = 0.830
- Hit@3 = 0.936
- Hit@5 = 0.936
- MRR@10 = 0.882

На уровне документа правильный guideline находился на первом месте
для всех 47 вопросов:

- Document Hit@1 = 1.00

Три strict Hit@5 failures были дополнительно проверены вручную.
Во всех случаях retriever возвращал содержательно релевантные chunks,
но их `section_path` отсутствовал в заранее заданном gold-наборе.

Это показывает ограничение section-level evaluation:
одна и та же клиническая информация может находиться в нескольких
recommendations, algorithms, appendices или background sections.

Поэтому strict Hit@k сохраняется как консервативная воспроизводимая метрика,
а ручной audit используется для интерпретации false negatives.

На данном benchmark нет оснований менять embedding model или chunking:
dense retrieval стабильно определяет правильный источник и в большинстве
случаев высоко ранжирует релевантные passages.

Ограничения оценки:

- real-dev subset мал (n=7);
- guideline-grounded вопросы являются контролируемыми и проще реальных
  пользовательских запросов;
- section_path является приближённой, а не полной passage-level
  разметкой релевантности.

In [248]:
import pandas as pd
import pyarrow

print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

pandas: 2.3.3
pyarrow: 25.0.1


In [249]:
RETRIEVAL_DATA_DIR = Path(
    "../data/processed/retrieval"
)

RETRIEVAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

chunks_df.to_parquet(
    RETRIEVAL_DATA_DIR / "chunks.parquet",
    index=False,
)

print(
    "Saved chunks:",
    RETRIEVAL_DATA_DIR / "chunks.parquet",
)

print(
    "Chunks:",
    len(chunks_df),
)

Saved chunks: ..\data\processed\retrieval\chunks.parquet
Chunks: 2135


In [250]:
np.save(
    RETRIEVAL_DATA_DIR / "embeddings.npy",
    document_embeddings,
)

print(
    "Saved embeddings:",
    RETRIEVAL_DATA_DIR / "embeddings.npy",
)

print(
    "Shape:",
    document_embeddings.shape,
)

Saved embeddings: ..\data\processed\retrieval\embeddings.npy
Shape: (2135, 768)


In [ ]:
import json

retrieval_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "query_prefix": BGE_QUERY_INSTRUCTION,
    "target_chunk_tokens": TARGET_CHUNK_TOKENS,
    "chunk_overlap_tokens": CHUNK_OVERLAP_TOKENS,
    "normalize_embeddings": True,
    "embedding_dimension": int(
        document_embeddings.shape[1]
    ),
    "num_chunks": int(
        len(chunks_df)
    ),
}

with open(
    RETRIEVAL_DATA_DIR / "config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        retrieval_config,
        f,
        indent=2,
        ensure_ascii=False,
    )